# INFO01_TIMING_FIXED (from INFO00_BN05_CONTROL_DIAGNOSTICS)

**تغییرات قطعی (مسیر آموزش را عوض نمی‌کنند):** ابزار اندازه‌گیری زمان اصلاح شده است.
- تایمر برازش محلی فقط حلقهٔ forward/backward/update را می‌شمارد؛ ساخت `tf.data`، ارزیابی محلی، ثبت پارامترهای BN، نوشتن CSV و کپی بهترین وزن‌ها بیرون از تایمرند.
- پیش از هر فاز طبقه‌بندی و هر برازش محلی، گراف آموزش روی یک دیتاست جداگانهٔ ۱ تا ۲ batchی (اندازهٔ کامل و باقی‌مانده) trace و کامپایل می‌شود و سپس تمام متغیرهای مدل و بهینه‌ساز **بیت‌به‌بیت** بازگردانده و بررسی می‌شوند. زمان این کار در `compile_warmup_sec` جدا ثبت می‌شود.
- پایان هر بازهٔ زمانی با خواندن یک مقدار از GPU همگام می‌شود.
- زمان منتخب = Σ زمان خالص فازهای طبقه‌بندی تا بهترین ایپاک + Σ زمان خالص برازش محلی تا بهترین ایپاک محلی.
- پس از بازگردانی بهترین وضعیت هر فاز، Validation دوباره سنجیده و با مقدار ثبت‌شده مقایسه می‌شود.

**آزمایش‌های کنترل‌شده (پیش‌فرض همه خاموش = همان مسیر آموزش INFO00):** `EXP_ENDPOINT2_PROTOCOL`، `EXP_S10_RELATIVE_SCALE`، `EXP_S10_HEAD_EPOCHS`، `EXP_RECOVERY_MAX_EPOCHS`، `EXP_LOCAL_MAX_EPOCHS` و پرچم موجود `BN_DIRECT_BIAS`. در هر اجرا فقط یک پرچم را تغییر دهید.

Test فقط پس از انتخاب نهایی ارزیابی می‌شود. LUT و دادهٔ CIFAR-10 را مانند قبل به Input اضافه کنید؛ GPU T4 و Restart & Run All.


In [ ]:
import os, gc, csv, json, math, time, copy, shutil, zipfile, hashlib, itertools, warnings, platform
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL","2")
os.environ.setdefault("TF_DETERMINISTIC_OPS","1")

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

SEED=42
INFORMATION_TRAIN_IMAGES=400
INFORMATION_LOCAL_HOLDOUT_IMAGES=200
INFORMATION_BATCH_SIZE=4
# BN experiment; change SEED here only for a new matched seed run.
BN_DIRECT_BIAS=False
BN_LEARN_SCALE=False
BN_REESTIMATE_STATS=False
BN_FIT_ALL_TRAIN=False
BN_FLOAT_FULL_RANGE=True
BN_FROM_FIRST_STEP=False
BN_BIAS_UPDATE_MULTIPLIER=100.0
BN_SCALE_UPDATE_MULTIPLIER=20.0

# ---------------- INFO01: measurement switches (definite fixes) ----------------
# Warm-up = one full + one remainder batch from a SEPARATE tiny dataset, then a
# bit-exact restore of every model/optimizer variable. Training batches/order unchanged.
TIMING_WARMUP_BEFORE_PHASE=True
TIMING_VERIFY_EXACT_RESTORE=True
VERIFY_RESTORED_VALIDATION=True   # re-evaluate validation after each best-state restore
# ---------------- INFO01: controlled experiments (defaults = INFO00 path) -------
# Change ONE switch per run. Selection stays validation-only.
EXP_ENDPOINT2_PROTOCOL='reference'   # 'reference' (12@1e-3,1@2e-4) | 'plateau' (baseline-like)
EXP_E2_PLATEAU=dict(lr=1.0e-3,patience=4,max_epochs=40,
                    low_lr=5.0e-4,low_patience=3,low_max_epochs=10,
                    lower_lr=2.5e-4,lower_patience=3,lower_max_epochs=10)
EXP_S10_RELATIVE_SCALE=False   # S10: one weight peak + one chain gain for all 10 classes
EXP_S10_HEAD_EPOCHS=None       # None keeps INFO00 (0 epochs)
EXP_RECOVERY_MAX_EPOCHS=None   # e.g. {6:1,9:1}; None keeps INFO00
EXP_LOCAL_MAX_EPOCHS=None      # e.g. {8:12}; None keeps INFO00

np.random.seed(SEED); tf.random.set_seed(SEED)
_gpus=tf.config.list_physical_devices("GPU")
if _gpus: tf.config.set_visible_devices(_gpus[:1],"GPU")
tf.keras.mixed_precision.set_global_policy("float32")
try: tf.config.experimental.enable_op_determinism()
except Exception as exc: print("Deterministic ops unavailable:",exc)
for gpu in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(gpu,True)
    except Exception: pass

# ---------------- User-facing mode switches ----------------
RUN_MODE="from_scratch"
RESUME_CHECKPOINT_PATH = None

ARTIFACT_VERSION="INFO01_TIMING_FIXED"
_EXP_TAGS=[]
if EXP_ENDPOINT2_PROTOCOL!='reference':_EXP_TAGS.append('E2'+EXP_ENDPOINT2_PROTOCOL.upper())
if EXP_S10_RELATIVE_SCALE:_EXP_TAGS.append('S10REL')
if EXP_S10_HEAD_EPOCHS:_EXP_TAGS.append(f'HEAD{int(EXP_S10_HEAD_EPOCHS)}')
if EXP_RECOVERY_MAX_EPOCHS:_EXP_TAGS.append('RECOV')
if EXP_LOCAL_MAX_EPOCHS:_EXP_TAGS.append('LOCAL')
if BN_DIRECT_BIAS:_EXP_TAGS.append('BNDIRECTBIAS')
assert EXP_ENDPOINT2_PROTOCOL in ('reference','plateau')
EXPERIMENT_NAME="INFO01_TIMING_FIXED"+("__"+"_".join(_EXP_TAGS) if _EXP_TAGS else "")
RUN_ID=time.strftime("%Y%m%d_%H%M%S")
OUT_DIR=Path("/kaggle/working")/f"{EXPERIMENT_NAME}_{RUN_ID}"
OUT_DIR.mkdir(parents=True,exist_ok=True)
RUN_T0=time.perf_counter()

# ---------------- Physical contract ----------------
V_LIMIT=0.5
ADC_BITS=8; DAC_BITS=8
ADC_SIGNED_QMAX=(1<<(ADC_BITS-1))-1       # 127
DAC_UNSIGNED_QMAX=(1<<DAC_BITS)-1        # 255
SIG_MIN_US=0.12; SIG_MAX_US=8.0
EPS=1e-8; VOLTAGE_TOL=2e-6
# Numerical headroom for BN->ReLU->DAC. This is not clipping: the affine BN is
# projected to a value slightly below VDD so float32 roundoff cannot cross 0.5 V.
BN_DAC_SAFE_VMAX = V_LIMIT - 1e-5
FINAL_LOGIT_TEMPERATURE_SHIFT=3  # ADC-code / 8; 0 reproduces V10.4 logit scale
ALL_INVERTER_DEPTHS=(1,2,3,4,5,6)
ODD_INVERTER_DEPTHS=(1,3,5)
EVEN_INVERTER_DEPTHS=(2,4,6)

# No minimum gain-utilization threshold. Hardware cost is secondary.
GAIN_EDGE_MARGIN=0.999

# ---------------- Robust PRE-JOINT DEPTH policy ----------------
# S3-S10 are random before Full-Joint. A near-zero random column can otherwise
# force the COMMON inverter depth upward. V10.4 uses a robust gain statistic ONLY
# to choose the common depth. After depth is chosen, every column independently
# receives the maximum useful gain realizable with that depth through its own R.
# Therefore ordinary columns are never capped by a percentile; only columns whose
# requested gain exceeds the PHYSICAL CAPACITY of the selected depth are limited.
ROBUST_INIT_DEPTH_REFERENCE=True
ROBUST_INIT_P_LO=95.0
ROBUST_INIT_P_HI=99.0
ROBUST_INIT_P95_MULT=1.50
ROBUST_INIT_MIN_COLUMNS=32       # S10 has only 10 outputs: use exact max there
ROBUST_INIT_ACTIVATION_MARGIN=1.02

# Event-driven patience. Epoch count is NOT fixed.
FRONT_PATIENCE=2
JOINT_PATIENCE=4
REFINE_PATIENCE=3
CHAIN_BALANCED_POINTS=129
GAIN_SPLIT_DELTA=(0.40,0.75)

# ---------------- Training controller ----------------
V7_PERFORMANCE_PATIENCE=2
V7_SAFETY_CAP=100
FRONT_LR=1e-3
JOINT_LR=5e-4
LOW_LR=2.5e-4
LOWER_LR=1.25e-4
SHIFTADD_QAT_LR=7.5e-5
SHIFTADD_QAT_PATIENCE=3
SHIFTADD_QAT_MAX_EPOCHS=16
BATCH_SIZE=32
EVAL_BATCH_SIZE=64
VAL_SIZE=5000
CALIBRATION_SAMPLES=200  # balanced TRAIN samples, streamed in batches
TRAIN_DIAGNOSTIC_SAMPLES=2000

BN_MOMENTUM=0.90
BN_EPSILON=1e-3
BN_RANGE_TOL=2e-6
SHIFTADD_EXP_MIN=-8
SHIFTADD_EXP_MAX=5
SHIFTADD_NEAREST_COEFFS=64
SHIFTADD_SAMPLE_POINTS_PER_CHANNEL=4096
SHIFTADD_BOUND_MARGIN_CODES=12

FRONT_ENDPOINT=2
REPEAT_POLICY="repeat_off"; LOADING_MODE="loading_off"

STAGE_ORDER=tuple(f"stage{i}" for i in range(1,11))
STAGE_SPECS={
    "stage1":  dict(kind="conv", cin=3,   cout=64,  kernel=3, main_rows=64,  main_cols=64,  branches=1,  output_groups=1, bias_in_main=True,  has_post=False),
    "stage2":  dict(kind="conv", cin=64,  cout=64,  kernel=3, main_rows=128, main_cols=128, branches=9,  output_groups=1, bias_in_main=False, has_post=True),
    "stage3":  dict(kind="conv", cin=64,  cout=128, kernel=3, main_rows=128, main_cols=128, branches=9,  output_groups=1, bias_in_main=False, has_post=True),
    "stage4":  dict(kind="conv", cin=128, cout=128, kernel=3, main_rows=128, main_cols=128, branches=18, output_groups=1, bias_in_main=False, has_post=True),
    "stage5":  dict(kind="conv", cin=128, cout=256, kernel=3, main_rows=128, main_cols=128, branches=18, output_groups=2, bias_in_main=False, has_post=True),
    "stage6":  dict(kind="conv", cin=256, cout=256, kernel=3, main_rows=128, main_cols=128, branches=36, output_groups=2, bias_in_main=False, has_post=True),
    "stage7":  dict(kind="conv", cin=256, cout=512, kernel=3, main_rows=128, main_cols=128, branches=36, output_groups=4, bias_in_main=False, has_post=True),
    "stage8":  dict(kind="conv", cin=512, cout=512, kernel=3, main_rows=128, main_cols=128, branches=72, output_groups=4, bias_in_main=False, has_post=True),
    "stage9":  dict(kind="dense",cin=512, cout=512, kernel=None, main_rows=128, main_cols=128, branches=8, output_groups=4, bias_in_main=False, has_post=True),
    "stage10": dict(kind="dense",cin=512, cout=10,  kernel=None, main_rows=128, main_cols=128, branches=8, output_groups=1, bias_in_main=False, has_post=True),
}
STAGE_SEEDS={name:20260910+i for i,name in enumerate(STAGE_ORDER)}

EVENTS=[]; CALIBRATION_ROWS=[]; CHAIN_ROWS=[]; EVAL_ROWS=[]; PHASE_ROWS=[]
BN_AUDIT_ROWS=[]; SHIFTADD_ROWS=[]; INTERFACE_ROWS=[]
GRADIENT_ROWS=[]; VTC_ROWS=[]; CODE_ROWS=[]; TIMING_ROWS=[]; CONVERSION_ROWS=[]
TIMING=dict(train_forward_backward_sec=0.0,validation_selection_sec=0.0,
            train_diagnostic_sec=0.0,hardware_codesign_sec=0.0,
            physical_audit_sec=0.0,final_test_sec=0.0,checkpoint_sec=0.0,diagnostic_sec=0.0)

def record_event(event,**kwargs):
    row={"wall_from_start_sec":time.perf_counter()-RUN_T0,"event":event,**kwargs}
    EVENTS.append(row)
    pd.DataFrame(EVENTS).to_csv(OUT_DIR/"events_live.csv",index=False)
    print("EVENT:",row)

def timed(bucket,fn,*args,**kwargs):
    t=time.perf_counter(); out=fn(*args,**kwargs); TIMING[bucket]+=time.perf_counter()-t; return out

record_event("run_started",artifact=ARTIFACT_VERSION,run_mode=RUN_MODE,seed=SEED,
             adc_bits=ADC_BITS,dac_bits=DAC_BITS)
record_event("inverter_parity_policy",policy="minimum_depth_no_parity_restriction",compensation="single signed-ADC polarity/rail swap after total analog chain")


# Controlled default: one GPU and the same effective training batch as V10.4.
# Seeing two T4 devices does not mean both perform training. No untested distributed BN is enabled.
CAL_BATCH_SIZE=16
STEPS_PER_EXECUTION=16
SHIFTADD_GROUPS=((1,2,3),(4,5,6),(7,8,9))
SHIFTADD_TOTAL_EPOCH_BUDGET=12
INTERMEDIATE_QAT_MAX_EPOCHS=2
TRAIN_SHIFTADD_BIAS=True
BN_ACCUMULATOR_BITS=16
DIAGNOSTIC_SAMPLES=40
VTC_SAMPLE_ROWS=2048
SAVE_OPTIMIZER=True
ONE_SHOT_CONVERSION_CONTROL=True
ROUNDING_CONVENTION="nearest_ties_to_even"
assert RUN_MODE in ("from_scratch","resume")
assert 0 <= FINAL_LOGIT_TEMPERATURE_SHIFT <= 8
if not tf.config.list_physical_devices("GPU"):
    warnings.warn("No GPU detected: enable the Kaggle GPU before the full run.")
CONFIG={k:globals()[k] for k in (
    "RUN_MODE","SEED","BATCH_SIZE","EVAL_BATCH_SIZE","CALIBRATION_SAMPLES",
    "FINAL_LOGIT_TEMPERATURE_SHIFT","STEPS_PER_EXECUTION","SHIFTADD_GROUPS",
    "SHIFTADD_TOTAL_EPOCH_BUDGET","TRAIN_SHIFTADD_BIAS","BN_ACCUMULATOR_BITS",
    "ADC_BITS","DAC_BITS","V_LIMIT","SIG_MIN_US","SIG_MAX_US","LOADING_MODE","REPEAT_POLICY")}
(OUT_DIR/"config.json").write_text(json.dumps(CONFIG,indent=2))
CONFIG['experiment_name']=EXPERIMENT_NAME
CONFIG['base_notebook_sha256']='3727679bd3b0e6e488ac1e798861510c9b51608eb72c7c77df131e54bdea7132'
CONFIG['BN_experiment']={k:globals()[k] for k in ('BN_DIRECT_BIAS', 'BN_LEARN_SCALE', 'BN_REESTIMATE_STATS', 'BN_FIT_ALL_TRAIN', 'BN_FLOAT_FULL_RANGE', 'BN_FROM_FIRST_STEP', 'BN_BIAS_UPDATE_MULTIPLIER', 'BN_SCALE_UPDATE_MULTIPLIER')}
print('EXPERIMENT',EXPERIMENT_NAME)
print("CONFIG:",CONFIG)


# ---------------- V11.0 prespecified curriculum ----------------
PROGRESSIVE_ENDPOINTS=(2,4,6,8,9)
PROGRESSIVE_PLAN={
    2: dict(lr=1.0e-3, patience=12, max_epochs=12),
    4: dict(lr=7.5e-4, patience=3, max_epochs=10),
    6: dict(lr=6.0e-4, patience=3, max_epochs=10),
    8: dict(lr=5.0e-4, patience=3, max_epochs=10),
    9: dict(lr=4.0e-4, patience=3, max_epochs=8),
}
S10_WARMUP_LR=1.0e-3
S10_WARMUP_MAX_EPOCHS=6
FULL_JOINT_LR=3.0e-4
FULL_JOINT_MAX_EPOCHS=15
FLOAT_LOW_LR=1.5e-4
FLOAT_LOW_MAX_EPOCHS=8
FLOAT_LOWER_LR=7.5e-5
FLOAT_LOWER_MAX_EPOCHS=6
V11_SHIFT_QAT_LR=5.0e-5
V11_SHIFT_QAT_MAX_EPOCHS=4
V11_SHIFT_QAT_PATIENCE=2
RUN_FINAL_TRAIN_EVAL=True
MIN_PHASE_GAIN_PP=0.0

for k in ('compile_warmup_sec','setup_sec','benchmark_sec','suffix_training_sec',
          'input_pipeline_build_sec','local_eval_sec','local_matching_to_best_sec',
          'restore_verification_sec'):
    TIMING[k]=0.0
ENVIRONMENT={'python':platform.python_version(),'tensorflow':tf.__version__,'keras':tf.keras.__version__,
             'visible_GPU':[tf.config.experimental.get_device_details(g).get('device_name',g.name) for g in _gpus[:1]],
             'precision':'float32','batch_size':BATCH_SIZE,'multi_GPU_training':False,
             'deterministic':True,'jit_compile':'LUT_only_XLA'}
CONFIG.update({
    'PROGRESSIVE_ENDPOINTS':PROGRESSIVE_ENDPOINTS,
    'PROGRESSIVE_PLAN':PROGRESSIVE_PLAN,
    'S10_WARMUP_LR':S10_WARMUP_LR,
    'S10_WARMUP_MAX_EPOCHS':S10_WARMUP_MAX_EPOCHS,
    'FULL_JOINT_LR':FULL_JOINT_LR,
    'FULL_JOINT_MAX_EPOCHS':FULL_JOINT_MAX_EPOCHS,
    'FLOAT_LOW_LR':FLOAT_LOW_LR,
    'FLOAT_LOW_MAX_EPOCHS':FLOAT_LOW_MAX_EPOCHS,
    'FLOAT_LOWER_LR':FLOAT_LOWER_LR,
    'FLOAT_LOWER_MAX_EPOCHS':FLOAT_LOWER_MAX_EPOCHS,
    'V11_SHIFT_QAT_LR':V11_SHIFT_QAT_LR,
    'V11_SHIFT_QAT_MAX_EPOCHS':V11_SHIFT_QAT_MAX_EPOCHS,
    'RUN_FINAL_TRAIN_EVAL':RUN_FINAL_TRAIN_EVAL,
})
(OUT_DIR/'config.json').write_text(json.dumps(_plain(CONFIG) if '_plain' in globals() else CONFIG,indent=2))
assert RUN_MODE=='from_scratch' and RESUME_CHECKPOINT_PATH is None
print('V11.0 CURRICULUM',PROGRESSIVE_PLAN)
CONFIG['INFO01']={k:globals()[k] for k in (
    'TIMING_WARMUP_BEFORE_PHASE','TIMING_VERIFY_EXACT_RESTORE','VERIFY_RESTORED_VALIDATION',
    'EXP_ENDPOINT2_PROTOCOL','EXP_E2_PLATEAU','EXP_S10_RELATIVE_SCALE','EXP_S10_HEAD_EPOCHS',
    'EXP_RECOVERY_MAX_EPOCHS','EXP_LOCAL_MAX_EPOCHS')}
CONFIG['INFO01']['parent']='INFO00_BN05_CONTROL_DIAGNOSTICS'
(OUT_DIR/'config.json').write_text(json.dumps(CONFIG,indent=2,default=str))


## مدل فیزیکی مشترک

LUT ذخیره‌شده HSPICE V7، مقاومت گسسته و درون‌یابی خطی فقط در محور ولتاژ. loading_off، repeat_off، رسانایی 0.12..8 µS، دامنه ±0.5V و ADC/DAC هشت‌بیتی حفظ شده‌اند.


In [ ]:
# PHAX V12.3.1: measured V7 HSPICE VTC -- no tanh, no parametric fit, no R interpolation.
# This is the source of physical single-inverter voltage transfer data.
# Physical VTC binary is a separate Kaggle Input (not embedded in the notebook source).
import io, zipfile, hashlib


def _find_v7_lut_bytes():
    for root in (Path('/kaggle/input'), Path('/mnt/data'), Path('.')):
        if not root.exists():
            continue
        direct=sorted(root.rglob('PHAX_HSPICE_V7_181x1001_REAL_RAW_LUT.npz'))
        if direct:
            print('External measured LUT:', direct[0]); return direct[0].read_bytes()
        npzs=sorted(root.rglob('full_domain_characterization_v7.npz'))
        if npzs:
            print('External original characterization:', npzs[0]); return npzs[0].read_bytes()
        archives=sorted(p for p in root.rglob('*.zip') if 'CORRECTED_TANH_FULL_DOMAIN_MODELS_V7' in p.name.upper())
        for p in archives:
            with zipfile.ZipFile(p) as z:
                if 'full_domain_characterization_v7.npz' in z.namelist():
                    print('Original V7 characterization ZIP:', p)
                    return z.read('full_domain_characterization_v7.npz')
    raise FileNotFoundError('Missing HSPICE LUT. Kaggle: Add Input -> Upload PHAX_HSPICE_V7_181x1001_REAL_RAW_LUT.npz, then Restart & Run All.')

VTC_NPZ_BYTES=_find_v7_lut_bytes()
VTC_MODEL_SHA256=hashlib.sha256(VTC_NPZ_BYTES).hexdigest()
with np.load(io.BytesIO(VTC_NPZ_BYTES),allow_pickle=False) as measured:
    RAW_VIN=np.asarray(measured['vin_V'],dtype=np.float64).reshape(-1)
    RAW_R_KOHM=np.asarray(measured['R_kohm'],dtype=np.float64).reshape(-1)
    RAW_VTC=np.asarray(measured['raw_vtc_V'],dtype=np.float64).T # shape [R, Vin]
    RAW_SLOPE_ABS=np.asarray(measured['physical_slope_abs'],dtype=np.float64).reshape(-1)
assert RAW_VTC.shape==(len(RAW_R_KOHM),len(RAW_VIN))
assert len(RAW_R_KOHM)==181 and len(RAW_VIN)==1001, 'Unexpected characterization layout'
assert np.max(np.abs(RAW_VTC))<=V_LIMIT+VOLTAGE_TOL
assert np.max(np.diff(RAW_VTC,axis=1))<1e-5, 'Expected monotone decreasing inverter curves'
assert np.all(np.diff(RAW_VIN)>0) and abs(RAW_VIN[0]+.5)<1e-6 and abs(RAW_VIN[-1]-.5)<1e-6

# HSPICE characterization is valid within the actual resistance/rail limits.
# 8,000 kohm is required by the historical V7 physical characterization contract;
# at 0.12 uS one device could reach 8,333 kohm, but that lies outside this V7 practical domain.
PHYSICAL_R_MIN=1.0
PHYSICAL_R_MAX=8000.0
VALID_R_MASK=(RAW_R_KOHM>=PHYSICAL_R_MIN-1e-10)&(RAW_R_KOHM<=PHYSICAL_R_MAX+1e-10)
VALID_R_IDS=np.flatnonzero(VALID_R_MASK)
VALID_R_SLOPE=RAW_SLOPE_ABS[VALID_R_IDS]
GAIN_MIN=float(np.min(VALID_R_SLOPE))
GAIN_MAX=float(np.max(VALID_R_SLOPE))
MODEL_SOURCE_MODE='DISCRETE_SLOPE_SELECTED_RAW_HSPICE_V7_LUT'
print('Direct measured VTC:',len(RAW_R_KOHM),'resistors,',len(RAW_VIN),'voltage points,',
      len(VALID_R_IDS),'resistors physically allowed; gain range',GAIN_MIN,GAIN_MAX)
print('Raw HSPICE checksum SHA256:',VTC_MODEL_SHA256)

# One resistor corresponds to exactly one measured VTC, forever frozen after calibration.
# No mapping neural networks and no on-the-fly interpolation along the resistor axis.
def select_nearest_physical_slope(required):
    g=np.asarray(required,dtype=np.float64)
    if not np.all(np.isfinite(g)) or np.any(g<0): raise ValueError('invalid requested inverter gain')
    nearest=np.argmin(np.abs(g.reshape(-1,1)-VALID_R_SLOPE.reshape(1,-1)),axis=1)
    ix=VALID_R_IDS[nearest].reshape(g.shape)
    return ix


def realize_lut_resistor(index):
    index=int(index)
    if index not in VALID_R_IDS: raise ValueError('selected resistance outside physical V7 range')
    r=float(RAW_R_KOHM[index]); total_g=1000.0/r  # kohm -> uS
    n=max(1,int(math.ceil(total_g/SIG_MAX_US-1e-12)))
    each=total_g/n
    if not (SIG_MIN_US-1e-9 <= each <= SIG_MAX_US+1e-9):
        raise ValueError(f'R={r} kohm not realizable within memristor bounds')
    return {'target_r_kohm':r,'actual_r_kohm':r,'lut_index':index,
            'memristor_count':n,'memristor_g_each_us':each,'explicit_g_us':total_g}

# A single flat 181*1001 VTC array is shared by all stages in TensorFlow.
# Trainable theta/G still receive exact autodiff of the local PWL segment.
RAW_VIN_TF=tf.constant(RAW_VIN.astype(np.float32),dtype=tf.float32)
RAW_VTC_TF_FLAT=tf.constant(RAW_VTC.astype(np.float32).reshape(-1),dtype=tf.float32)
LUT_NV=len(RAW_VIN)
LUT_INVSTEP=float((LUT_NV-1)/(RAW_VIN[-1]-RAW_VIN[0]))




# ---------- FAST VTC: precomputed exact piecewise coefficients; ONE GPU gather ----------
# Convert each ORIGINAL adjacent measured HSPICE pair to a line segment ONCE.
# No tanh, neural fitting, resistor interpolation, or voltage-grid approximation.
# Original VTC knots remain available unchanged for verification and diagnostics.
_VTC_DX=np.diff(RAW_VIN.astype(np.float64))
_FAST_LUT_SLOPE=np.diff(RAW_VTC.astype(np.float64),axis=1)/_VTC_DX[None,:]
_FAST_LUT_INTERCEPT=RAW_VTC[:,:-1]-_FAST_LUT_SLOPE*RAW_VIN[:-1][None,:]
# Each segment: y(x) = slope*x + intercept.
# 181 resistors * 1000 voltage segments * 2 coefficients = about 1.45 MB float32.
_FAST_LUT_SEGMENTS=np.stack([_FAST_LUT_SLOPE,_FAST_LUT_INTERCEPT],axis=-1).astype(np.float32)
FAST_LUT_TABLE_TF=tf.constant(_FAST_LUT_SEGMENTS.reshape(-1,2),dtype=tf.float32)
FAST_LUT_GRID_LEFT=float(RAW_VIN[0])
FAST_LUT_GRID_INVSTEP=float((LUT_NV-1)/(RAW_VIN[-1]-RAW_VIN[0]))
FAST_LUT_MODE='HSPICE_EXACT_SEGMENTS_XLA_JIT_SINGLE_GATHER_FLOAT32'


# The old 1-gather implementation remains only as a numerical reference/benchmark.
# During actual training the LUT uses the isolated JIT-compiled XLA kernel below.
def _lut_eval_tensor_single_gather(x,index_per_column):
    x=tf.cast(x,tf.float32)
    ri=tf.constant(np.asarray(index_per_column,dtype=np.int32),dtype=tf.int32)
    if ri.shape.rank!=1:raise ValueError('one discrete LUT resistor per column required')
    if x.shape[-1] is not None and int(x.shape[-1])!=int(ri.shape[0]):
        raise ValueError('inverter channel count differs from assigned hardware')
    xc=tf.clip_by_value(x,FAST_LUT_GRID_LEFT,float(RAW_VIN[-1]))
    j=tf.clip_by_value(tf.cast(tf.floor((xc-FAST_LUT_GRID_LEFT)*FAST_LUT_GRID_INVSTEP),tf.int32),0,LUT_NV-2)
    a=tf.gather(FAST_LUT_TABLE_TF,j+ri*(LUT_NV-1),axis=0)
    return a[...,0]*xc+a[...,1]

@tf.function(jit_compile=True,reduce_retracing=True)
def _lut_xla_core(x,ri):
    # Keep R indices as an argument, rather than closing over thousands of
    # differently shaped constants: XLA compiles for tensor shapes, not R values.
    xc=tf.clip_by_value(tf.cast(x,tf.float32),FAST_LUT_GRID_LEFT,float(RAW_VIN[-1]))
    j=tf.clip_by_value(tf.cast(tf.floor((xc-FAST_LUT_GRID_LEFT)*FAST_LUT_GRID_INVSTEP),tf.int32),0,LUT_NV-2)
    coeff=tf.gather(FAST_LUT_TABLE_TF,j+tf.cast(ri,tf.int32)*(LUT_NV-1),axis=0)
    return coeff[...,0]*xc+coeff[...,1]


def _lut_eval_tensor(x,index_per_column):
    # The only VTC used by the CNN train/eval/calibration graph.
    # GradientTape differentiates through the compiled XLA function.
    x=tf.cast(x,tf.float32)
    ri=tf.constant(np.asarray(index_per_column,dtype=np.int32),dtype=tf.int32)
    if ri.shape.rank!=1:raise ValueError('one discrete LUT resistor per column required')
    if x.shape[-1] is not None and int(x.shape[-1])!=int(ri.shape[0]):
        raise ValueError('inverter channel count differs from assigned hardware')
    return _lut_xla_core(x,ri)


def _lut_eval_numpy(x,index_per_column,derivative=False):
    x=np.asarray(x,dtype=np.float64)
    ri=np.asarray(index_per_column,dtype=np.int32)
    if x.shape[-1]!=len(ri):raise ValueError('bad LUT channel shape')
    if np.max(np.abs(x))>V_LIMIT+VOLTAGE_TOL:raise ValueError('inverter LUT input beyond rails')
    xc=np.clip(x,float(RAW_VIN[0]),float(RAW_VIN[-1]))
    u=(xc-float(RAW_VIN[0]))*LUT_INVSTEP
    j=np.clip(np.floor(u).astype(np.int32),0,LUT_NV-2)
    j=np.where(xc<RAW_VIN[j],np.maximum(j-1,0),j)
    j=np.where(xc>=RAW_VIN[j+1],np.minimum(j+1,LUT_NV-2),j)
    vl=RAW_VIN[j]; vr=RAW_VIN[j+1]
    y0=RAW_VTC[ri,j]; y1=RAW_VTC[ri,j+1]
    if derivative: return (y1-y0)/(vr-vl)
    return y0+(xc-vl)*(y1-y0)/(vr-vl)

# No fitted tanh, Chebyshev residual, interpolated R, or neural mapping in the physical forward.
CONFIG.update(vtc_mode=MODEL_SOURCE_MODE,real_HSPICE_LUT_used=True,
              vtc_sampled_R_count=int(len(RAW_R_KOHM)),vtc_realizable_R_count=int(len(VALID_R_IDS)),
              vtc_sampled_Vin_count=int(LUT_NV),vtc_data_sha256=VTC_MODEL_SHA256,
              vtc_R_policy='nearest_discrete_measured_physical_slope',
              vtc_forward='measured_PWL_single_gather_XLA_precomputed_segment_coefficients')
CONFIG.update(lut_gpu_optimization=FAST_LUT_MODE,lut_stage_lookup_ops=1,
              lut_xla_isolated_compilation=True,
              lut_coefficients_derived_from_original_HSPICE_knots=True)
(OUT_DIR/'config.json').write_text(json.dumps(CONFIG,indent=2,default=str))

# Hardware slope table as a standalone verification artifact.
pd.DataFrame({'R_index':np.arange(len(RAW_R_KOHM)),
              'R_kohm':RAW_R_KOHM,'slope_abs':RAW_SLOPE_ABS,
              'physical_R_allowed':VALID_R_MASK.astype(int),
              'conductance_equivalent_uS':1000.0/RAW_R_KOHM}).to_csv(
    OUT_DIR/'measured_hspice_R_slope_table.csv',index=False)


In [ ]:
# ========================= MINIMUM COMMON DEPTH; EXACT DISCRETE SLOPE->R =========================
# Retain the original V12.3.1 minimum-depth policy, patterns, and objective.
# Replace ONLY tanh-based gain-to-R mapping and VTC evaluation with measured HSPICE entries.

def _chain_polarity(depth):
    return -1 if int(depth)%2 else 1


def _split_patterns(depth):
    pats=[np.zeros(depth,np.float64)]
    if depth<=1:return pats
    last=depth-1
    for dlt in GAIN_SPLIT_DELTA:
        for i in range(depth-1):
            p=np.zeros(depth);p[i]=dlt;p[last]-=dlt;pats.append(p)
            p=np.zeros(depth);p[i]=-dlt;p[last]+=dlt;pats.append(p)
    p=np.array([.35 if i%2==0 else -.35 for i in range(depth)],np.float64);p-=np.mean(p)
    pats.extend([p,-p]);out=[];seen=set()
    for p in pats:
        key=tuple(np.round(p,6))
        if key not in seen:seen.add(key);out.append(p)
    return out


def cascade_np(x,spec,return_nodes=False):
    y=np.asarray(x,np.float64);nodes=[]
    for ids in spec['lut_index_levels']:
        y=_lut_eval_numpy(y,ids)
        if return_nodes:nodes.append(y.copy())
    return (y,nodes) if return_nodes else y


def cascade_tf(x,spec,return_nodes=False):
    y=tf.cast(x,tf.float32);nodes=[]
    for ids in spec['lut_index_levels']:
        y=_lut_eval_tensor(y,ids)
        if return_nodes:nodes.append(y)
    return (y,nodes) if return_nodes else y


def cascade_interval_tf(lower,upper,spec):
    lo=tf.cast(lower,tf.float32);hi=tf.cast(upper,tf.float32)
    for ids in spec['lut_index_levels']:
        flo=_lut_eval_tensor(lo,ids)
        fhi=_lut_eval_tensor(hi,ids)
        lo=tf.minimum(flo,fhi);hi=tf.maximum(flo,fhi)
    return lo,hi


def _per_column_max_abs(raw):
    a=np.asarray(raw,np.float64)
    return np.max(np.abs(a),axis=tuple(range(a.ndim-1)))


def _scalar_chain_eval(x,r_ids):
    y=np.asarray(x,np.float64)
    for idx in r_ids:
        y=np.interp(y,RAW_VIN,RAW_VTC[int(idx)])
    return y


def _scalar_full_domain_node_max(r_ids):
    lo=-V_LIMIT;hi=V_LIMIT;mx=0.0
    for idx in r_ids:
        f0=float(np.interp(lo,RAW_VIN,RAW_VTC[int(idx)]))
        f1=float(np.interp(hi,RAW_VIN,RAW_VTC[int(idx)]))
        lo,hi=min(f0,f1),max(f0,f1)
        mx=max(mx,abs(lo),abs(hi))
        if mx>V_LIMIT+VOLTAGE_TOL:return mx
    return mx


def _column_candidate(max_abs,depth,offsets,target_gain_limit=None):
    max_abs=float(max(max_abs,1e-9));pol=_chain_polarity(depth)
    requested=V_LIMIT/max_abs
    capacity=(GAIN_MAX*GAIN_EDGE_MARGIN)**depth
    limited=requested if target_gain_limit is None else min(requested,float(target_gain_limit))
    total=min(limited,capacity)
    per=np.exp(math.log(max(total,1e-12))/depth+np.asarray(offsets,np.float64))
    if np.any(per<GAIN_MIN) or np.any(per>GAIN_MAX*1.001):return None
    ids=select_nearest_physical_slope(per).tolist()
    loads=[realize_lut_resistor(ix) for ix in ids]
    fullmx=_scalar_full_domain_node_max(ids)
    if fullmx>V_LIMIT+VOLTAGE_TOL:return None
    x=np.linspace(-max_abs,max_abs,CHAIN_BALANCED_POINTS)
    y=_scalar_chain_eval(x,ids)
    target=pol*total*x
    span=max(float(np.ptp(target)),1e-9)
    err=float(np.sqrt(np.mean((y-target)**2))/span)
    util=float(np.max(np.abs(y))/V_LIMIT)
    y0=float(_scalar_chain_eval([0],ids)[0])
    mem=sum(z['memristor_count'] for z in loads)
    score=err+0.30*max(0.0,1.0-util)+2e-5*mem+0.02*abs(y0)/V_LIMIT
    true_slope=float(np.prod(RAW_SLOPE_ABS[np.asarray(ids,np.int32)]))
    true_secant=float(abs((y[-1]-y[0])/(2*max_abs)))
    return dict(ids=ids,rseq=[z['actual_r_kohm'] for z in loads],loads=loads,
                nrmse=err,util=util,score=score,requested_gain=requested,target_gain=total,
                actual_center_slope_product=true_slope,actual_secant_gain=true_secant,
                dc_offset_v=y0,gain_limited=bool(total<requested*(1-1e-12)),
                full_domain_node_abs_max_v=fullmx)


def _robust_initial_depth_plan(maxv,enabled=False):
    req=np.asarray([V_LIMIT/max(float(v),1e-9) for v in maxv],np.float64)
    C=len(req)
    info={'enabled_requested':bool(enabled),'active':False,'columns':int(C),
          'requested_p50':float(np.percentile(req,50)),
          'requested_p95':float(np.percentile(req,ROBUST_INIT_P_LO)),
          'requested_p99':float(np.percentile(req,ROBUST_INIT_P_HI)),
          'requested_max':float(np.max(req)),'depth_reference_gain':float(np.max(req))}
    if not enabled or not ROBUST_INIT_DEPTH_REFERENCE or C<ROBUST_INIT_MIN_COLUMNS:
        return req,float(np.max(req)),info
    p95=info['requested_p95'];p99=info['requested_p99']
    depth_ref=float(min(p99,ROBUST_INIT_P95_MULT*p95))
    depth_ref=max(depth_ref,p95)
    if info['requested_max']<=depth_ref*ROBUST_INIT_ACTIVATION_MARGIN:
        return req,float(np.max(req)),info
    info.update(active=True,depth_reference_gain=depth_ref)
    return req,depth_ref,info


def _minimum_gain_sufficient_depth_from_reference(g,allowed_depths):
    req=float(g);ordered=tuple(sorted(int(d) for d in allowed_depths))
    for d in ordered:
        if (GAIN_MAX*GAIN_EDGE_MARGIN)**d>=req*(1-1e-12):return d,req,False
    return ordered[-1],req,True


def synthesize_common_depth_per_column(raw,label,allowed_depths,robust_initial=False):
    maxv=_per_column_max_abs(raw);C=len(maxv)
    requested,depth_ref,robust=_robust_initial_depth_plan(maxv,enabled=robust_initial)
    start_depth,depth_ref_used,depth_limited=_minimum_gain_sufficient_depth_from_reference(depth_ref,allowed_depths)
    requested_max=float(np.max(requested));ordered=tuple(sorted(int(d) for d in allowed_depths))
    selected=None
    # This is NOT a performance optimization over extra stages: use the first
    # physically feasible MINIMUM depth, exactly as in the baseline controller.
    for depth in (d for d in ordered if d>=start_depth):
        capacity=float((GAIN_MAX*GAIN_EDGE_MARGIN)**depth)
        pats=_split_patterns(depth);chosen=[]
        for j in range(C):
            candidates=[]
            limit=min(float(requested[j]),capacity)
            for p in pats:
                cand=_column_candidate(maxv[j],depth,p,target_gain_limit=limit)
                if cand is not None:candidates.append(cand)
            if not candidates:chosen=None;break
            chosen.append(min(candidates,key=lambda z:z['score']))
        if chosen is not None:
            selected=(depth,chosen);break
    if selected is None:raise RuntimeError(label+': cannot synthesize a feasible minimum-depth measured-VTC chain')
    depth,chosen=selected
    capacity=float((GAIN_MAX*GAIN_EDGE_MARGIN)**depth)
    util=np.asarray([c['util'] for c in chosen]);err=np.asarray([c['nrmse'] for c in chosen])
    mem=sum(sum(z['memristor_count'] for z in c['loads']) for c in chosen)
    stage_score=float(np.median(err)+.45*np.percentile(err,90)+.35*(1-np.percentile(util,10))
                      +.10*(1-np.median(util))+1e-7*mem)
    indices=[[int(chosen[j]['ids'][l]) for j in range(C)] for l in range(depth)]
    levels=[[float(RAW_R_KOHM[ix]) for ix in layer] for layer in indices]
    counts=[[int(chosen[j]['loads'][l]['memristor_count']) for j in range(C)] for l in range(depth)]
    limitmask=np.asarray([v>capacity*(1+1e-12) for v in requested],bool)
    spec={'label':label,'n_inv':int(depth),'polarity':int(_chain_polarity(depth)),
          'R_kohm_levels':levels,'lut_index_levels':indices,'memristor_count_levels':counts,
          'column_calibration_abs_max_v':[float(v) for v in maxv],
          'column_target_gain':[float(c['target_gain']) for c in chosen],
          'column_requested_gain':[float(c['requested_gain']) for c in chosen],
          'column_actual_center_slope_gain':[float(c['actual_center_slope_product']) for c in chosen],
          'column_actual_secant_gain':[float(c['actual_secant_gain']) for c in chosen],
          'column_gain_limited_by_depth_capacity':limitmask.tolist(),
          'column_nrmse':err.tolist(),'column_utilization':util.tolist(),
          'median_utilization':float(np.median(util)),
          'p10_utilization':float(np.percentile(util,10)),
          'median_nrmse':float(np.median(err)),'p90_nrmse':float(np.percentile(err,90)),
          'memristors_per_branch_all_columns':int(mem),'selection_score':float(stage_score),
          'depth_policy':'robust_reference_for_depth__per_column_full_capacity' if robust['active'] else 'minimum_gain_sufficient',
          'gain_required_max':requested_max,'gain_target_max':float(np.max([c['target_gain'] for c in chosen])),
          'gain_capacity_selected':capacity,'gain_limited_by_max_depth':bool(depth_limited),
          'robust_initial_depth_policy':copy.deepcopy(robust),
          'safe_for_every_input_in_pm_0p5':True,
          'full_domain_node_abs_max_v':float(max(c['full_domain_node_abs_max_v'] for c in chosen)),
          'physical_model':MODEL_SOURCE_MODE}
    brief={k:spec[k] for k in ('n_inv','gain_required_max','gain_target_max','gain_capacity_selected',
                                'median_utilization','p10_utilization','median_nrmse',
                                'p90_nrmse','memristors_per_branch_all_columns')}
    brief.update(depth_robust_active=bool(robust['active']),
                 depth_reference_gain=float(robust['depth_reference_gain']),
                 capacity_limited_columns=int(np.sum(limitmask)),
                 capacity_limited_fraction=float(np.mean(limitmask)))
    print('COLUMN_CHAIN_DIRECT_HSPICE:',label,brief)
    CHAIN_ROWS.append({'label':label,**brief,'gain_limited_by_max_depth':bool(depth_limited),
                       'selection_score':float(stage_score)})
    return spec


In [ ]:
_t_bench=time.perf_counter()
# Independent reference LUT (for validation and benchmarking only).
# Declared in this cell, so it cannot be missing when this cell runs after the
# preceding hardware/LUT initialization cell. Training uses _lut_eval_tensor FAST.
def _lut_eval_tensor_reference(x,index_per_column):
    x=tf.cast(x,tf.float32)
    ri=tf.constant(np.asarray(index_per_column,dtype=np.int32),tf.int32)
    if ri.shape.rank != 1: raise ValueError('one fixed LUT resistance index per column required')
    if x.shape[-1] is not None and int(x.shape[-1])!=int(ri.shape[0]):
        raise ValueError('bad inverter output channel count')
    # There is no intentional physical clipping: values beyond VDD raise in audits.
    # The clip below only handles possible ±single-ulp rounding at a characterized rail.
    xc=tf.clip_by_value(x,float(RAW_VIN[0]),float(RAW_VIN[-1]))
    u=(xc-float(RAW_VIN[0]))*LUT_INVSTEP
    j=tf.clip_by_value(tf.cast(tf.floor(u),tf.int32),0,LUT_NV-2)
    # Check actual sample locations: original MATLAB voltage grid has tiny rounding jitter.
    j=tf.where(xc<tf.gather(RAW_VIN_TF,j),tf.maximum(j-1,0),j)
    j=tf.where(xc>=tf.gather(RAW_VIN_TF,j+1),tf.minimum(j+1,LUT_NV-2),j)
    va=tf.gather(RAW_VIN_TF,j); vb=tf.gather(RAW_VIN_TF,j+1)
    a=(xc-va)/(vb-va)
    address=j+ri*LUT_NV
    ya=tf.gather(RAW_VTC_TF_FLAT,address)
    yb=tf.gather(RAW_VTC_TF_FLAT,address+1)
    return ya+(yb-ya)*a

# Sanity tests BEFORE expensive model training. No tanh or fitted surrogate.
assert len(VALID_R_IDS)==171
check_ids=np.asarray([VALID_R_IDS[0],VALID_R_IDS[len(VALID_R_IDS)//2],VALID_R_IDS[-1]],np.int32)
xk=np.broadcast_to(RAW_VIN[:,None],(LUT_NV,len(check_ids))).copy()
ye=_lut_eval_numpy(xk,check_ids)
assert np.max(np.abs(ye-RAW_VTC[check_ids].T))<2e-14
assert np.all(np.diff(ye,axis=0)<1e-6)
# Verify that selected resistor is truly the closest characterized local slope.
for g in [0.5,1.0,2.0,4.0,7.0]:
    got=int(select_nearest_physical_slope(np.asarray([g]))[0])
    assert abs(RAW_SLOPE_ABS[got]-g)<=np.min(abs(VALID_R_SLOPE-g))+1e-12
# One channel / one selected measured table, with a differentiable gradient.
x=tf.Variable(np.array([[0.003,0.011,-0.006]],np.float32))
with tf.GradientTape() as tape:
    y=_lut_eval_tensor(x,check_ids)
    scalar_test_loss=tf.reduce_sum(y)
dy=tape.gradient(scalar_test_loss,x)
assert dy is not None and bool(tf.reduce_all(tf.math.is_finite(dy)))
expected=_lut_eval_numpy(x.numpy().astype(np.float64),check_ids)
assert np.max(np.abs(y.numpy()-expected))<2e-6
print('Direct HSPICE LUT + XLA knot check, resistor selection and GradientTape: PASS')

# Verify FAST lookup against the SLOW REFERENCE, including near nonlinear switching.
# Reference exists only for this diagnostic; training always uses the fast path.
fast_rng=np.random.default_rng(SEED+190)
fast_test_x=tf.constant(fast_rng.uniform(-.5,.5,size=(4,16,16,3)).astype(np.float32))
ref=tf.cast(_lut_eval_tensor_reference(fast_test_x,check_ids),tf.float32)
new=tf.cast(_lut_eval_tensor(fast_test_x,check_ids),tf.float32)
abs_diff=float(tf.reduce_max(tf.abs(ref-new)).numpy())
print('XLA vs raw-voltage reference max forward error (V):',abs_diff)
assert abs_diff<=2e-6, 'Fast LUT differs from original beyond physical numerical tolerance'
with tf.GradientTape(persistent=True) as gp:
    xx=tf.Variable(fast_test_x)
    yr=_lut_eval_tensor_reference(xx,check_ids)
    yf=_lut_eval_tensor(xx,check_ids)
    lr=tf.reduce_sum(yr);lf=tf.reduce_sum(yf)
gr=gp.gradient(lr,xx);gf=gp.gradient(lf,xx);del gp
assert gr is not None and gf is not None
assert bool(tf.reduce_all(tf.math.is_finite(gf)))
# Gradients near measured knots can legitimately differ by an adjacent-segment choice;
# away from such knots they must be consistent.
mask=tf.abs((fast_test_x-FAST_LUT_GRID_LEFT)*FAST_LUT_GRID_INVSTEP -
            tf.round((fast_test_x-FAST_LUT_GRID_LEFT)*FAST_LUT_GRID_INVSTEP)) > 0.003
if bool(tf.reduce_any(mask)):
    max_grad_err=float(tf.reduce_max(tf.abs(tf.boolean_mask(gr-gf,mask))).numpy())
    print('XLA vs raw-voltage reference max local-slope difference away from knots:',max_grad_err)
    assert max_grad_err<0.015, 'Unexpected gradient discrepancy; inspect LUT interval indexing'
# Direct reference: at every original measured voltage knot, the value is preserved.
# For float32 and the tiny nonuniformity of the original MAT grid, use 1.5 uV tolerance.
knot_test=np.broadcast_to(RAW_VIN[:,None].astype(np.float32),(LUT_NV,3))
knot_ys=_lut_eval_tensor(tf.constant(knot_test),check_ids).numpy()
knot_error=float(np.max(np.abs(knot_ys-RAW_VTC[check_ids].T)))
print('FAST vs raw HSPICE measured knots max error (V):',knot_error)
assert knot_error<1.5e-6


# A truly trainable proxy conductance must receive finite *nonzero* gradient
# THROUGH the compiled physical LUT, not merely with respect to its input x.
phi=tf.Variable(tf.zeros([2,3],dtype=tf.float32))
probe_x=tf.constant([[1.0,0.2]],dtype=tf.float32)
with tf.GradientTape() as t:
    proxy_G=tf.sigmoid(phi)  # strictly monotone proxy for theta->conductance
    probe_raw=0.02*tf.matmul(probe_x,proxy_G)
    probe_vtc=_lut_eval_tensor(probe_raw,check_ids)
    proxy_loss=tf.reduce_sum(probe_vtc)
proxy_grad=t.gradient(proxy_loss,phi)
assert proxy_grad is not None and bool(tf.reduce_all(tf.math.is_finite(proxy_grad)))
assert float(tf.reduce_max(tf.abs(proxy_grad)).numpy())>1e-9, 'LUT must propagate gradient to trainable theta'
print('XLA gradient reaches trainable conductance: PASS; max |dL/dtheta| =',float(tf.reduce_max(tf.abs(proxy_grad)).numpy()))

# Isolated, synchronized benchmark: compare reference / single-gather / XLA
# without timing the first compilation in the steady-state millisecond figures.
BENCH_IDS=np.asarray(VALID_R_IDS[np.linspace(0,len(VALID_R_IDS)-1,64,dtype=int)],np.int32)

def _make_lut_fb(fn):
    @tf.function(reduce_retracing=True)
    def work(z):
        with tf.GradientTape() as tape:
            tape.watch(z)
            vv=fn(z,BENCH_IDS)
            scalar=tf.reduce_sum(vv)
        dz=tape.gradient(scalar,z)
        return scalar+tf.reduce_sum(dz)  # host .numpy() synchronizes the GPU
    return work

def _time_synced(work,z,loops=8,warmups=3):
    start=time.perf_counter()
    first=float(work(z).numpy())
    first_s=time.perf_counter()-start
    for _ in range(warmups): _=float(work(z).numpy())
    start=time.perf_counter()
    for _ in range(loops): _=float(work(z).numpy())
    return dict(first_compile_and_execution_s=round(first_s,3),steady_forward_backward_ms=round(1000*(time.perf_counter()-start)/loops,4),last_result=first)

XLA_LUT_BENCHMARK=[]
for workload,shape,iterations in [('medium',(8,16,16,64),10),('large',(32,32,32,64),6)]:
    rng=np.random.default_rng(SEED+190)
    z=tf.convert_to_tensor(rng.uniform(-.35,.35,size=shape).astype(np.float32))
    for label,fn in [('TF LUT reference',_lut_eval_tensor_reference),('TF LUT single-gather',_lut_eval_tensor_single_gather),('TF LUT XLA',_lut_eval_tensor)]:
        w=_make_lut_fb(fn)
        result=_time_synced(w,z,loops=iterations)
        XLA_LUT_BENCHMARK.append(dict(workload=workload,method=label,**result))
        print('XLA BENCH',workload,label,result['steady_forward_backward_ms'],'ms',
              '(compile+first',result['first_compile_and_execution_s'],'s)')
    del z
pd.DataFrame(XLA_LUT_BENCHMARK).drop(columns=['last_result']).to_csv(OUT_DIR/'lut_xla_microbenchmark.csv',index=False)
CONFIG['lut_gpu_benchmark']= [{k:v for k,v in row.items() if k!='last_result'} for row in XLA_LUT_BENCHMARK]
(OUT_DIR/'config.json').write_text(json.dumps(CONFIG,indent=2,default=str))
print('XLA correctness & isolated benchmark: PASS. Full VGG9 training follows.')

TIMING['benchmark_sec']+=time.perf_counter()-_t_bench


In [ ]:
# ================== CIFAR-10 INPUT & DATA SPLIT (RESTORED) ==================
# This block was inadvertently omitted in the first LUT notebook.
# ========================= CIFAR-10 =========================
def find_cifar10_batches_dir():
    candidates=[]
    for root in [Path("/kaggle/input"),Path("/mnt/data")]:
        if not root.exists(): continue
        for p in root.rglob("data_batch_1"):
            if all((p.parent/f"data_batch_{i}").exists() for i in range(1,6)) and (p.parent/"test_batch").exists():
                candidates.append(p.parent)
    if not candidates: raise FileNotFoundError("CIFAR-10 Python batches not found")
    return sorted(candidates,key=lambda p:len(str(p)))[0]

def load_cifar_batch(path):
    import pickle
    with open(path,"rb") as f: d=pickle.load(f,encoding="bytes")
    x=d[b"data"].reshape(-1,3,32,32).transpose(0,2,3,1)
    y=np.asarray(d.get(b"labels",d.get(b"fine_labels")),np.int64)
    return x,y

def load_cifar10_train_only():
    root=find_cifar10_batches_dir(); xs=[]; ys=[]
    for i in range(1,6):
        x,y=load_cifar_batch(root/f"data_batch_{i}"); xs.append(x); ys.append(y)
    return np.concatenate(xs),np.concatenate(ys)

def balanced_indices(y,n,seed):
    rng=np.random.default_rng(seed); classes=np.unique(y); each=n//len(classes); rem=n%len(classes); out=[]
    for k,c in enumerate(classes):
        idx=np.flatnonzero(y==c); take=each+(k<rem); out.extend(rng.choice(idx,take,replace=False))
    return rng.permutation(np.asarray(out,np.int64))

def augment_stateless(x,y,seed):
    x=tf.image.resize_with_crop_or_pad(x,40,40)
    x=tf.image.stateless_random_crop(x,[32,32,3],seed)
    x=tf.image.stateless_random_flip_left_right(x,seed+tf.constant([7,13],tf.int32))
    return x,y

def _train_pipeline(x,y,epoch_seed,batch):
    ds=tf.data.Dataset.from_tensor_slices((x,y)).shuffle(len(x),seed=epoch_seed,reshuffle_each_iteration=False)
    ds=ds.enumerate().map(lambda i,xy: augment_stateless(xy[0],xy[1],tf.stack([tf.cast(epoch_seed,tf.int32),tf.cast(i,tf.int32)])),num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def make_train_ds(epoch_seed,batch):
    # Identical op sequence to INFO00; the shared pipeline also builds warm-up batches.
    return _train_pipeline(x_train,y_train,epoch_seed,batch)

def make_eval_ds(x,y,batch):
    return tf.data.Dataset.from_tensor_slices((x,y)).batch(batch).prefetch(tf.data.AUTOTUNE)

_t_data=time.perf_counter()
x_all,y_all=load_cifar10_train_only()
x_all=x_all.astype(np.float32)/255.0-0.5
rng=np.random.default_rng(SEED); perm=rng.permutation(len(x_all)); val_idx=perm[:VAL_SIZE]; tr_idx=perm[VAL_SIZE:]
x_val=x_all[val_idx]; y_val=y_all[val_idx]; x_train=x_all[tr_idx]; y_train=y_all[tr_idx]
cal_local=balanced_indices(y_train,CALIBRATION_SAMPLES,SEED+77); x_cal=x_train[cal_local]; y_cal=y_train[cal_local]
SPLIT_FINGERPRINT=hashlib.sha256(np.asarray(perm,np.int64).tobytes()).hexdigest()
print("train/val/cal:",len(x_train),len(x_val),len(x_cal),"split",SPLIT_FINGERPRINT)


diag_local=balanced_indices(y_train,DIAGNOSTIC_SAMPLES,SEED+991)
x_diag,y_diag=x_train[diag_local],y_train[diag_local]
probe_local=balanced_indices(y_train,20,SEED+992)
x_probe,y_probe=x_train[probe_local],y_train[probe_local]
TIMING['setup_sec']+=time.perf_counter()-_t_data


In [ ]:
# ========================= PHAX STAGES + QUANTIZERS =========================
def conductance(theta): return SIG_MIN_US+(SIG_MAX_US-SIG_MIN_US)*tf.sigmoid(theta)
def symmetric_drive(x): x=tf.cast(x,tf.float32); return x,-x

def _signed_adc_code_tf(v):
    return tf.round(tf.cast(v,tf.float32)/V_LIMIT*float(ADC_SIGNED_QMAX))

def signed_adc_qdq_value(v): return _signed_adc_code_tf(v)/float(ADC_SIGNED_QMAX)*V_LIMIT

@tf.custom_gradient
def signed_adc_qdq_ste(v):
    v=tf.cast(v,tf.float32)
    chk=tf.debugging.assert_less_equal(tf.reduce_max(tf.abs(v)),V_LIMIT+VOLTAGE_TOL)
    with tf.control_dependencies([chk]): q=tf.identity(signed_adc_qdq_value(v))
    return q,lambda dy:tf.cast(dy,tf.float32)

@tf.custom_gradient
def signed_adc_code_ste(v):
    v=tf.cast(v,tf.float32)
    chk=tf.debugging.assert_less_equal(tf.reduce_max(tf.abs(v)),V_LIMIT+VOLTAGE_TOL)
    with tf.control_dependencies([chk]): q=tf.identity(_signed_adc_code_tf(v))
    return q,lambda dy:tf.cast(dy,tf.float32)*(float(ADC_SIGNED_QMAX)/V_LIMIT)

def final_adc_code_logits(v): return signed_adc_code_ste(v)/float(1<<FINAL_LOGIT_TEMPERATURE_SHIFT)

def _unsigned_dac_code_tf(x): return tf.round(tf.cast(x,tf.float32)/V_LIMIT*float(DAC_UNSIGNED_QMAX))
def unsigned_dac_qdq_value(x): return _unsigned_dac_code_tf(x)/float(DAC_UNSIGNED_QMAX)*V_LIMIT

@tf.custom_gradient
def unsigned_dac_qdq_ste(x):
    x=tf.cast(x,tf.float32)
    c1=tf.debugging.assert_greater_equal(tf.reduce_min(x),-VOLTAGE_TOL)
    c2=tf.debugging.assert_less_equal(tf.reduce_max(x),V_LIMIT+VOLTAGE_TOL)
    with tf.control_dependencies([c1,c2]): q=tf.identity(unsigned_dac_qdq_value(x))
    return q,lambda dy:tf.cast(dy,tf.float32)

@tf.custom_gradient
def unsigned_dac_from_code_ste(q):
    q=tf.cast(q,tf.float32)
    c1=tf.debugging.assert_greater_equal(tf.reduce_min(q),-1e-5)
    c2=tf.debugging.assert_less_equal(tf.reduce_max(q),float(DAC_UNSIGNED_QMAX)+1e-5)
    with tf.control_dependencies([c1,c2]):
        qr=tf.round(q); v=qr/float(DAC_UNSIGNED_QMAX)*V_LIMIT
    def grad(dy): return tf.cast(dy,tf.float32)*(V_LIMIT/float(DAC_UNSIGNED_QMAX))
    return v,grad

def stage_index(stage): return int(stage.replace("stage",""))

def aggregator_design(stage):
    s=STAGE_SPECS[stage]
    if stage=="stage1": return None
    empty=128-s["branches"]-2
    return {"repeat":1,"empty_rows":int(empty),"g_signal_us":np.full((s["branches"],s["cout"]),SIG_MAX_US,np.float32),
            "signal_cell_g_us":SIG_MAX_US,"empty_cell_g_us":SIG_MIN_US,"bias_input_polarity":1.0}

class VGGPHAXStage(tf.keras.layers.Layer):
    def __init__(self,stage):
        super().__init__(name=stage); self.stage=stage; self.spec=copy.deepcopy(STAGE_SPECS[stage]); s=self.spec
        rng=np.random.default_rng(STAGE_SEEDS[stage])
        if stage=="stage1": shape=(27,s["cout"])
        elif s["kind"]=="conv": shape=(9,s["cin"]//64,64,s["cout"])
        else: shape=(s["cin"]//64,64,s["cout"])
        self.theta_wp=self.add_weight(name="theta_wp",shape=shape,initializer=tf.constant_initializer(rng.normal(0,.2,shape).astype(np.float32)),trainable=True)
        self.theta_wn=self.add_weight(name="theta_wn",shape=shape,initializer=tf.constant_initializer(rng.normal(0,.2,shape).astype(np.float32)),trainable=True)
        self.theta_bp=self.add_weight(name="theta_bp",shape=(s["cout"],),initializer="zeros",trainable=True)
        self.theta_bn=self.add_weight(name="theta_bn",shape=(s["cout"],),initializer="zeros",trainable=True)
        self.hardware=None

    def raw_main(self,x):
        s=self.spec; xp,xn=symmetric_drive(x); sp,sn=conductance(self.theta_wp),conductance(self.theta_wn); bp,bn=conductance(self.theta_bp),conductance(self.theta_bn)
        if self.stage=="stage1":
            p=tf.image.extract_patches(xp,[1,3,3,1],[1,1,1,1],[1,1,1,1],"SAME"); pn=-p
            den=tf.reduce_sum(sp,0)+tf.reduce_sum(sn,0)+bp+bn+8.0*SIG_MIN_US
            y=tf.einsum("bhwi,io->bhwo",p,(sp-sn)/(den[None,:]+EPS))
            y+=(V_LIMIT*bp-V_LIMIT*bn)/(den+EPS); return tf.expand_dims(y,0),None,None
        if s["kind"]=="conv":
            p=tf.image.extract_patches(xp,[1,3,3,1],[1,1,1,1],[1,1,1,1],"SAME"); pn=-p
            b=tf.shape(xp)[0]; h=tf.shape(xp)[1]; w=tf.shape(xp)[2]; g=s["cin"]//64
            p=tf.reshape(p,[b,h,w,9,g,64]); pn=tf.reshape(pn,[b,h,w,9,g,64])
            den=tf.reduce_sum(sp,2)+tf.reduce_sum(sn,2); wp=sp/(den[:,:,None,:]+EPS); wn=sn/(den[:,:,None,:]+EPS)
            y=tf.einsum("bhwkgc,kgco->kgbhwo",p,(sp-sn)/(den[:,:,None,:]+EPS))
            return tf.reshape(y,[9*g,b,h,w,s["cout"]]),bp,bn
        g=s["cin"]//64; xr=tf.reshape(xp,[-1,g,64]); xnr=-xr
        den=tf.reduce_sum(sp,1)+tf.reduce_sum(sn,1); wp=sp/(den[:,None,:]+EPS); wn=sn/(den[:,None,:]+EPS)
        y=tf.einsum("bgc,gco->gbo",xr,(sp-sn)/(den[:,None,:]+EPS)); return y,bp,bn

    def _aggregate_tf(self,main,bp,bn):
        agg=self.hardware["aggregator"]; g=tf.constant(agg["g_signal_us"],tf.float32)
        shape=[g.shape[0]]+[1]*(main.shape.rank-2)+[g.shape[1]]
        num=tf.reduce_sum(main*tf.reshape(g,shape),0); den=tf.reduce_sum(g,0)
        bshape=[1]*(main.shape.rank-2)+[self.spec["cout"]]; pol=float(agg["bias_input_polarity"])
        num+=pol*(V_LIMIT*tf.reshape(bp,bshape)-V_LIMIT*tf.reshape(bn,bshape)); den+=bp+bn+agg["empty_rows"]*SIG_MIN_US
        return num/(den+EPS)

    def hard_raw_main_bounds_tf(self):
        s=self.spec; sp=conductance(self.theta_wp); sn=conductance(self.theta_wn); bp=conductance(self.theta_bp); bn=conductance(self.theta_bn)
        if self.stage=="stage1":
            den=tf.reduce_sum(sp,0)+tf.reduce_sum(sn,0)+bp+bn+8.0*SIG_MIN_US
            radius=V_LIMIT*tf.reduce_sum(tf.abs(sp-sn),0); bias=V_LIMIT*(bp-bn)
            return (bias-radius)/(den+EPS),(bias+radius)/(den+EPS)
        diff=sp-sn; term=V_LIMIT*diff
        if s["kind"]=="conv":
            den=tf.reduce_sum(sp,2)+tf.reduce_sum(sn,2)
            lo=tf.reduce_sum(tf.minimum(term,0.0),2)/(den+EPS); hi=tf.reduce_sum(tf.maximum(term,0.0),2)/(den+EPS)
            return tf.reshape(lo,[s["branches"],s["cout"]]),tf.reshape(hi,[s["branches"],s["cout"]])
        den=tf.reduce_sum(sp,1)+tf.reduce_sum(sn,1)
        return tf.reduce_sum(tf.minimum(term,0.0),1)/(den+EPS),tf.reduce_sum(tf.maximum(term,0.0),1)/(den+EPS)

    def _adc_sign_compensation(self):
        # Physical inverter parity: odd depth flips sign, even depth preserves it.
        # We do NOT add an inverter merely to restore parity. Instead, the final
        # signed-ADC interface swaps polarity (or equivalently negates the signed
        # ADC code) once per stage. This has no effect on voltage magnitude.
        p=float(self.hardware["main_chain"]["polarity"])
        if self.stage!="stage1": p*=float(self.hardware["post_chain"]["polarity"])
        return p

    def hard_analog_output_bounds_tf(self):
        lo,hi=self.hard_raw_main_bounds_tf(); lo,hi=cascade_interval_tf(lo,hi,self.hardware["main_chain"])
        if self.stage!="stage1":
            agg=self.hardware["aggregator"]; g=tf.constant(agg["g_signal_us"],tf.float32); bp=conductance(self.theta_bp); bn=conductance(self.theta_bn)
            pol=float(agg["bias_input_polarity"]); fixed=pol*(V_LIMIT*bp-V_LIMIT*bn); den=tf.reduce_sum(g,0)+bp+bn+agg["empty_rows"]*SIG_MIN_US
            alo=(tf.reduce_sum(lo*g,0)+fixed)/(den+EPS); ahi=(tf.reduce_sum(hi*g,0)+fixed)/(den+EPS)
            lo,hi=cascade_interval_tf(alo,ahi,self.hardware["post_chain"])
        # Return LOGICAL signed-ADC bounds after free polarity compensation.
        p=self._adc_sign_compensation()
        if p<0: return -hi,-lo
        return lo,hi

    def hard_analog_output_bounds_np(self):
        lo,hi=self.hard_analog_output_bounds_tf(); return lo.numpy().astype(np.float64),hi.numpy().astype(np.float64)

    def analog_linear(self,x,return_trace=False):
        main,bp,bn=self.raw_main(x); tr={}
        if return_trace: main2,nodes=cascade_tf(main,self.hardware["main_chain"],True)
        else: main2=cascade_tf(main,self.hardware["main_chain"]); nodes=()
        if return_trace:
            tr["main_raw"]=main
            for k,n in enumerate(nodes,1): tr[f"main_inv_{k}"]=n
        if self.stage=="stage1": physical_out=main2[0]
        else:
            avg=self._aggregate_tf(main2,bp,bn)
            if return_trace:
                tr["averager_raw"]=avg
                physical_out,pnodes=cascade_tf(avg,self.hardware["post_chain"],True)
            else: physical_out=cascade_tf(avg,self.hardware["post_chain"]); pnodes=()
            if return_trace:
                for k,n in enumerate(pnodes,1): tr[f"post_inv_{k}"]=n
        p=self._adc_sign_compensation()
        out=physical_out*p
        if return_trace:
            tr["analog_output_to_adc_physical"]=physical_out
            tr["analog_output_to_adc"]=out
        return (out,tr) if return_trace else out

    def call(self,x,training=False,return_trace=False):
        del training; return self.analog_linear(x,return_trace)

    def normalization_audit(self):
        sp=conductance(self.theta_wp).numpy().astype(np.float64); sn=conductance(self.theta_wn).numpy().astype(np.float64)
        bp=conductance(self.theta_bp).numpy().astype(np.float64); bn=conductance(self.theta_bn).numpy().astype(np.float64)
        if self.stage=="stage1":
            den=np.sum(sp,0)+np.sum(sn,0)+bp+bn+8*SIG_MIN_US; err=float(np.max(np.abs((np.sum(sp,0)+np.sum(sn,0)+bp+bn+8*SIG_MIN_US)/den-1)))
            return {"main_column_sum_max_error":err,"averager_column_sum_max_error":None}
        den=np.sum(sp,axis=-2)+np.sum(sn,axis=-2); err=float(np.max(np.abs((np.sum(sp,axis=-2)+np.sum(sn,axis=-2))/(den+EPS)-1)))
        agg=aggregator_design(self.stage); sig=np.sum(agg["g_signal_us"],0); total=sig+bp+bn+agg["empty_rows"]*SIG_MIN_US
        aerr=float(np.max(np.abs((sig+bp+bn+agg["empty_rows"]*SIG_MIN_US)/(total+EPS)-1)))
        return {"main_column_sum_max_error":err,"averager_column_sum_max_error":aerr}

def apply_aggregator_np(main,agg,bp,bn):
    y=np.asarray(main,np.float64); g=np.asarray(agg["g_signal_us"],np.float64); shape=(g.shape[0],)+(1,)*(y.ndim-2)+(g.shape[1],)
    num=np.sum(y*g.reshape(shape),0); den=np.sum(g,0); bshape=(1,)*(y.ndim-2)+(y.shape[-1],); pol=float(agg["bias_input_polarity"])
    num+=pol*(V_LIMIT*np.asarray(bp).reshape(bshape)-V_LIMIT*np.asarray(bn).reshape(bshape)); den+=np.asarray(bp)+np.asarray(bn)+agg["empty_rows"]*SIG_MIN_US
    return (num/(den+EPS)).astype(np.float32)


In [ ]:
# Streaming calibration: same per-column peak/depth policy, reduced peak memory.
def _stage_input_single(backbone,x,target_stage):
    z=tf.convert_to_tensor(x,tf.float32)
    for i in range(1,int(target_stage)):
        lin=backbone.stage(i)(z,training=False)
        z=backbone.interface_after(i,lin,training=False,quantize=True,pool_gap=True)
    return z.numpy()

def stage_input_for(backbone,x,target_stage):
    x=np.asarray(x,np.float32)
    return np.concatenate([_stage_input_single(backbone,x[j:j+CAL_BATCH_SIZE],target_stage)
                           for j in range(0,len(x),CAL_BATCH_SIZE)],axis=0)

def calibrate_stage(layer,stage_input,tag):
    t=time.perf_counter(); x=np.asarray(stage_input,np.float32); C=layer.spec["cout"]
    if np.max(np.abs(x))>V_LIMIT+VOLTAGE_TOL: raise RuntimeError("calibration input exceeds VDD")
    maxv=np.zeros(C,np.float64)
    for j in range(0,len(x),CAL_BATCH_SIZE):
        raw,_,_=layer.raw_main(tf.convert_to_tensor(x[j:j+CAL_BATCH_SIZE]))
        axes=tuple(range(raw.shape.rank-1))
        maxv=np.maximum(maxv,tf.reduce_max(tf.abs(raw),axis=axes).numpy())
    robust=bool(str(tag).startswith("full/initial") and layer.stage not in ("stage1","stage2"))
    s10_common=bool(EXP_S10_RELATIVE_SCALE and layer.stage=="stage10")
    if s10_common:
        # EXPERIMENT: identical chain for all classes -> identical per-class gain,
        # so the relative logit scale of the mapped head is preserved.
        maxv=np.full_like(maxv,float(np.max(maxv)))
    mc=synthesize_common_depth_per_column(maxv[None,:],tag+"/main",ALL_INVERTER_DEPTHS,robust_initial=robust)
    design={"stage":layer.stage,"model":"learned_tanh_proxy","main_chain":mc,
            "repeat_policy":REPEAT_POLICY,"loading_mode":LOADING_MODE}
    if layer.stage!="stage1":
        agg=aggregator_design(layer.stage); agg["bias_input_polarity"]=float(mc["polarity"])
        design["aggregator"]=agg; layer.hardware=design; postmax=np.zeros(C,np.float64)
        for j in range(0,len(x),CAL_BATCH_SIZE):
            raw,bp,bn=layer.raw_main(tf.convert_to_tensor(x[j:j+CAL_BATCH_SIZE]))
            post=layer._aggregate_tf(cascade_tf(raw,mc),bp,bn)
            postmax=np.maximum(postmax,tf.reduce_max(tf.abs(post),axis=tuple(range(post.shape.rank-1))).numpy())
        if s10_common: postmax=np.full_like(postmax,float(np.max(postmax)))
        pc=synthesize_common_depth_per_column(postmax[None,:],tag+"/post",ALL_INVERTER_DEPTHS,robust_initial=robust)
        design["post_chain"]=pc
    # Keras 3 wraps assigned dictionaries: publish the complete design again.
    layer.hardware=design
    node_max=0.0
    for j in range(0,len(x),CAL_BATCH_SIZE):
        _,tr=layer(tf.convert_to_tensor(x[j:j+CAL_BATCH_SIZE]),return_trace=True)
        node_max=max(node_max,max(float(tf.reduce_max(tf.abs(v)).numpy()) for v in tr.values()))
    if node_max>V_LIMIT+VOLTAGE_TOL: raise RuntimeError("physical calibration voltage violation")
    norm=layer.normalization_audit()
    if norm["main_column_sum_max_error"]>2e-6: raise RuntimeError("main normalization failed")
    if norm["averager_column_sum_max_error"] is not None and norm["averager_column_sum_max_error"]>2e-6:
        raise RuntimeError("averager normalization failed")
    row={"tag":tag,"stage":layer.stage,"main_depth":mc["n_inv"],
         "post_depth":design.get("post_chain",{}).get("n_inv"),"physical_node_abs_max_v":node_max,
         "elapsed_sec":time.perf_counter()-t}
    CALIBRATION_ROWS.append(row); print("CALIBRATION:",row)
    return copy.deepcopy(design)

def calibrate_stages(backbone,x,stages,tag):
    for i in stages: calibrate_stage(backbone.stage(i),stage_input_for(backbone,x,i),f"{tag}/stage{i}")

def snapshot_hardware(backbone,endpoint=10):
    return {f"stage{i}":copy.deepcopy(backbone.stage(i).hardware) for i in range(1,endpoint+1)}

def restore_hardware(backbone,snap):
    for k,v in snap.items(): backbone.stage(stage_index(k)).hardware=copy.deepcopy(v)

def _plain(obj):
    if isinstance(obj,dict): return {str(k):_plain(v) for k,v in obj.items()}
    if isinstance(obj,(list,tuple)): return [_plain(v) for v in obj]
    if isinstance(obj,np.ndarray): return obj.tolist()
    if isinstance(obj,np.generic): return obj.item()
    return obj

def hardware_cost(backbone):
    inv=mem=0
    for i in range(1,11):
        layer=backbone.stage(i); hw=layer.hardware; branches=layer.spec["branches"]
        for key,mult in (("main_chain",branches),("post_chain",1)):
            if key in hw:
                spec=hw[key]; inv+=mult*spec["n_inv"]*layer.spec["cout"]
                mem+=mult*sum(sum(r) for r in spec["memristor_count_levels"])
    return {"inverter_instances":int(inv),"load_memristors":int(mem)}


In [ ]:
def round_ste(x):
    x=tf.cast(x,tf.float32)
    return x+tf.stop_gradient(tf.round(x)-x)

def shift_tables(spec):
    q=np.arange(-ADC_SIGNED_QMAX,ADC_SIGNED_QMAX+1,dtype=np.float64)[:,None]
    out=np.zeros((len(q),len(spec["bias"])),np.float64)
    for key in ("term1","term2"):
        scale=np.exp2(np.asarray(spec[key]["exp"],np.float64))[None,:]
        sign=np.asarray(spec[key]["sign"],np.float64)[None,:]
        out+=np.round(q*scale)*sign
    return out

# ========================= FLOAT BN BOOTSTRAP + HARDWARE SHIFT-ADD BN =========================
class HardwareConvertibleBN(tf.keras.layers.Layer):
    def __init__(self,channels,name):
        super().__init__(name=name); self.channels=int(channels); self.momentum=BN_MOMENTUM; self.epsilon=BN_EPSILON
        self.shift_spec=None
    def build(self,input_shape):
        self.gamma=self.add_weight(name="gamma",shape=(self.channels,),initializer="ones",trainable=True)
        self.beta=self.add_weight(name="beta",shape=(self.channels,),initializer="zeros",trainable=True)
        self.moving_mean=self.add_weight(name="moving_mean",shape=(self.channels,),initializer="zeros",trainable=False)
        self.moving_variance=self.add_weight(name="moving_variance",shape=(self.channels,),initializer="ones",trainable=False)
        self.stats_frozen=self.add_weight(name="stats_frozen",shape=(),dtype=tf.bool,initializer=tf.keras.initializers.Constant(False),trainable=False)
        self.theta_shift_bias=self.add_weight(name="theta_shift_bias",shape=(self.channels,),initializer="zeros",trainable=TRAIN_SHIFTADD_BIAS)
        super().build(input_shape)
    def set_stats_frozen(self,flag=True): self.stats_frozen.assign(bool(flag))
    def _safe_bn(self,x,mean,var,lo,hi):
        lo=tf.fill(tf.shape(lo),tf.constant(-V_LIMIT,tf.float32));hi=tf.fill(tf.shape(hi),tf.constant(V_LIMIT,tf.float32))
        inv=tf.math.rsqrt(tf.maximum(var,0.0)+self.epsilon); a=self.gamma*inv; c=self.beta-a*mean
        lo=tf.stop_gradient(tf.cast(lo,tf.float32)); hi=tf.stop_gradient(tf.cast(hi,tf.float32))
        ymax=tf.maximum(a*lo+c,a*hi+c); scale=tf.where(ymax>BN_DAC_SAFE_VMAX,BN_DAC_SAFE_VMAX/tf.maximum(ymax,EPS),tf.ones_like(ymax))
        return ((x-mean)*inv*self.gamma+self.beta)*scale
    def call(self,x,training=False,input_lower=None,input_upper=None):
        if input_lower is None: raise ValueError("hard interval required")
        def batch():
            axes=list(range(x.shape.rank-1)); m,v=tf.nn.moments(x,axes); self.moving_mean.assign(self.momentum*self.moving_mean+(1-self.momentum)*m); self.moving_variance.assign(self.momentum*self.moving_variance+(1-self.momentum)*v)
            return self._safe_bn(x,m,v,input_lower,input_upper)
        def moving(): return self._safe_bn(x,self.moving_mean,self.moving_variance,input_lower,input_upper)
        if isinstance(training,bool): return (tf.cond(self.stats_frozen,moving,batch) if training else moving())
        return tf.cond(tf.cast(training,tf.bool),lambda:tf.cond(self.stats_frozen,moving,batch),moving)
    def effective_affine_voltage(self,input_lower,input_upper):
        g=self.gamma.numpy().astype(np.float64); b=self.beta.numpy().astype(np.float64); m=self.moving_mean.numpy().astype(np.float64); v=self.moving_variance.numpy().astype(np.float64)
        inv=1/np.sqrt(np.maximum(v,0)+self.epsilon); a=g*inv; c=b-a*m; lo=np.asarray(input_lower); hi=np.asarray(input_upper)
        lo=np.full_like(a,-V_LIMIT);hi=np.full_like(a,V_LIMIT)
        ymax=np.maximum(a*lo+c,a*hi+c); scale=np.ones_like(ymax); mask=ymax>BN_DAC_SAFE_VMAX; scale[mask]=BN_DAC_SAFE_VMAX/np.maximum(ymax[mask],EPS)
        return a*scale,c*scale,scale

    def install_shift_spec(self,spec):
        if not self.built: self.build((None,self.channels))
        s=copy.deepcopy(spec); tables=shift_tables(s)
        accmin=-(1<<(BN_ACCUMULATOR_BITS-1)); accmax=(1<<(BN_ACCUMULATOR_BITS-1))-1
        lower=np.maximum(accmin,accmin-tables.min(axis=0))
        upper=np.minimum(DAC_UNSIGNED_QMAX-tables.max(axis=0),accmax)
        if np.any(lower>upper): raise RuntimeError("no legal 16-bit bias interval")
        bias=np.asarray(s["bias"],np.float64)
        if np.any(bias<lower) or np.any(bias>upper): raise RuntimeError("illegal initial shift bias")
        s["bias_lower"]=lower.astype(int).tolist(); s["bias_upper"]=upper.astype(int).tolist()
        frac=np.clip((bias-lower)/np.maximum(upper-lower,1),1e-7,1-1e-7)
        self.shift_spec=s; self.theta_shift_bias.assign(np.log(frac/(1-frac)).astype(np.float32))
    def current_bias(self):
        s=self.shift_spec
        if TRAIN_SHIFTADD_BIAS:
            lo=tf.constant(s["bias_lower"],tf.float32); hi=tf.constant(s["bias_upper"],tf.float32)
            return round_ste(lo+(hi-lo)*tf.sigmoid(self.theta_shift_bias))
        return tf.constant(s["bias"],tf.float32)
    def export_shift_spec(self):
        if self.shift_spec is None: return None
        s=copy.deepcopy(self.shift_spec); s["bias"]=self.current_bias().numpy().astype(int).tolist()
        return s
    def shiftadd(self,q):
        if self.shift_spec is None: raise RuntimeError("shift specification missing")
        q=tf.cast(q,tf.float32); out=tf.zeros_like(q)
        for key in ("term1","term2"):
            sign=tf.constant(self.shift_spec[key]["sign"],tf.float32)
            scale=tf.constant(np.exp2(np.asarray(self.shift_spec[key]["exp"],np.float32)))
            # Forward equals exact integer shifts with nearest-even rounding.
            # Backward includes BOTH right-shift contributions, with their proper scales.
            out+=sign*round_ste(q*scale)
        out+=self.current_bias()
        amin=float(-(1<<(BN_ACCUMULATOR_BITS-1))); amax=float((1<<(BN_ACCUMULATOR_BITS-1))-1)
        checks=[tf.debugging.assert_greater_equal(tf.reduce_min(out),amin),
                tf.debugging.assert_less_equal(tf.reduce_max(out),amax)]
        with tf.control_dependencies(checks): out=tf.nn.relu(out)
        chk=tf.debugging.assert_less_equal(tf.reduce_max(out),float(DAC_UNSIGNED_QMAX),message=self.name+": DAC overflow")
        with tf.control_dependencies([chk]): return tf.identity(out)

class VGGPHAXBackbone(tf.keras.Model):
    def __init__(self):
        super().__init__(name="vgg9_phax_tanh_fast"); self._stages=[VGGPHAXStage(f"stage{i}") for i in range(1,11)]
        self._bns=[HardwareConvertibleBN(STAGE_SPECS[f"stage{i}"]["cout"],f"stage{i}_bn") for i in range(1,10)]
        self.interface_mode="float_bn"; self.shift_active={i:False for i in range(1,10)}
    def stage(self,i): return self._stages[int(i)-1]
    def bn(self,i): return self._bns[int(i)-1]
    def set_bn_stats_frozen(self,flag=True):
        for b in self._bns: b.set_stats_frozen(flag)
        record_event("bn_stats_freeze",frozen=bool(flag))

    def uses_shift(self,i): return bool(self.shift_active[int(i)])
    def activate_shift_stages(self,stages):
        for i in stages:
            if self.bn(i).shift_spec is None: raise RuntimeError("activate only a fitted BN")
            self.shift_active[int(i)]=True
        self.interface_mode="shiftadd" if all(self.shift_active.values()) else "hybrid"
        bump_hardware_revision("activate_shift_stages:"+str(list(stages)))
    def interface_after(self,i,lin,training=False,quantize=True,pool_gap=True):
        if self.uses_shift(i):
            code=self.bn(i).shiftadd(signed_adc_code_ste(lin))
            if pool_gap:
                if i in (2,4,6): code=tf.nn.max_pool2d(code,2,2,"SAME")
                if i==8: code=tf.reduce_mean(code,axis=(1,2))
            return unsigned_dac_from_code_ste(code) if quantize else code*V_LIMIT/DAC_UNSIGNED_QMAX
        av=signed_adc_qdq_ste(lin); lo,hi=self.stage(i).hard_analog_output_bounds_tf()
        z=tf.nn.relu(self.bn(i)(av,training=training,input_lower=signed_adc_qdq_value(lo),input_upper=signed_adc_qdq_value(hi)))
        if pool_gap:
            if i in (2,4,6): z=tf.nn.max_pool2d(z,2,2,"SAME")
            if i==8: z=tf.reduce_mean(z,axis=(1,2))
        return unsigned_dac_qdq_ste(z) if quantize else z
    def forward_to(self,x,endpoint,training=False):
        z=x
        for i in range(1,int(endpoint)+1):
            lin=self.stage(i)(z,training=training)
            if i==10: z=final_adc_code_logits(lin)
            else: z=self.interface_after(i,lin,training=training,quantize=(i<endpoint),pool_gap=(i!=8 or endpoint>=9))
        return z
    def call(self,x,training=False): return self.forward_to(x,10,training)

def seeded_conv(filters,seed): return tf.keras.layers.Conv2D(filters,3,padding="same",use_bias=True,kernel_initializer=tf.keras.initializers.HeNormal(seed=seed),bias_initializer="zeros")
def seeded_dense(units,seed): return tf.keras.layers.Dense(units,use_bias=True,kernel_initializer=tf.keras.initializers.HeNormal(seed=seed),bias_initializer="zeros")
def seeded_bn(name): return tf.keras.layers.BatchNormalization(momentum=BN_MOMENTUM,epsilon=BN_EPSILON,name=name)
class TemporaryFrontDigitalTail(tf.keras.Model):
    def __init__(self):
        super().__init__(); b=SEED+10000
        self.c3=seeded_conv(128,b+3); self.b3=seeded_bn("t3"); self.c4=seeded_conv(128,b+4); self.b4=seeded_bn("t4")
        self.c5=seeded_conv(256,b+5); self.b5=seeded_bn("t5"); self.c6=seeded_conv(256,b+6); self.b6=seeded_bn("t6")
        self.c7=seeded_conv(512,b+7); self.b7=seeded_bn("t7"); self.c8=seeded_conv(512,b+8); self.b8=seeded_bn("t8")
        self.fc=seeded_dense(512,b+9); self.b9=seeded_bn("t9"); self.out=seeded_dense(10,b+10)
    def call(self,z,training=False):
        z=tf.nn.relu(self.b3(self.c3(z),training=training)); z=tf.nn.relu(self.b4(self.c4(z),training=training)); z=tf.nn.max_pool2d(z,2,2,"SAME")
        z=tf.nn.relu(self.b5(self.c5(z),training=training)); z=tf.nn.relu(self.b6(self.c6(z),training=training)); z=tf.nn.max_pool2d(z,2,2,"SAME")
        z=tf.nn.relu(self.b7(self.c7(z),training=training)); z=tf.nn.relu(self.b8(self.c8(z),training=training)); z=tf.reduce_mean(z,(1,2)); z=tf.nn.relu(self.b9(self.fc(z),training=training)); return self.out(z)
class FrontClassifier(tf.keras.Model):
    def __init__(self,b,t): super().__init__(); self.backbone=b; self.tail=t
    def call(self,x,training=False): return self.tail(self.backbone.forward_to(x,2,training),training=training)
class FullPHAXClassifier(tf.keras.Model):
    def __init__(self,b): super().__init__(); self.backbone=b
    def call(self,x,training=False): return self.backbone(x,training=training)


# -------- Shift-Add coefficient library --------
def _coeff_library():
    best={}
    # one term
    for e in range(SHIFTADD_EXP_MIN,SHIFTADD_EXP_MAX+1):
        for s in (-1,1):
            m=s*(2.0**e); best[round(m,12)]=(m,(s,e),(0,0),1)
    # two terms; retain lowest complexity canonical representation for each value
    for e1 in range(SHIFTADD_EXP_MIN,SHIFTADD_EXP_MAX+1):
        for e2 in range(SHIFTADD_EXP_MIN,SHIFTADD_EXP_MAX+1):
            for s1 in (-1,1):
                for s2 in (-1,1):
                    m=s1*(2.0**e1)+s2*(2.0**e2)
                    if abs(m)<1e-12: continue
                    key=round(m,12); cand=(m,(s1,e1),(s2,e2),2)
                    if key not in best or cand[3]<best[key][3]: best[key]=cand
    return list(best.values())
SHIFT_COEFFS=_coeff_library()

def _term_np(q,term):
    s,e=term; q=np.asarray(q,np.float64)
    if s==0: return np.zeros_like(q)
    v=q*(2.0**e) if e>=0 else np.round(q/(2.0**(-e)))
    return s*v

def _base_table_for_candidate(cand):
    q=np.arange(-ADC_SIGNED_QMAX,ADC_SIGNED_QMAX+1,dtype=np.float64)
    return _term_np(q,cand[1])+_term_np(q,cand[2])


# BN01/02/06/07: exact integer forward, float shadow parameters for learning.
# These flags are set in the FIRST notebook cell, before layers are constructed.
if BN_DIRECT_BIAS or BN_LEARN_SCALE:
    _OriginalHardwareConvertibleBN=HardwareConvertibleBN
    class HardwareConvertibleBN(_OriginalHardwareConvertibleBN):
        def build(self,input_shape):
            # Add state BEFORE the parent marks the Keras layer built.
            self.bn_bias_shadow=self.add_weight(name='bn_bias_shadow',shape=(self.channels,),initializer='zeros',trainable=BN_DIRECT_BIAS)
            self.bn_scale_shadow=self.add_weight(name='bn_scale_shadow',shape=(self.channels,),initializer='ones',trainable=BN_LEARN_SCALE)
            super().build(input_shape)

        def install_shift_spec(self,spec):
            super().install_shift_spec(spec)
            self.shift_spec['installed_coeff_bias_lower']=list(self.shift_spec['bias_lower'])
            self.shift_spec['installed_coeff_bias_upper']=list(self.shift_spec['bias_upper'])
            # Save the ORIGINAL sigmoid parameter interval across snapshots.
            # A learned scale may have a different legal bias interval later.
            for key in ('bias_lower','bias_upper'):
                saved=spec.get('legacy_'+key,self.shift_spec[key])
                self.shift_spec['legacy_'+key]=list(saved)
                if not BN_DIRECT_BIAS:self.shift_spec[key]=list(saved)
            if not BN_DIRECT_BIAS:
                lo=np.asarray(self.shift_spec['bias_lower'],np.float64)
                hi=np.asarray(self.shift_spec['bias_upper'],np.float64)
                frac=np.clip((np.asarray(spec['bias'])-lo)/np.maximum(hi-lo,1),1e-7,1-1e-7)
                self.theta_shift_bias.assign(np.log(frac/(1-frac)).astype(np.float32))
            self.bn_bias_shadow.assign(np.asarray(spec['bias'],np.float32))
            m=sum(np.asarray(spec[k]['sign'])*np.exp2(spec[k]['exp']) for k in ('term1','term2'))
            self.bn_scale_shadow.assign(m.astype(np.float32))

        def _proposed_bias(self):
            if BN_DIRECT_BIAS:return round_ste(self.bn_bias_shadow)
            return super().current_bias()

        def _selected_terms(self):
            if self.shift_spec is None:raise RuntimeError('Shift-Add specification missing')
            if BN_LEARN_SCALE:
                # Tables are built from the SAME canonical one/two-shift library
                # used by the base notebook. Selection always obeys all-code bounds.
                lib=SHIFT_COEFFS
                lowers=tf.constant(BN_LIBRARY_BIAS_LOWER,tf.float32)[:,None]
                uppers=tf.constant(BN_LIBRARY_BIAS_UPPER,tf.float32)[:,None]
                legal=lowers<=uppers
                if not BN_DIRECT_BIAS:
                    proposed=self._proposed_bias()[None,:]
                    legal=tf.logical_and(legal,tf.logical_and(proposed>=lowers,proposed<=uppers))
                else:
                    legal=tf.broadcast_to(legal,[len(lib),self.channels])
                check=tf.debugging.assert_equal(tf.reduce_all(tf.reduce_any(legal,axis=0)),True,
                                                message='No legal shift coefficient for current bias')
                with tf.control_dependencies([check]):
                    distance=tf.abs(tf.constant(BN_LIBRARY_M,tf.float32)[:,None]-self.bn_scale_shadow[None,:])
                    ids=tf.argmin(tf.where(legal,distance,tf.constant(np.inf,tf.float32)),axis=0,output_type=tf.int32)
                signs=[tf.gather(tf.constant([c[k][0] for c in lib],tf.float32),ids) for k in (1,2)]
                exps=[tf.gather(tf.constant([c[k][1] for c in lib],tf.int32),ids) for k in (1,2)]
                lo=tf.gather(tf.constant(BN_LIBRARY_BIAS_LOWER,tf.float32),ids)
                hi=tf.gather(tf.constant(BN_LIBRARY_BIAS_UPPER,tf.float32),ids)
                # Equal scalar slopes can have DIFFERENT integer functions when
                # their two terms round separately. Preserve the installed exact
                # representation on ties, rather than silently canonicalizing it.
                old_s=[tf.constant(self.shift_spec[k]['sign'],tf.float32) for k in ('term1','term2')]
                old_e=[tf.constant(self.shift_spec[k]['exp'],tf.int32) for k in ('term1','term2')]
                old_m=sum(s*tf.pow(2.,tf.cast(e,tf.float32)) for s,e in zip(old_s,old_e))
                new_m=sum(s*tf.pow(2.,tf.cast(e,tf.float32)) for s,e in zip(signs,exps))
                old_lo=tf.constant(self.shift_spec['installed_coeff_bias_lower'],tf.float32)
                old_hi=tf.constant(self.shift_spec['installed_coeff_bias_upper'],tf.float32)
                old_legal=old_lo<=old_hi
                if not BN_DIRECT_BIAS:
                    pb=self._proposed_bias();old_legal=tf.logical_and(old_legal,tf.logical_and(pb>=old_lo,pb<=old_hi))
                keep=tf.logical_and(tf.equal(new_m,old_m),old_legal)
                signs=[tf.where(keep,old,new) for old,new in zip(old_s,signs)]
                exps=[tf.where(keep,old,new) for old,new in zip(old_e,exps)]
                lo=tf.where(keep,old_lo,lo);hi=tf.where(keep,old_hi,hi)
                return signs,exps,lo,hi
            signs=[tf.constant(self.shift_spec[k]['sign'],tf.float32) for k in ('term1','term2')]
            exps=[tf.constant(self.shift_spec[k]['exp'],tf.int32) for k in ('term1','term2')]
            return signs,exps,tf.constant(self.shift_spec['bias_lower'],tf.float32),tf.constant(self.shift_spec['bias_upper'],tf.float32)

        def _bias_for_terms(self,lo,hi):
            if BN_DIRECT_BIAS:return round_ste(tf.clip_by_value(self.bn_bias_shadow,lo,hi))
            return self._proposed_bias()

        def current_bias(self):
            _,_,lo,hi=self._selected_terms()
            return self._bias_for_terms(lo,hi)

        def shiftadd(self,q):
            q=tf.cast(q,tf.float32);signs,exps,lo,hi=self._selected_terms()
            powers=[tf.pow(2.,tf.cast(e,tf.float32)) for e in exps]
            bias=self._bias_for_terms(lo,hi)
            # Each shift rounds SEPARATELY, including negative inputs and right
            # shifts. Do not replace this by round(q * sum_of_coefficients).
            hard=sum(s*tf.round(q*p) for s,p in zip(signs,powers))+bias
            mh=sum(s*p for s,p in zip(signs,powers))
            m=self.bn_scale_shadow+tf.stop_gradient(mh-self.bn_scale_shadow) if BN_LEARN_SCALE else mh
            soft=q*m+bias
            pre=soft+tf.stop_gradient(hard-soft)
            amin=float(-(1<<(BN_ACCUMULATOR_BITS-1)));amax=float((1<<(BN_ACCUMULATOR_BITS-1))-1)
            checks=[tf.debugging.assert_greater_equal(tf.reduce_min(pre),amin),
                    tf.debugging.assert_less_equal(tf.reduce_max(pre),amax)]
            with tf.control_dependencies(checks):out=tf.nn.relu(pre)
            ck=tf.debugging.assert_less_equal(tf.reduce_max(out),float(DAC_UNSIGNED_QMAX))
            with tf.control_dependencies([ck]):return tf.identity(out)

        def export_shift_spec(self):
            if self.shift_spec is None:return None
            signs,exps,lo,hi=self._selected_terms();s=copy.deepcopy(self.shift_spec)
            for key,sg,ex in zip(('term1','term2'),signs,exps):
                s[key]={'sign':sg.numpy().astype(int).tolist(),'exp':ex.numpy().astype(int).tolist()}
            s['bias']=self._bias_for_terms(lo,hi).numpy().astype(int).tolist()
            s['bias_lower']=lo.numpy().astype(int).tolist();s['bias_upper']=hi.numpy().astype(int).tolist()
            s['m_effective']=(signs[0]*tf.pow(2.,tf.cast(exps[0],tf.float32))+signs[1]*tf.pow(2.,tf.cast(exps[1],tf.float32))).numpy().astype(float).tolist()
            return s

        def experiment_variables(self):
            return ([self.bn_bias_shadow] if BN_DIRECT_BIAS else [])+([self.bn_scale_shadow] if BN_LEARN_SCALE else [])

        def project_after_update(self):
            if BN_DIRECT_BIAS and self.shift_spec is not None:
                _,_,lo,hi=self._selected_terms()
                self.bn_bias_shadow.assign(tf.clip_by_value(self.bn_bias_shadow,lo,hi))


In [ ]:
# Histogram-weighted fitting: all calibration observations, no huge activation archive.
SHIFT_Q=np.arange(-ADC_SIGNED_QMAX,ADC_SIGNED_QMAX+1,dtype=np.float64)
SHIFT_BASE_TABLES=np.asarray([_base_table_for_candidate(c) for c in SHIFT_COEFFS])
SHIFT_M=np.asarray([c[0] for c in SHIFT_COEFFS])

def code_histogram(codes,qmin,qmax):
    a=np.asarray(codes); C=a.shape[-1]; flat=np.rint(a).astype(np.int64).reshape(-1,C)
    if np.any(flat<qmin) or np.any(flat>qmax): raise RuntimeError("code histogram out of range")
    width=qmax-qmin+1; ids=(flat-qmin)+np.arange(C,dtype=np.int64)[None,:]*width
    return np.bincount(ids.ravel(),minlength=C*width).reshape(C,width)

def synthesize_one_bn(backbone,x,stage,label):
    i=int(stage); b=backbone.bn(i); C=b.channels; hist=np.zeros((C,255),np.int64)
    for j in range(0,len(x),CAL_BATCH_SIZE):
        inp=_stage_input_single(backbone,x[j:j+CAL_BATCH_SIZE],i)
        lin=backbone.stage(i)(tf.convert_to_tensor(inp),training=False)
        hist+=code_histogram(_signed_adc_code_tf(lin).numpy(),-127,127)
    lo,hi=backbone.stage(i).hard_analog_output_bounds_tf()
    a,c,proj=b.effective_affine_voltage(signed_adc_qdq_value(lo).numpy(),signed_adc_qdq_value(hi).numpy())
    M=a*(DAC_UNSIGNED_QMAX/ADC_SIGNED_QMAX); B=c*(DAC_UNSIGNED_QMAX/V_LIMIT)
    spec={"term1":{"sign":[],"exp":[]},"term2":{"sign":[],"exp":[]},"bias":[],"m_effective":[],
          "float_m_target":M.tolist(),"float_b_target":B.tolist(),"certificate":"all_ADC_codes_minus127_to_plus127"}
    rows=[]; accmin=-(1<<(BN_ACCUMULATOR_BITS-1)); accmax=(1<<(BN_ACCUMULATOR_BITS-1))-1
    for ch in range(C):
        h=hist[ch].astype(np.float64); h/=max(float(h.sum()),1.0)
        target=np.maximum(M[ch]*SHIFT_Q+B[ch],0)
        near=np.argsort(np.abs(SHIFT_M-M[ch]),kind="stable")[:SHIFTADD_NEAREST_COEFFS].tolist()
        small=np.flatnonzero(np.abs(SHIFT_M)<=abs(M[ch])+1e-12)
        small=small[np.argsort(-np.abs(SHIFT_M[small]),kind="stable")[:16]].tolist()
        # A steep float BN can make every nearby candidate dead under the full
        # ADC-range certificate. Always include useful smaller legal slopes.
        fallback=[]
        for value in (.125,.25,.5,.75,1.,1.25,1.5,1.75,2.):
            for sign in (-1.,1.):
                fallback.append(int(np.argmin(np.abs(SHIFT_M-sign*value))))
        ids=list(dict.fromkeys(near+small+fallback)); best=None
        for k in ids:
            cand=SHIFT_COEFFS[k]; table=SHIFT_BASE_TABLES[k]
            bmin=int(max(accmin,accmin-table.min())); bmax=int(min(255-table.max(),accmax))
            if bmin>bmax: continue
            bc=int(round(B[ch])); choices={max(bmin,min(bc,bmax)),bmax}
            for d in (1,2,4,8,16):
                choices.add(max(bmin,min(bc-d,bmax))); choices.add(max(bmin,min(bc+d,bmax)))
            for bias in sorted(choices):
                raw=table+bias; pred=np.maximum(raw,0)
                if raw.min()<accmin or raw.max()>accmax or pred.max()>255: continue
                mse=float(np.sum(h*(pred-target)**2)/(255.0**2))
                zm=float(np.sum(h*((pred<=0)!=(target<=0))))
                score=mse+0.02*zm+1e-6*cand[3]
                if best is None or score<best[0]: best=(score,k,bias,mse,zm,float(pred.max()))
        if best is None: raise RuntimeError(f"no legal shift candidate S{i} C{ch}")
        _,k,bias,mse,zm,hardmax=best; cand=SHIFT_COEFFS[k]
        for key,term in (("term1",cand[1]),("term2",cand[2])):
            spec[key]["sign"].append(int(term[0])); spec[key]["exp"].append(int(term[1]))
        spec["bias"].append(int(bias)); spec["m_effective"].append(float(cand[0]))
        rows.append({"label":label,"stage":i,"channel":ch,"m_float":M[ch],"b_float":B[ch],
                     "m_shiftadd":cand[0],"bias_int":bias,"mse_norm":mse,"zero_mask_mismatch":zm,
                     "float_projection_scale":proj[ch],"hard_output_max":hardmax})
    b.install_shift_spec(spec); SHIFTADD_ROWS.extend(rows)
    pd.DataFrame(SHIFTADD_ROWS).to_csv(OUT_DIR/"shiftadd_fit_per_channel.csv",index=False)
    record_event("bn_fitted",label=label,stage=i,mean_mse=float(np.mean([r["mse_norm"] for r in rows])),
                 mean_zero_mismatch=float(np.mean([r["zero_mask_mismatch"] for r in rows])))
    return rows

def shiftadd_hard_legality_audit(backbone,label):
    rows=[]; accmin=-(1<<(BN_ACCUMULATOR_BITS-1)); accmax=(1<<(BN_ACCUMULATOR_BITS-1))-1
    for i in range(1,10):
        if not backbone.uses_shift(i): continue
        s=backbone.bn(i).export_shift_spec(); base=shift_tables(s); raw=base+np.asarray(s["bias"])[None,:]
        if raw.min()<accmin or raw.max()>accmax or np.maximum(raw,0).max()>255:
            raise RuntimeError(f"illegal full-range shift BN S{i}")
        for ch in range(raw.shape[1]):
            exact=float(s["m_effective"][ch]); legacy=0.0
            for key in ("term1","term2"):
                if s[key]["exp"][ch]>=0: legacy+=s[key]["sign"][ch]*2.**s[key]["exp"][ch]
            rows.append({"label":label,"stage":i,"channel":ch,"bias_int":s["bias"][ch],
                         "dac_max_all_adc_codes":float(np.maximum(raw[:,ch],0).max()),
                         "accumulator_min":float(raw[:,ch].min()),"accumulator_max":float(raw[:,ch].max()),
                         "correct_STE_slope":exact,"legacy_without_right_STE_slope":legacy,
                         "legacy_slope_missing":bool(abs(exact-legacy)>1e-12)})
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR/f"shiftadd_hard_audit_{label}.csv",index=False)
    return rows


# Shared diagnostics are read-only and present in EVERY experiment, including BN00.
BN_PARAMETER_ROWS=[];BN_STATS_ROWS=[];BN_REFERENCE={}
TIMING['bn_experiment_diagnostic_sec']=0.0
BN_LIBRARY_M=np.asarray([c[0] for c in SHIFT_COEFFS],np.float32)
_bn_tables=np.asarray([_base_table_for_candidate(c) for c in SHIFT_COEFFS])
_amin=-(1<<(BN_ACCUMULATOR_BITS-1));_amax=(1<<(BN_ACCUMULATOR_BITS-1))-1
BN_LIBRARY_BIAS_LOWER=np.maximum(_amin,_amin-_bn_tables.min(1)).astype(np.float32)
BN_LIBRARY_BIAS_UPPER=np.minimum(255-_bn_tables.max(1),_amax).astype(np.float32)
del _bn_tables

def bn_apply_gradients(opt,pairs):
    # Adam is invariant to uniform gradient multiplication; use an explicit
    # parameter-update multiplier instead. The forward still uses legal codes.
    watched=[];pair_ids={id(v) for g,v in pairs}
    if BN_DIRECT_BIAS or BN_LEARN_SCALE:
        for b in backbone._bns:
            for v in b.experiment_variables():
                if id(v) in pair_ids:
                    factor=BN_BIAS_UPDATE_MULTIPLIER if v is b.bn_bias_shadow else BN_SCALE_UPDATE_MULTIPLIER
                    watched.append((v,tf.identity(v),factor))
    opt.apply_gradients(pairs)
    for v,old,factor in watched:v.assign(old+float(factor)*(v-old))
    if BN_DIRECT_BIAS:
        for b in backbone._bns:
            if b.shift_spec is not None and id(b.bn_bias_shadow) in pair_ids:b.project_after_update()

def bn_record_parameters(backbone,label):
    for i in range(1,10):
        b=backbone.bn(i)
        if not backbone.uses_shift(i) or b.shift_spec is None:continue
        s=b.export_shift_spec();ref=BN_REFERENCE.setdefault(i,copy.deepcopy(s))
        m=np.asarray(s['m_effective']);bias=np.asarray(s['bias']);rm=np.asarray(ref['m_effective']);rb=np.asarray(ref['bias'])
        row={'label':label,'stage':i,'bias_changed_fraction':float(np.mean(bias!=rb)),
             'scale_changed_fraction':float(np.mean(m!=rm)),
             'bias_min':float(bias.min()),'bias_max':float(bias.max()),
             'm_min':float(m.min()),'m_max':float(m.max()),
             'bias_on_upper_fraction':float(np.mean(bias==np.asarray(s['bias_upper'])))}
        if hasattr(b,'bn_bias_shadow'):
            row['bias_shadow_mean_abs_delta_from_initial']=float(np.mean(np.abs(b.bn_bias_shadow.numpy()-rb)))
            row['scale_shadow_mean_abs_delta_from_initial']=float(np.mean(np.abs(b.bn_scale_shadow.numpy()-rm)))
        BN_PARAMETER_ROWS.append(row)
    if BN_PARAMETER_ROWS:pd.DataFrame(BN_PARAMETER_ROWS).to_csv(OUT_DIR/'bn_parameter_history.csv',index=False)

def bn_export_final_channels(backbone):
    rows=[]
    for i in range(1,10):
        b=backbone.bn(i);s=b.export_shift_spec();ref=BN_REFERENCE.get(i,s)
        for ch in range(b.channels):
            row={'stage':i,'channel':ch,'bias_initial':ref['bias'][ch],'bias_final':s['bias'][ch],
                 'scale_initial':ref['m_effective'][ch],'scale_final':s['m_effective'][ch]}
            for key in ('term1','term2'):
                row[key+'_sign']=s[key]['sign'][ch];row[key+'_exp']=s[key]['exp'][ch]
            rows.append(row)
    pd.DataFrame(rows).to_csv(OUT_DIR/'bn_final_per_channel.csv',index=False)

def bn_reestimate_train_statistics(backbone,stage):
    # Streaming Chan/Welford merge. Per-batch spatial moments use float32;
    # merging across batches uses float64. No weight update, dropout, augmentation
    # or moving-stat update in ANY upstream layer takes place during this scan.
    b=backbone.bn(stage);count=0;mean=np.zeros(b.channels,np.float64);m2=np.zeros(b.channels,np.float64)
    old_mean=b.moving_mean.numpy().copy();old_var=b.moving_variance.numpy().copy()
    for j in range(0,len(x_train),CAL_BATCH_SIZE):
        inp=_stage_input_single(backbone,x_train[j:j+CAL_BATCH_SIZE],stage)
        lin=backbone.stage(stage)(tf.convert_to_tensor(inp),training=False)
        av=signed_adc_qdq_value(lin)
        bm,bv=tf.nn.moments(av,list(range(av.shape.rank-1)))
        n=int(np.prod(av.shape[:-1]));bm=bm.numpy().astype(np.float64);bv=bv.numpy().astype(np.float64)
        if count==0:mean=bm;m2=bv*n;count=n
        else:
            delta=bm-mean;total=count+n
            m2+=bv*n+delta*delta*(count*n/total);mean+=delta*(n/total);count=total
    if count==0:raise RuntimeError('Empty TRAIN statistics scan')
    var=m2/count;b.moving_mean.assign(mean.astype(np.float32));b.moving_variance.assign(var.astype(np.float32))
    row={'stage':stage,'train_images':len(x_train),'observations_per_channel':count,
         'mean_abs_change':float(np.mean(np.abs(mean-old_mean))),
         'variance_abs_change':float(np.mean(np.abs(var-old_var))),'source':'TRAIN_no_augmentation'}
    BN_STATS_ROWS.append(row);pd.DataFrame(BN_STATS_ROWS).to_csv(OUT_DIR/'bn_train_stat_reestimation.csv',index=False)
    record_event('bn_train_statistics_reestimated',**row)

_bn_base_synthesize=synthesize_one_bn
def synthesize_one_bn(backbone,x,stage,label):
    if BN_FROM_FIRST_STEP:
        record_event('bn_conversion_skipped_already_integer_from_start',stage=int(stage),label=label)
        return []
    if BN_REESTIMATE_STATS:bn_reestimate_train_statistics(backbone,int(stage))
    fit_x=x_train if BN_FIT_ALL_TRAIN else x
    rows=_bn_base_synthesize(backbone,fit_x,stage,label)
    BN_REFERENCE[int(stage)]=copy.deepcopy(backbone.bn(stage).export_shift_spec())
    return rows

def bn_initialize_hard_from_start(backbone):
    if not BN_FROM_FIRST_STEP:return
    for i in range(1,10):
        c=backbone.bn(i).channels
        s={'term1':{'sign':[1]*c,'exp':[0]*c},'term2':{'sign':[0]*c,'exp':[0]*c},
           'bias':[32]*c,'m_effective':[1.]*c,'certificate':'all_ADC_codes_minus127_to_plus127'}
        backbone.bn(i).install_shift_spec(s);BN_REFERENCE[i]=copy.deepcopy(backbone.bn(i).export_shift_spec())
    backbone.activate_shift_stages(range(1,10))
    record_event('all_physical_interfaces_integer_from_first_step',initial_gain=1.,initial_bias=32)


In [ ]:
# Training metrics stay on the device; grouped execution reduces Python synchronizations.
loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
HARDWARE_REVISION=0; _EVAL_CACHE={}; CUM_EPOCH=0

def bump_hardware_revision(reason):
    global HARDWARE_REVISION,_EVAL_CACHE
    HARDWARE_REVISION+=1; _EVAL_CACHE={}
    record_event("hardware_revision",revision=HARDWARE_REVISION,reason=reason)

def make_train_step(model,opt):
    acc=tf.Variable(tf.zeros([3],tf.float32),trainable=False)
    @tf.function(reduce_retracing=True)
    def step(x,y):
        with tf.GradientTape() as tape:
            logits=model(x,training=True); loss=loss_fn(y,logits)
        vv=model.trainable_variables; gg=tape.gradient(loss,vv)
        pairs=[(g,v) for g,v in zip(gg,vv) if g is not None]
        if not pairs: raise RuntimeError("all gradients are disconnected")
        bn_apply_gradients(opt,pairs)
        n=tf.cast(tf.shape(y)[0],tf.float32)
        correct=tf.reduce_sum(tf.cast(tf.equal(tf.argmax(logits,-1,output_type=tf.int64),tf.cast(y,tf.int64)),tf.float32))
        return acc.assign_add(tf.stack([loss*n,correct,n]))
    @tf.function(reduce_retracing=True)
    def group(iterator,count):
        def body(i):
            x,y=next(iterator)
            with tf.control_dependencies([step(x,y)]): return [i+1]
        return tf.while_loop(lambda i:i<count,body,[tf.constant(0)],parallel_iterations=1)[0]
    step.accumulator=acc; step.group=group
    return step

def train_epoch(model,step,seed):
    step.accumulator.assign(tf.zeros([3])); ds=make_train_ds(seed,BATCH_SIZE); it=iter(ds)
    count=int(math.ceil(len(x_train)/BATCH_SIZE))
    for j in range(0,count,STEPS_PER_EXECUTION):
        step.group(it,tf.constant(min(STEPS_PER_EXECUTION,count-j),tf.int32))
    ls,correct,n=step.accumulator.numpy().astype(np.float64)
    return {"loss":float(ls/n),"accuracy":float(correct/n)}

# ---------------- INFO01 timing helpers ----------------
def _state_arrays(variables): return [v.numpy().copy() for v in variables]
def _assign_arrays(variables,arrays):
    for v,a in zip(variables,arrays): v.assign(a)
def _arrays_equal(variables,arrays):
    return all(np.array_equal(v.numpy(),a) for v,a in zip(variables,arrays))
def _sync_on(variables):
    # Host read of values written by the last queued update: waits for the GPU stream.
    vs=[v for v in variables if tf.as_dtype(v.dtype).is_floating]   # Keras 3 dtype may be a str
    if vs: float(tf.add_n([tf.reduce_sum(tf.cast(v,tf.float32)) for v in vs]).numpy())

def train_epoch_timed(model,step,seed):
    """Same batches and order as train_epoch. The training interval starts after
    the tf.data object and iterator exist and ends after a host read of the on-device
    accumulator, i.e. after every queued forward/backward/update has finished."""
    t0=time.perf_counter()
    step.accumulator.assign(tf.zeros([3])); ds=make_train_ds(seed,BATCH_SIZE); it=iter(ds)
    count=int(math.ceil(len(x_train)/BATCH_SIZE))
    build=time.perf_counter()-t0
    t1=time.perf_counter()
    for j in range(0,count,STEPS_PER_EXECUTION):
        step.group(it,tf.constant(min(STEPS_PER_EXECUTION,count-j),tf.int32))
    ls,correct,n=step.accumulator.numpy().astype(np.float64)
    train=time.perf_counter()-t1
    return {"loss":float(ls/n),"accuracy":float(correct/n)},{"pipeline_build_sec":build,"train_sec":train}

def warmup_classification_step(model,opt,step):
    """Trace and compile this phase's train graph for the full and the remainder batch
    shape on a separate tiny dataset, then restore all model/optimizer variables
    bit-exactly. The real epoch iterator and data order are untouched."""
    if not TIMING_WARMUP_BEFORE_PHASE: return
    mv=list(model.variables); ov=list(opt.variables)
    saved_m=_state_arrays(mv); saved_o=_state_arrays(ov); saved_acc=step.accumulator.numpy().copy()
    rem=len(x_train)%BATCH_SIZE; n=BATCH_SIZE+rem
    ds=_train_pipeline(x_train[:n],y_train[:n],0,BATCH_SIZE)
    step.group(iter(ds),tf.constant(2 if rem else 1,tf.int32)); _=step.accumulator.numpy()
    _assign_arrays(mv,saved_m); _assign_arrays(ov,saved_o); step.accumulator.assign(saved_acc)
    if TIMING_VERIFY_EXACT_RESTORE and not (_arrays_equal(mv,saved_m) and _arrays_equal(ov,saved_o)):
        raise RuntimeError('warm-up restore is not bit-exact')

def _make_eval(model):
    acc=tf.Variable(tf.zeros([3],tf.float32),trainable=False)
    @tf.function(reduce_retracing=True)
    def step(x,y):
        logits=model(x,training=False)
        ls=tf.keras.losses.sparse_categorical_crossentropy(y,logits,from_logits=True)
        c=tf.reduce_sum(tf.cast(tf.equal(tf.argmax(logits,-1,output_type=tf.int64),tf.cast(y,tf.int64)),tf.float32))
        return acc.assign_add(tf.stack([tf.reduce_sum(ls),c,tf.cast(tf.shape(y)[0],tf.float32)]))
    step.accumulator=acc
    return step

def evaluate(model,x,y,batch=EVAL_BATCH_SIZE):
    key=(id(model),HARDWARE_REVISION)
    if key not in _EVAL_CACHE: _EVAL_CACHE[key]=_make_eval(model)
    st=_EVAL_CACHE[key]; st.accumulator.assign(tf.zeros([3]))
    for xb,yb in make_eval_ds(x,y,batch): st(xb,yb)
    ls,c,n=st.accumulator.numpy().astype(np.float64)
    return {"loss":float(ls/n),"accuracy":float(c/n)}

def better(a,b):
    return a["accuracy"]>b["accuracy"]+1e-12 or (abs(a["accuracy"]-b["accuracy"])<=1e-12 and a["loss"]<b["loss"])

def snapshot_state(model,backbone,opt=None):
    return {"weights":[v.numpy().copy() for v in model.variables],"hardware":snapshot_hardware(backbone),
            "shift_specs":{i:backbone.bn(i).export_shift_spec() for i in range(1,10)},
            "shift_active":dict(backbone.shift_active),"mode":backbone.interface_mode,
            "optimizer":[v.numpy().copy() for v in opt.variables] if opt is not None else None}

def restore_state(model,backbone,state,opt=None):
    restore_hardware(backbone,state["hardware"])
    for i,s in state["shift_specs"].items():
        if s is None: backbone.bn(i).shift_spec=None
        else: backbone.bn(i).install_shift_spec(s)
    backbone.shift_active={int(i):bool(v) for i,v in state["shift_active"].items()}; backbone.interface_mode=state["mode"]
    if len(model.variables)!=len(state["weights"]): raise RuntimeError("checkpoint variable count mismatch")
    for v,a in zip(model.variables,state["weights"]): v.assign(a)
    if opt is not None and state.get("optimizer") is not None:
        saved=state["optimizer"]
        if len(saved)==len(opt.variables):
            for v,a in zip(opt.variables,saved): v.assign(a)
        elif len(saved)==2:
            for v,a in zip(opt.variables[:2],saved): v.assign(a)
            for v in opt.variables[2:]: v.assign(tf.zeros_like(v))
        else: raise RuntimeError("optimizer-state mismatch")
    bump_hardware_revision("restore_complete_state")

def save_checkpoint(model,backbone,label,metric,opt=None,kind="full"):
    t=time.perf_counter(); d=OUT_DIR/"checkpoints"/label; d.mkdir(parents=True,exist_ok=True)
    np.savez(d/"weights.npz",**{f"v{i:04d}":v.numpy() for i,v in enumerate(model.variables)})
    (d/"hardware.json").write_text(json.dumps(_plain(snapshot_hardware(backbone))))
    (d/"shiftadd_bn.json").write_text(json.dumps({str(i):_plain(backbone.bn(i).export_shift_spec()) for i in range(1,10)}))
    meta={"artifact":ARTIFACT_VERSION,"kind":kind,"completed_epochs":CUM_EPOCH,"metric":metric,
          "shift_active":backbone.shift_active,"mode":backbone.interface_mode,"label":label,
          "split_fingerprint":SPLIT_FINGERPRINT,"vtc_model_sha256":VTC_MODEL_SHA256,
          "logit_temperature_shift":FINAL_LOGIT_TEMPERATURE_SHIFT,"train_shiftadd_bias":TRAIN_SHIFTADD_BIAS,
          "optimizer_config":tf.keras.optimizers.serialize(opt) if opt is not None and SAVE_OPTIMIZER and kind=="full" else None}
    if meta["optimizer_config"] is not None:
        np.savez(d/"optimizer.npz",**{f"o{i:04d}":v.numpy() for i,v in enumerate(opt.variables)})
    (d/"meta.json").write_text(json.dumps(_plain(meta),indent=2))
    TIMING["checkpoint_sec"]+=time.perf_counter()-t
    return d

def load_full_checkpoint(backbone,path):
    global CUM_EPOCH
    d=Path(path)
    if d.is_file():
        target=OUT_DIR/"resume_input"; target.mkdir(exist_ok=True)
        with zipfile.ZipFile(d) as z:
            for name in z.namelist():
                dest=(target/name).resolve()
                if not dest.is_relative_to(target.resolve()): raise RuntimeError("unsafe checkpoint ZIP path")
            z.extractall(target)
        found=list(target.rglob("meta.json"))
        if len(found)!=1: raise RuntimeError("attach one full checkpoint ZIP, not a multi-checkpoint results ZIP")
        d=found[0].parent
    meta=json.loads((d/"meta.json").read_text())
    if not meta.get("optimizer_config") or not (d/"optimizer.npz").is_file(): raise RuntimeError("V10.6 requires a full Adam optimizer checkpoint")
    if meta["kind"]!="full": raise RuntimeError("front checkpoints require the temporary digital-tail architecture")
    if meta["split_fingerprint"]!=SPLIT_FINGERPRINT or meta["vtc_model_sha256"]!=VTC_MODEL_SHA256:
        raise RuntimeError("dataset split or inverter model differs from checkpoint")
    if meta["logit_temperature_shift"]!=FINAL_LOGIT_TEMPERATURE_SHIFT or meta["train_shiftadd_bias"]!=TRAIN_SHIFTADD_BIAS:
        raise RuntimeError("use the checkpoint's temperature and bias-training configuration")
    restore_hardware(backbone,json.loads((d/"hardware.json").read_text()))
    for i in range(1,10):
        if not backbone.bn(i).built: backbone.bn(i).build((None,backbone.bn(i).channels))
    specs=json.loads((d/"shiftadd_bn.json").read_text())
    for i,s in specs.items():
        if s is not None: backbone.bn(int(i)).install_shift_spec(s)
    backbone.shift_active={int(i):bool(v) for i,v in meta["shift_active"].items()}; backbone.interface_mode=meta["mode"]
    model=FullPHAXClassifier(backbone); _=model(tf.convert_to_tensor(x_val[:1]),training=False)
    with np.load(d/"weights.npz") as w:
        vals=[w[k] for k in sorted(w.files)]
    if len(vals)!=len(model.variables): raise RuntimeError("incompatible checkpoint variables")
    for v,a in zip(model.variables,vals): v.assign(a)
    opt=tf.keras.optimizers.deserialize(meta["optimizer_config"]) if meta.get("optimizer_config") else tf.keras.optimizers.Adam(SHIFTADD_QAT_LR)
    opt.build(model.trainable_variables)
    if meta.get("optimizer_config"):
        with np.load(d/"optimizer.npz") as w: vals=[w[k] for k in sorted(w.files)]
        if len(vals)!=len(opt.variables): raise RuntimeError("optimizer checkpoint mismatch")
        for v,a in zip(opt.variables,vals): v.assign(a)
    CUM_EPOCH=int(meta["completed_epochs"]); bump_hardware_revision("full_checkpoint_loaded")
    return model,opt


In [ ]:
# Diagnostics operate on TRAIN samples only. They never select the official test.
def gradient_probe(model,backbone,label):
    stats=[(b,b.moving_mean.numpy().copy(),b.moving_variance.numpy().copy()) for b in backbone._bns if b.built]
    with tf.GradientTape() as tape:
        logits=model(tf.convert_to_tensor(x_probe),training=True)
        loss=loss_fn(tf.convert_to_tensor(y_probe),logits)
    vv=model.trainable_variables; gg=tape.gradient(loss,vv); byid={id(v):g for v,g in zip(vv,gg)}
    for b,m,v in stats: b.moving_mean.assign(m); b.moving_variance.assign(v)
    rows=[]
    for i in range(1,11):
        vs=backbone.stage(i).trainable_variables; gs=[byid.get(id(v)) for v in vs]
        active=[g for g in gs if g is not None]; pcount=sum(int(np.prod(v.shape)) for v in vs)
        norm=float(tf.linalg.global_norm(active).numpy()) if active else 0.0
        zeros=sum(int(tf.reduce_sum(tf.cast(g==0,tf.int64)).numpy()) for g in active)
        present=sum(int(np.prod(g.shape)) for g in active)
        row={"label":label,"stage":i,"training_mode_probe":True,"loss":float(loss.numpy()),
             "analog_gradient_rms":norm/math.sqrt(max(pcount,1)),"analog_gradient_norm":norm,
             "parameter_count":pcount,"variables_with_none_gradient":sum(g is None for g in gs),
             "zero_fraction_present_gradients":zeros/max(present,1)}
        rows.append(row)
    GRADIENT_ROWS.extend(rows); pd.DataFrame(GRADIENT_ROWS).to_csv(OUT_DIR/"gradient_diagnostics.csv",index=False)
    print("GRADIENT_PROBE:",label,"loss",float(loss.numpy()))
    print(pd.DataFrame(rows)[["stage","analog_gradient_rms","variables_with_none_gradient"]].to_string(index=False))
    return rows

def hist_quantile(h,qmin,p):
    cumulative=np.cumsum(h,axis=1); target=p*np.maximum(cumulative[:,-1],1)
    return qmin+np.argmax(cumulative>=target[:,None],axis=1)

def _vtc_rows(raw,spec,label,stage,chain):
    C=raw.shape[-1]
    flat=np.asarray(raw,np.float64).reshape(-1,C)
    idx=np.linspace(0,len(flat)-1,min(VTC_SAMPLE_ROWS,len(flat)),dtype=int)
    z=flat[idx];jac=np.ones_like(z);reference=np.ones(C)
    for ids in spec['lut_index_levels']:
        deriv=_lut_eval_numpy(z,ids,derivative=True)
        jac*=deriv
        reference*=RAW_SLOPE_ABS[np.asarray(ids,np.int32)]
        z=_lut_eval_numpy(z,ids)
    ratio=np.abs(jac)/np.maximum(reference[None,:],1e-15)
    return [{'label':label,'stage':stage,'chain':chain,'channel':ch,
             'gain_abs_p10':float(np.percentile(np.abs(jac[:,ch]),10)),
             'gain_abs_median':float(np.median(np.abs(jac[:,ch]))),
             'relative_slope_median':float(np.median(ratio[:,ch])),
             'small_slope_fraction':float(np.mean(ratio[:,ch]<.1))} for ch in range(C)]

def deep_audit(backbone,x,y,label):
    adc={i:np.zeros((STAGE_SPECS[f"stage{i}"]["cout"],255),np.int64) for i in range(1,11)}
    dac={i:np.zeros((STAGE_SPECS[f"stage{i}"]["cout"],256),np.int64) for i in range(1,10)}
    vmax={i:0.0 for i in range(1,11)}; codes10=[]; volts10=[]; vtc=[]
    for j in range(0,len(x),CAL_BATCH_SIZE):
        z=tf.convert_to_tensor(x[j:j+CAL_BATCH_SIZE],tf.float32)
        for i in range(1,11):
            lin,tr=backbone.stage(i)(z,return_trace=True); q=_signed_adc_code_tf(lin).numpy()
            adc[i]+=code_histogram(q,-127,127)
            vmax[i]=max(vmax[i],max(float(tf.reduce_max(tf.abs(v)).numpy()) for v in tr.values()))
            if j==0:
                vtc.extend(_vtc_rows(tr["main_raw"].numpy(),backbone.stage(i).hardware["main_chain"],label,i,"main"))
                if i>1: vtc.extend(_vtc_rows(tr["averager_raw"].numpy(),backbone.stage(i).hardware["post_chain"],label,i,"post"))
            if i==10: codes10.append(q); volts10.append(lin.numpy()); continue
            z=backbone.interface_after(i,lin,training=False,quantize=True,pool_gap=True)
            dc=_unsigned_dac_code_tf(z).numpy(); dac[i]+=code_histogram(dc,0,255)
    if max(vmax.values())>V_LIMIT+VOLTAGE_TOL: raise RuntimeError("audit detected physical voltage violation")
    rows=[]
    for i in range(1,11):
        h=adc[i]; cnt=np.maximum(h.sum(1),1); p01=hist_quantile(h,-127,.01); p99=hist_quantile(h,-127,.99)
        for ch in range(h.shape[0]):
            row={"label":label,"stage":i,"channel":ch,"adc_unique":int(np.count_nonzero(h[ch])),
                 "adc_p01":int(p01[ch]),"adc_p99":int(p99[ch]),"adc_p01_p99_span_codes":int(p99[ch]-p01[ch]),
                 "adc_near_rail_fraction":float(h[ch,np.abs(SHIFT_Q)>=.95*127].sum()/cnt[ch]),
                 "physical_node_abs_max_v_stage":vmax[i]}
            if i<10:
                dh=dac[i][ch]; n=max(dh.sum(),1); nz=np.flatnonzero(dh[1:])+1
                row.update(dac_unique=int(np.count_nonzero(dh)),dac_zero_fraction=float(dh[0]/n),
                           dac_p99=int(hist_quantile(dh[None,:],0,.99)[0]),
                           dac_positive_unique=int(len(nz)),shift_active=backbone.uses_shift(i))
            rows.append(row)
    CODE_ROWS.extend(rows); VTC_ROWS.extend(vtc)
    pd.DataFrame(CODE_ROWS).to_csv(OUT_DIR/"adc_dac_per_channel.csv",index=False)
    pd.DataFrame(VTC_ROWS).to_csv(OUT_DIR/"vtc_derivative_per_channel.csv",index=False)
    q=np.concatenate(codes10); v=np.concatenate(volts10); pred=q.argmax(-1); margin=np.sort(q,axis=-1)[:,-1]-np.sort(q,axis=-1)[:,-2]
    logit={"label":label,"accuracy_train_diagnostic":float(np.mean(pred==y)),
           "median_adc_code_margin":float(np.median(margin)),"top1_tie_fraction":float(np.mean(margin==0)),
           "final_ADC_argmax_change_fraction":float(np.mean(pred!=v.argmax(-1)))}
    for temperature in (1,8,16):
        logit[f"crossentropy_T{temperature}"]=float(tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(y,q/temperature,from_logits=True)).numpy())
    (OUT_DIR/f"logit_diagnostic_{label}.json").write_text(json.dumps(logit,indent=2))
    brief=pd.DataFrame(rows).groupby("stage").agg(adc_unique_median=("adc_unique","median"),adc_span_median=("adc_p01_p99_span_codes","median"))
    print("DEEP_AUDIT:",label); print(brief.to_string()); print("LOGIT_DIAGNOSTIC:",logit)
    return {"physical_node_max_by_stage":vmax,"logit":logit}

def trained_gain_audit(backbone,x,label):
    rows=[]
    for i in range(1,11):
        inp=stage_input_for(backbone,x[:CAL_BATCH_SIZE],i)
        _,tr=backbone.stage(i)(tf.convert_to_tensor(inp),return_trace=True)
        for key,rawkey in (("main_chain","main_raw"),("post_chain","averager_raw")):
            if key not in backbone.stage(i).hardware: continue
            peak=_per_column_max_abs(tr[rawkey].numpy()); req=V_LIMIT/np.maximum(peak,1e-9)
            current=np.asarray(backbone.stage(i).hardware[key]["column_target_gain"])
            for ch in range(len(peak)):
                rows.append({"label":label,"stage":i,"chain":key,"channel":ch,"requested_gain_on_probe":req[ch],
                             "fixed_target_gain":current[ch],"requested_over_fixed":req[ch]/max(current[ch],1e-12)})
    pd.DataFrame(rows).to_csv(OUT_DIR/f"trained_gain_{label}.csv",index=False)
    return rows

def flush_reports():
    for name,rows in (("evaluation_history",EVAL_ROWS),("events",EVENTS),("chain_summary",CHAIN_ROWS),
                      ("calibration_history",CALIBRATION_ROWS),("epoch_timing",TIMING_ROWS),("conversion_history",CONVERSION_ROWS)):
        if rows: pd.DataFrame(rows).to_csv(OUT_DIR/f"{name}.csv",index=False)

def run_phase(model,backbone,opt,label,lr,patience,max_epochs,seed_base,kind="full",recover_target=None):
    global CUM_EPOCH
    opt.learning_rate.assign(lr); step=make_train_step(model,opt)
    metric=timed("validation_selection_sec",evaluate,model,x_val,y_val)
    best_metric=dict(metric); best_state=snapshot_state(model,backbone,opt); wait=0; completed=0
    save_checkpoint(model,backbone,label+"_best",best_metric,opt,kind)
    for epoch in range(1,max_epochs+1):
        t=time.perf_counter()
        try: tr=train_epoch(model,step,seed_base+epoch)
        except tf.errors.InvalidArgumentError as exc:
            restore_state(model,backbone,best_state,opt)
            save_checkpoint(model,backbone,label+"_best",best_metric,opt,kind)
            record_event("training_assertion_restored_best",phase=label,reason=str(exc)[:1000])
            raise RuntimeError("training assertion failed; the preceding valid checkpoint was preserved") from exc
        train_sec=time.perf_counter()-t; TIMING["train_forward_backward_sec"]+=train_sec
        CUM_EPOCH+=1; completed+=1
        if any(backbone.shift_active.values()): shiftadd_hard_legality_audit(backbone,label+"_latest")
        t=time.perf_counter(); va=evaluate(model,x_val,y_val); val_sec=time.perf_counter()-t
        TIMING["validation_selection_sec"]+=val_sec
        row={"cumulative_optimizer_epoch":CUM_EPOCH,"phase":label,"local_epoch":epoch,"train_loss":tr["loss"],
             "train_accuracy":tr["accuracy"],"val_loss":va["loss"],"val_accuracy":va["accuracy"],"lr":float(lr),
             "active_shift_stages":str([i for i in range(1,10) if backbone.uses_shift(i)])}
        EVAL_ROWS.append(row); TIMING_ROWS.append({"phase":label,"epoch":CUM_EPOCH,"train_epoch_sec":train_sec,"validation_sec":val_sec})
        print("EVAL:",row,"train_sec=",round(train_sec,2),flush=True)
        if better(va,best_metric):
            best_metric=dict(va); best_state=snapshot_state(model,backbone,opt); wait=0
            save_checkpoint(model,backbone,label+"_best",best_metric,opt,kind)
        else: wait+=1
        flush_reports()
        if kind=="full" and epoch==1: timed("diagnostic_sec",gradient_probe,model,backbone,label+"_epoch1")
        if recover_target is not None and va["accuracy"]>=recover_target: break
        if wait>=patience: break
    restore_state(model,backbone,best_state,opt)
    save_checkpoint(model,backbone,label+"_best",best_metric,opt,kind)
    record_event("phase_finished",phase=label,epochs_run=completed,best=best_metric)
    return best_metric,completed


# Read-only probes: no GradientTape, optimizer update, augmentation, or BN updates.
# Error/entropy statistics are diagnostic proxies, not estimates of lost task information.
INFORMATION_ROWS=[]
INFORMATION_HEAD_ROWS=[]
INFORMATION_HEAD_FIT_ROWS=[]
INFORMATION_CHANNEL_ROWS=[]
INFORMATION_STAGE_ROWS=[]
INFORMATION_VTC_ROWS=[]
TIMING['information_diagnostic_sec']=0.0
_information_idx=balanced_indices(y_train,min(INFORMATION_TRAIN_IMAGES,len(y_train)),SEED+140701)
INFORMATION_X=x_train[_information_idx]
INFORMATION_Y=y_train[_information_idx]
INFORMATION_PROBE_SHA256=hashlib.sha256(np.asarray(_information_idx,np.int64).tobytes()).hexdigest()

def _information_csv(rows,name):
    if rows:pd.DataFrame(rows).to_csv(OUT_DIR/name,index=False)

class InformationPair:
    """Streaming per-channel paired moments; float64 reductions, bounded storage."""
    def __init__(self):self.state=None
    def add(self,reference,actual):
        a=np.asarray(reference,np.float64);b=np.asarray(actual,np.float64)
        if a.shape!=b.shape:raise ValueError(('information probe shape mismatch',a.shape,b.shape))
        a=a.reshape(-1,a.shape[-1]);b=b.reshape(-1,b.shape[-1])
        v=[np.full(a.shape[1],len(a),np.float64),a.sum(0),b.sum(0),
           (a*a).sum(0),(b*b).sum(0),(a*b).sum(0),((a-b)**2).sum(0),
           ((a<=0)!=(b<=0)).sum(0).astype(np.float64)]
        if self.state is None:self.state=v
        else:
            for dst,src in zip(self.state,v):dst+=src
    def rows(self,label,stage,comparison,data_source):
        n,sa,sb,saa,sbb,sab,se,sz=self.state
        ma=sa/n;mb=sb/n;va=np.maximum(saa/n-ma*ma,0);vb=np.maximum(sbb/n-mb*mb,0)
        cov=sab/n-ma*mb;den=np.sqrt(va*vb)
        corr=np.divide(cov,den,out=np.zeros_like(cov),where=den>1e-15)
        return [dict(label=label,stage=int(stage),comparison=comparison,data_source=data_source,
            channel=c,observations=int(n[c]),mse=float(se[c]/n[c]),
            nmse=float(se[c]/max(saa[c],1e-12)),correlation=float(corr[c]) if den[c]>1e-15 else None,
            reference_mean=float(ma[c]),actual_mean=float(mb[c]),
            reference_variance=float(va[c]),actual_variance=float(vb[c]),
            zero_mask_change_fraction=float(sz[c]/n[c])) for c in range(len(n))]

def _information_pool(stage,z):
    if int(stage) in (2,4,6):z=tf.nn.max_pool2d(z,2,2,'SAME')
    if int(stage)==8:z=tf.reduce_mean(z,axis=(1,2))
    return z

def _information_float_values(backbone,stage,lin):
    # Fixed inference affine using the current BN05 bounds. Used only on a fresh
    # Float layer / at conversion, never as a reference to a post-QAT frozen BN.
    b=backbone.bn(stage);lo,hi=backbone.stage(stage).hard_analog_output_bounds_tf()
    a,c,_=b.effective_affine_voltage(signed_adc_qdq_value(lo).numpy(),signed_adc_qdq_value(hi).numpy())
    a=tf.constant(a,tf.float32);c=tf.constant(c,tf.float32)
    adc=signed_adc_qdq_value(lin)
    no_adc=_information_pool(stage,tf.nn.relu(a*lin+c))
    with_adc=_information_pool(stage,tf.nn.relu(a*adc+c))
    return no_adc,with_adc

def information_local_fit(backbone,stage,xin,target,label):
    # The original local holdout is distinct from its 6000 local fit images.
    start=V12_LOCAL_TRAIN_SAMPLES
    count=min(INFORMATION_LOCAL_HOLDOUT_IMAGES,len(xin)-start)
    if count<=0:raise ValueError('No local holdout data for information probe')
    meters={k:InformationPair() for k in ['teacher_vs_analog_float_no_ADC_DAC',
             'teacher_vs_float_ADC_before_DAC','teacher_vs_deployed_interface']}
    for j in range(start,start+count,INFORMATION_BATCH_SIZE):
        stop=min(j+INFORMATION_BATCH_SIZE,start+count)
        x=tf.convert_to_tensor(xin[j:stop],tf.float32);ref=target[j:stop]
        lin=backbone.stage(stage)(x,training=False)
        ideal,adc=_information_float_values(backbone,stage,lin)
        deployed=backbone.interface_after(stage,lin,training=False,quantize=True,pool_gap=True)
        for meter,pred in zip(meters.values(),[ideal,adc,deployed]):meter.add(ref,pred.numpy())
    for name,meter in meters.items():
        INFORMATION_ROWS.extend(meter.rows(label,stage,name,'local_fit_holdout_subset'))
    _information_csv(INFORMATION_ROWS,'information_paired_errors.csv')

def information_conversion(backbone,stages,label):
    # Same current prefix inputs for all paired alternatives. Newly converted
    # stages only: gamma/beta and moving moments still describe their Float fit.
    stages=tuple(map(int,stages));meters={i:{k:InformationPair() for k in (
        'ADC_effect_through_float_BN','float_to_continuous_shift_coefficients',
        'separate_integer_term_rounding','total_float_to_shiftadd_before_DAC',
        'DAC_rounding_after_shiftadd')} for i in stages}
    for j in range(0,len(INFORMATION_X),INFORMATION_BATCH_SIZE):
        z=tf.convert_to_tensor(INFORMATION_X[j:j+INFORMATION_BATCH_SIZE],tf.float32)
        for i in range(1,max(stages)+1):
            lin=backbone.stage(i)(z,training=False)
            deployed=backbone.interface_after(i,lin,training=False,quantize=True,pool_gap=True)
            if i in stages:
                ideal,flt=_information_float_values(backbone,i,lin)
                b=backbone.bn(i);spec=b.export_shift_spec()
                m=sum(np.asarray(spec[k]['sign'])*np.exp2(spec[k]['exp']) for k in ('term1','term2'))
                q=signed_adc_code_ste(lin)
                continuous=_information_pool(i,tf.nn.relu(q*tf.constant(m,tf.float32)+tf.constant(spec['bias'],tf.float32)))*V_LIMIT/255.
                integer=_information_pool(i,b.shiftadd(q))*V_LIMIT/255.
                pairs=[(ideal,flt),(flt,continuous),(continuous,integer),(flt,integer),(integer,deployed)]
                for meter,(a,c) in zip(meters[i].values(),pairs):meter.add(a.numpy(),c.numpy())
            z=deployed
    for i in stages:
        for name,meter in meters[i].items():
            INFORMATION_ROWS.extend(meter.rows(label,i,name,'fixed_TRAIN_probe_at_conversion'))
    _information_csv(INFORMATION_ROWS,'information_paired_errors.csv')

def _information_prefix9(backbone,x):
    z=tf.convert_to_tensor(x,tf.float32)
    for i in range(1,10):
        lin=backbone.stage(i)(z,training=False)
        z=backbone.interface_after(i,lin,training=False,quantize=True,pool_gap=True)
    return z

def information_head(backbone,suffix,label):
    # The frozen internal digital head is an intervention on the SAME prefix,
    # not the external 91.93% digital baseline or a claimed accuracy ceiling.
    values={k:[] for k in ('digital_with_bias','digital_without_bias','physical_volts','physical_ADC')}
    bias=suffix.out.bias.numpy().copy() if suffix.out.use_bias else np.zeros(10,np.float32)
    for j in range(0,len(INFORMATION_X),INFORMATION_BATCH_SIZE):
        z=_information_prefix9(backbone,INFORMATION_X[j:j+INFORMATION_BATCH_SIZE])
        digital=suffix.out(z,training=False).numpy()
        volts=backbone.stage(10)(z,training=False)
        adc=signed_adc_code_ste(volts).numpy()
        for name,v in zip(values,[digital,digital-bias,volts.numpy(),adc]):values[name].append(v)
    values={k:np.concatenate(v).astype(np.float64) for k,v in values.items()}
    # Fit a per-class affine ONLY on the first half of this TRAIN probe.
    # It is diagnostic-only, unconstrained, and NEVER used in deployed inference.
    split=len(INFORMATION_Y)//2
    if split<10:raise ValueError('Head information probe needs at least 20 TRAIN images')
    x=values['physical_ADC'][:split];y=values['digital_with_bias'][:split]
    mx=x.mean(0);my=y.mean(0);vx=np.mean((x-mx)**2,0)
    slope=np.sum((x-mx)*(y-my),0)/(np.sum((x-mx)**2,0)+1e-8)
    intercept=my-slope*mx
    values['diagnostic_only_fitted_affine']=values['physical_ADC']*slope+intercept
    ref=values['digital_with_bias'][split:].argmax(1)
    for name,logits in values.items():
        logits=logits[split:];pred=logits.argmax(1)
        ordered=np.sort(logits,axis=1);margin=ordered[:,-1]-ordered[:,-2]
        INFORMATION_HEAD_ROWS.append(dict(label=label,variant=name,
            data_source='second_half_of_fixed_TRAIN_probe',images=len(pred),fit_images=split,
            accuracy=float(np.mean(pred==INFORMATION_Y[split:])),
            disagreement_with_current_internal_digital_head=float(np.mean(pred!=ref)),
            top1_tie_fraction=float(np.mean(margin==0)),median_margin_native_units=float(np.median(margin)),
            diagnostic_only=(name=='diagnostic_only_fitted_affine'),
            digital_bias_abs_max=float(np.max(np.abs(bias)))))
    for c in range(len(slope)):
        INFORMATION_HEAD_FIT_ROWS.append(dict(label=label,channel=c,
            diagnostic_affine_slope=float(slope[c]),diagnostic_affine_bias=float(intercept[c]),
            ADC_fit_variance=float(vx[c]),digital_head_bias=float(bias[c]),diagnostic_only=True))
    _information_csv(INFORMATION_HEAD_ROWS,'information_head_counterfactuals.csv')
    _information_csv(INFORMATION_HEAD_FIT_ROWS,'information_head_affine_fit.csv')

def _information_hist_stats(h,qmin):
    total=max(int(h.sum()),1);codes=qmin+np.arange(len(h));p=h/total
    used=p>0;mean=float(np.sum(p*codes));variance=float(np.sum(p*(codes-mean)**2))
    return dict(observations=int(h.sum()),unique_codes=int(np.count_nonzero(h)),
        code_entropy_bits=float(-np.sum(p[used]*np.log2(p[used]))),
        code_mean=mean,code_variance=variance,
        p01=int(qmin+np.searchsorted(np.cumsum(h),.01*total)),
        p99=int(qmin+np.searchsorted(np.cumsum(h),.99*total)),
        zero_fraction=float(h[-qmin]/total))

def information_final_boundaries(backbone,label):
    ah={i:np.zeros((STAGE_SPECS[f'stage{i}']['cout'],255),np.int64) for i in range(1,11)}
    dh={i:np.zeros((STAGE_SPECS[f'stage{i}']['cout'],256),np.int64) for i in range(1,10)}
    adc_error={i:InformationPair() for i in range(1,11)}
    dac_error={i:InformationPair() for i in range(1,10)}
    maxima={i:0. for i in range(1,11)};out_of_range={i:0 for i in range(1,10)}
    for j in range(0,len(INFORMATION_X),INFORMATION_BATCH_SIZE):
        z=tf.convert_to_tensor(INFORMATION_X[j:j+INFORMATION_BATCH_SIZE],tf.float32)
        for i in range(1,11):
            # Full node traces are retained only for the first tiny batch.
            if j==0:
                lin,tr=backbone.stage(i)(z,training=False,return_trace=True)
                maxima[i]=max(float(tf.reduce_max(tf.abs(v)).numpy()) for v in tr.values())
                INFORMATION_VTC_ROWS.extend(_vtc_rows(tr['main_raw'].numpy(),backbone.stage(i).hardware['main_chain'],label,i,'main'))
                if i>1:INFORMATION_VTC_ROWS.extend(_vtc_rows(tr['averager_raw'].numpy(),backbone.stage(i).hardware['post_chain'],label,i,'post'))
                del tr
            else:lin=backbone.stage(i)(z,training=False)
            q=signed_adc_code_ste(lin);reconstructed=q*V_LIMIT/127.
            ah[i]+=code_histogram(q.numpy(),-127,127)
            adc_error[i].add(lin.numpy(),reconstructed.numpy())
            if i==10:continue
            before=backbone.interface_after(i,lin,training=False,quantize=False,pool_gap=True)
            z=backbone.interface_after(i,lin,training=False,quantize=True,pool_gap=True)
            raw_code=before.numpy()*255./V_LIMIT
            out_of_range[i]+=int(np.sum((raw_code < -1e-4)|(raw_code > 255.+1e-4)))
            dh[i]+=code_histogram(_unsigned_dac_code_tf(z).numpy(),0,255)
            dac_error[i].add(before.numpy(),z.numpy())
    for i in range(1,11):
        for kind,h,qmin in [('ADC',ah[i],-127)]+([('DAC',dh[i],0)] if i<10 else []):
            for c in range(len(h)):
                row=dict(label=label,stage=i,channel=c,boundary=kind,
                         sample_source='fixed_TRAIN_probe',**_information_hist_stats(h[c],qmin))
                count=max(int(h[c].sum()),1)
                row['rail_code_fraction']=float((h[c,0]+h[c,-1])/count) if kind=='ADC' else float(h[c,255]/count)
                row['near_upper_rail_fraction']=float(h[c,-6:].sum()/count)
                INFORMATION_CHANNEL_ROWS.append(row)
            total=max(int(h.sum()),1)
            INFORMATION_STAGE_ROWS.append(dict(label=label,stage=i,boundary=kind,channels=len(h),
                constant_channels=int(np.sum(np.count_nonzero(h,axis=1)<=1)),
                allzero_channels=int(np.sum(h[:, -qmin]==h.sum(1))),
                median_entropy_bits=float(np.median([_information_hist_stats(v,qmin)['code_entropy_bits'] for v in h])),
                median_p99=float(np.median([_information_hist_stats(v,qmin)['p99'] for v in h])),
                upper_rail_fraction=float(h[:,-1].sum()/total),
                near_upper_rail_fraction=float(h[:,-6:].sum()/total),
                raw_DAC_out_of_range_count=out_of_range.get(i) if kind=='DAC' else None,
                physical_node_max_v_first_small_batch=maxima[i]))
        INFORMATION_ROWS.extend(adc_error[i].rows(label,i,'final_ADC_voltage_quantization','fixed_TRAIN_probe'))
        if i<10:INFORMATION_ROWS.extend(dac_error[i].rows(label,i,'final_DAC_voltage_quantization','fixed_TRAIN_probe'))
    _information_csv(INFORMATION_ROWS,'information_paired_errors.csv')
    _information_csv(INFORMATION_CHANNEL_ROWS,'information_boundary_channels.csv')
    _information_csv(INFORMATION_STAGE_ROWS,'information_boundary_stages.csv')
    _information_csv(INFORMATION_VTC_ROWS,'information_vtc_slopes.csv')

def information_transition_ledger():
    rows=[]
    def add(kind,stage,before,after):
        rows.append(dict(event=kind,stage=str(stage),data_source='full_5000_validation',
            before_accuracy=float(before),after_accuracy=float(after),delta_pp=100.*float(after-before)))
    for r in V12_ENDPOINT_ROWS:
        i=int(r['endpoint'])
        if i==2:
            before=r['float_refined']['accuracy'];immediate=r['transition']['accuracy']
        else:
            t=next(x for x in TRANSITION_ROWS if x['label']==f'endpoint{i}_after_local_match')
            add('replace_digital_block_by_locally_matched_physical_float',i,t['previous_endpoint_accuracy'],r['transition']['accuracy'])
            add('float_classification_recovery',i,r['transition']['accuracy'],r['float_recovered']['accuracy'])
            before=r['float_recovered']['accuracy'];immediate=r['shift_immediate']['accuracy']
        add('Float_to_ShiftAdd_conversion',i,before,immediate)
        add('classification_QAT_recovery',i,immediate,r['shift_selected']['accuracy'])
    phase={r['phase']:r for r in PHASE_ROWS}
    h=phase['v12_stage10_head']
    add('replace_internal_digital_output_by_physical_S10',10,V12_ENDPOINT_ROWS[-1]['shift_selected']['accuracy'],h['initial_validation']['accuracy'])
    for name in ('v12_stage10_head','v12_tail_coadapt','v12_guarded_full_polish'):
        p=phase[name];add(name,10,p['initial_validation']['accuracy'],p['validation']['accuracy'])
    pd.DataFrame(rows).to_csv(OUT_DIR/'information_transition_ledger.csv',index=False)
    return rows

def information_write_summary():
    ledger=information_transition_ledger()
    report=dict(probe_train_images=len(INFORMATION_X),probe_indices_sha256=INFORMATION_PROBE_SHA256,
        probe_batch_size=INFORMATION_BATCH_SIZE,local_holdout_probe_images=INFORMATION_LOCAL_HOLDOUT_IMAGES,
        entropy_is_not_task_information=True,paired_error_contributions_are_not_additive=True,
        stale_float_BN_not_used_as_post_QAT_reference=True,
        head_affine_is_diagnostic_only=True,official_test_used_for_probes=False,
        internal_digital_head_is_not_external_baseline=True,
        transition_ledger=ledger,head_counterfactuals=INFORMATION_HEAD_ROWS,
        head_recovery_config=INFORMATION_HEAD_CONFIG)
    (OUT_DIR/'information_summary.json').write_text(json.dumps(_plain(report),indent=2))
    print('INFORMATION: largest negative validation transitions')
    print(pd.DataFrame(sorted(ledger,key=lambda r:r['delta_pp'])[:5]).to_string(index=False))
    return report


In [ ]:
# Small runtime checks, before the hours-long CNN training.
def run_self_checks():
    s={"term1":{"sign":[1,1,1,-1,1],"exp":[0,-1,1,-2,-1]},
       "term2":{"sign":[1,1,-1,1,0],"exp":[-2,-3,-2,-3,-1]},
       "bias":[64,80,0,80,80],"m_effective":[1.25,.625,1.75,-.125,.5]}
    b=HardwareConvertibleBN(5,"selfcheck_bn"); b.install_shift_spec(s)
    q=tf.Variable(np.repeat(np.array([-3,-1,0,1,3],np.float32)[:,None],5,axis=1))
    with tf.GradientTape() as tape: yy=b.shiftadd(q); total=tf.reduce_sum(yy)
    grad=tape.gradient(total,q).numpy()
    actual_bias=b.current_bias().numpy(); raw=np.zeros_like(q.numpy())
    for key in ("term1","term2"):
        raw+=np.rint(q.numpy()*np.exp2(np.array(s[key]["exp"],np.float32)))*np.array(s[key]["sign"])
    expected=np.maximum(raw+actual_bias,0)
    np.testing.assert_array_equal(yy.numpy(),expected)
    expected_grad=np.asarray(s["m_effective"])[None,:]*(raw+actual_bias>0)
    np.testing.assert_allclose(grad,expected_grad,atol=1e-6)
    allq=tf.constant(np.repeat(SHIFT_Q[:,None],5,axis=1),tf.float32)
    ref=np.maximum(shift_tables(s)+actual_bias[None,:],0)
    np.testing.assert_array_equal(b.shiftadd(allq).numpy(),ref)
    # Fused differential expression retains separate hardware branches and normalization.
    st=VGGPHAXStage("stage1"); xx=tf.constant(np.random.default_rng(123).uniform(-.5,.5,(2,4,4,3)),tf.float32)
    p=tf.image.extract_patches(xx,[1,3,3,1],[1,1,1,1],[1,1,1,1],"SAME")
    sp,sn=conductance(st.theta_wp),conductance(st.theta_wn); bp,bn=conductance(st.theta_bp),conductance(st.theta_bn)
    den=tf.reduce_sum(sp,0)+tf.reduce_sum(sn,0)+bp+bn+8*SIG_MIN_US
    old=tf.einsum("bhwi,io->bhwo",p,sp/(den[None,:]+EPS))+tf.einsum("bhwi,io->bhwo",-p,sn/(den[None,:]+EPS))
    old+=(V_LIMIT*bp-V_LIMIT*bn)/(den+EPS)
    new=st.raw_main(xx)[0][0]; error=float(tf.reduce_max(tf.abs(old-new)).numpy())
    if error>2e-6: raise RuntimeError("fused crossbar mismatch")
    result={"right_shift_STE":"passed","all_255_ADC_codes":"passed","fused_raw_error_v":error,
            "rounding":ROUNDING_CONVENTION,"full_Kaggle_training_executed_locally":False}
    (OUT_DIR/"runtime_self_checks.json").write_text(json.dumps(result,indent=2)); print("SELF_CHECKS:",result)
    return result
SELF_CHECK_RESULT=run_self_checks()


## کنترل‌کننده hybrid و پارامترهای V12

فاز endpoint=2 هنوز مسئله classification را از صفر حل می‌کند. از endpointهای بعدی، هر stage ابتدا با teacher داخلی هم‌مقیاس و local-match می‌شود؛ بنابراین recovery کامل شبکه به حداکثر دو epoch محدود شده است. `save_best=False` در فازهای میانی فقط از نوشتن checkpointهای بزرگ جلوگیری می‌کند؛ بهترین state همچنان در RAM نگه داشته و restore می‌شود.


In [ ]:

class ProgressiveDigitalSuffix(TemporaryFrontDigitalTail):
    def call_from(self,endpoint,z,training=False):
        endpoint=int(endpoint)
        if endpoint<4:
            z=tf.nn.relu(self.b3(self.c3(z),training=training))
            z=tf.nn.relu(self.b4(self.c4(z),training=training))
            z=tf.nn.max_pool2d(z,2,2,'SAME')
        if endpoint<6:
            z=tf.nn.relu(self.b5(self.c5(z),training=training))
            z=tf.nn.relu(self.b6(self.c6(z),training=training))
            z=tf.nn.max_pool2d(z,2,2,'SAME')
        if endpoint<8:
            z=tf.nn.relu(self.b7(self.c7(z),training=training))
            z=tf.nn.relu(self.b8(self.c8(z),training=training))
            z=tf.reduce_mean(z,axis=(1,2))
        # endpoint 8 already passed the deployed GAP inside physical_prefix.
        if endpoint<9:
            z=tf.nn.relu(self.b9(self.fc(z),training=training))
        if endpoint<10:
            z=self.out(z)
        return z


def physical_prefix(backbone,x,endpoint,training=False):
    endpoint=int(endpoint);z=x
    if endpoint not in PROGRESSIVE_ENDPOINTS:
        raise ValueError('unsupported progressive endpoint')
    for i in range(1,endpoint+1):
        lin=backbone.stage(i)(z,training=training)
        # Every prefix boundary is a deployed boundary: ADC + BN/ReLU + DAC.
        z=backbone.interface_after(i,lin,training=training,quantize=True,pool_gap=True)
    return z


class ProgressiveHybridClassifier(tf.keras.Model):
    def __init__(self,b,suffix):
        super().__init__(name='v11_progressive_hybrid')
        self.backbone=b;self.suffix=suffix;self.endpoint=2
    def set_endpoint(self,endpoint):
        endpoint=int(endpoint)
        if endpoint not in PROGRESSIVE_ENDPOINTS:raise ValueError(endpoint)
        self.endpoint=endpoint
        bump_hardware_revision('progressive_endpoint_'+str(endpoint))
    def call(self,x,training=False):
        z=physical_prefix(self.backbone,x,self.endpoint,training=training)
        return self.suffix.call_from(self.endpoint,z,training=training)


def phase_checkpoint_kind(model,label):
    return 'full' if isinstance(model,FullPHAXClassifier) else 'hybrid_'+label


def run_phase_v111(model,backbone,opt,label,lr,patience,max_epochs,seed_base,
                   save_best=True,diagnose_gradient=False):
    global CUM_EPOCH
    opt.learning_rate.assign(float(lr));step=make_train_step(model,opt)
    initial=timed('validation_selection_sec',evaluate,model,x_val,y_val)
    best_metric=dict(initial);best_state=snapshot_state(model,backbone,opt)
    best_epoch=0;best_prefix_sec=0.0;prefix_sec=0.0;wait=0;rows=[]
    kind=phase_checkpoint_kind(model,label)
    if save_best:save_checkpoint(model,backbone,label+'_epoch0',best_metric,opt,kind)
    warm_t=time.perf_counter()
    if int(max_epochs)>0:warmup_classification_step(model,opt,step)
    warmup_sec=time.perf_counter()-warm_t;TIMING['compile_warmup_sec']+=warmup_sec
    build_total=0.0
    for epoch in range(1,int(max_epochs)+1):
        try:tr,ti=train_epoch_timed(model,step,seed_base+epoch)
        except tf.errors.InvalidArgumentError as exc:
            restore_state(model,backbone,best_state,opt)
            record_event('training_assertion_restored_best',phase=label,reason=str(exc)[:1000])
            raise RuntimeError('training assertion failed; best valid state restored') from exc
        sec=ti['train_sec'];TIMING['train_forward_backward_sec']+=sec
        TIMING['input_pipeline_build_sec']+=ti['pipeline_build_sec'];build_total+=ti['pipeline_build_sec']
        if not isinstance(model,FullPHAXClassifier):TIMING['suffix_training_sec']+=sec
        prefix_sec+=sec;CUM_EPOCH+=1
        if any(backbone.shift_active.values()):
            timed('physical_audit_sec',shiftadd_hard_legality_audit,backbone,label+'_epoch'+str(epoch))
        va=timed('validation_selection_sec',evaluate,model,x_val,y_val)
        timed('bn_experiment_diagnostic_sec',bn_record_parameters,backbone,label+'_epoch'+str(epoch))
        if better(va,best_metric):
            best_metric=dict(va);best_state=snapshot_state(model,backbone,opt)
            best_epoch=epoch;best_prefix_sec=prefix_sec;wait=0
            if save_best:save_checkpoint(model,backbone,label+'_best',best_metric,opt,kind)
        else:wait+=1
        row={'cumulative_optimizer_epoch':CUM_EPOCH,'phase':label,'local_epoch':epoch,
             'train_loss':tr['loss'],'train_accuracy':tr['accuracy'],
             'val_loss':va['loss'],'val_accuracy':va['accuracy'],'lr':float(lr),
             'train_epoch_sec':sec,'pipeline_build_sec':ti['pipeline_build_sec'],'train_prefix_sec':prefix_sec,
             'best_local_epoch':best_epoch,'best_prefix_training_sec':best_prefix_sec,
             'active_shift_stages':str([i for i in range(1,10) if backbone.uses_shift(i)])}
        EVAL_ROWS.append(row);TIMING_ROWS.append(row.copy());rows.append(row);flush_reports()
        print('V11 EVAL',row,flush=True)
        if diagnose_gradient and epoch==1:
            timed('diagnostic_sec',gradient_probe,model,backbone,label+'_epoch1')
        if wait>=int(patience):break
    restore_state(model,backbone,best_state,opt)
    restored_check=None
    if VERIFY_RESTORED_VALIDATION and len(rows)>0:
        chk=timed('restore_verification_sec',evaluate,model,x_val,y_val)
        restored_check={'accuracy':chk['accuracy'],'loss':chk['loss'],
                        'matches_selected':bool(abs(chk['accuracy']-best_metric['accuracy'])<1e-12)}
        if not restored_check['matches_selected']:
            record_event('restored_validation_mismatch',phase=label,selected=best_metric,restored=chk)
            warnings.warn(f'{label}: restored state validation {chk} != selected {best_metric}')
    summary={'phase':label,'initial_validation':initial,'validation':best_metric,
             'epochs_executed':len(rows),'best_local_epoch':best_epoch,
             'train_to_best_sec':best_prefix_sec,'actual_training_sec':prefix_sec,
             'compile_warmup_sec':warmup_sec,'pipeline_build_sec':build_total,
             'restored_validation_check':restored_check,
             'lr':float(lr),'model_kind':kind}
    PHASE_ROWS.append(summary);pd.DataFrame(PHASE_ROWS).to_csv(OUT_DIR/'phase_summary.csv',index=False)
    if save_best:save_checkpoint(model,backbone,label+'_selected',best_metric,opt,kind)
    record_event('phase_finished',**summary)
    return dict(best_metric)




In [ ]:
# ---------------- V12 prespecified controller ----------------
V12_LOCAL_TRAIN_SAMPLES=6000
V12_LOCAL_HOLDOUT_SAMPLES=1000
V12_LOCAL_EPOCHS=5
V12_LOCAL_BATCH=32
V12_LOCAL_LR=7.5e-4
V12_TARGET_VMAX=0.45
V12_RECOVERY_LR=2.0e-4
V12_RECOVERY_EPOCHS=2
V12_BLOCK_QAT_LR=5.0e-5
V12_BLOCK_QAT_EPOCHS=2
V12_HEAD_EPOCHS=0
V12_HEAD_PATIENCE=1
INFORMATION_HEAD_CONFIG={'experiment': 'INFO00_BN05_CONTROL_DIAGNOSTICS', 'head_epochs': 0, 'head_lr': 0.0005, 'head_patience': 1, 'trainable_during_head_recovery': [10], 'R_and_depth_fixed': True, 'full_polish_max_epochs': 5, 'full_polish_lr': 5e-06, 'full_polish_patience': 2}
V12_HEAD_LR=0.0005
V12_POLISH_STAGES=(5,6,7,8,9,10)
V12_POLISH_EPOCHS=1
V12_POLISH_LR=5.0e-5
V12_MAPPING_AMPLITUDE_FRACTION=0.45
V12_FUNCTION_GUARD_ATOL=2e-4

# V12.0 boundary evidence: endpoint2 and the S6/S8 local fits were still
# improving at their caps. Later stages keep short, stage-specific budgets.
V123_ENDPOINT2_MAX_EPOCHS=12
V123_ENDPOINT2_PATIENCE=12
V123_ENDPOINT2_QAT_PATIENCE=2
V123_ENDPOINT2_REFINE_LR=2.0e-4
V123_ENDPOINT2_REFINE_MAX_EPOCHS=1
V123_ENDPOINT2_REFINE_PATIENCE=2
V123_LOCAL_MAX_EPOCHS={3:5,4:6,5:5,6:7,7:5,8:8,9:5}
V123_LOCAL_PATIENCE=2
V123_RECOVERY_MAX_EPOCHS={4:1,6:0,8:2,9:0}
V123_BLOCK_QAT_MAX_EPOCHS={2:2,4:2,6:1,8:1,9:0}
V123_LOWER_POLISH_LR=1.5e-5
V123_LOWER_POLISH_EPOCHS=0
V123_LOWER_POLISH_PATIENCE=1
V123_FULL_POLISH_STAGES=tuple(range(1,11))
V123_FULL_POLISH_LR=5.0e-6
V123_FULL_POLISH_EPOCHS=5
V123_FULL_POLISH_PATIENCE=2
FINAL_ADAPTATION_CONFIG={'comparison_family': 'PHAX_FINAL_ADAPTATION_5E_V1', 'variant': 'INFO00_BN05_CONTROL_DIAGNOSTICS', 'parent_experiment': 'BN05_FLOAT_BN_SAME_FULL_ADC_RANGE', 'parent_notebook_sha256': 'ec7af7260e1291b17bafadb9f09727f0b49ce45fda2a8839f55e0fc5e0b5d268', 'max_epochs': 5, 'patience': 2, 'learning_rate': 5e-06, 'active_stages': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 'hardware_refresh_during_final_adaptation': False, 'selection_source': 'validation_only', 'restore_best_state_including_optimizer': True}

# INFO01 experiment overrides (no effect with default switches).
if EXP_RECOVERY_MAX_EPOCHS:V123_RECOVERY_MAX_EPOCHS.update({int(k):int(v) for k,v in EXP_RECOVERY_MAX_EPOCHS.items()})
if EXP_LOCAL_MAX_EPOCHS:V123_LOCAL_MAX_EPOCHS.update({int(k):int(v) for k,v in EXP_LOCAL_MAX_EPOCHS.items()})
if EXP_S10_HEAD_EPOCHS is not None:
    V12_HEAD_EPOCHS=int(EXP_S10_HEAD_EPOCHS);INFORMATION_HEAD_CONFIG['head_epochs']=V12_HEAD_EPOCHS

for k in ('local_cache_sec','local_matching_sec','function_preserving_sec'):
    TIMING[k]=0.0
LOCAL_TIMING_ROWS=[]

V12_CONFIG={
    'local_train_samples':V12_LOCAL_TRAIN_SAMPLES,
    'local_holdout_samples':V12_LOCAL_HOLDOUT_SAMPLES,
    'local_epochs_cap':V12_LOCAL_EPOCHS,'local_batch':V12_LOCAL_BATCH,
    'local_max_epochs_by_stage':V123_LOCAL_MAX_EPOCHS,'local_patience':V123_LOCAL_PATIENCE,
    'local_lr':V12_LOCAL_LR,'target_vmax':V12_TARGET_VMAX,
    'recovery_lr':V12_RECOVERY_LR,'recovery_epochs':V12_RECOVERY_EPOCHS,
    'block_qat_lr':V12_BLOCK_QAT_LR,'block_qat_epochs_by_endpoint':V123_BLOCK_QAT_MAX_EPOCHS,
    'recovery_epochs_by_endpoint':V123_RECOVERY_MAX_EPOCHS,
    'endpoint2_max_epochs':V123_ENDPOINT2_MAX_EPOCHS,'endpoint2_patience':V123_ENDPOINT2_PATIENCE,
    'endpoint2_refine_lr':V123_ENDPOINT2_REFINE_LR,
    'endpoint2_refine_max_epochs':V123_ENDPOINT2_REFINE_MAX_EPOCHS,
    'lower_polish_lr':V123_LOWER_POLISH_LR,'lower_polish_epochs':V123_LOWER_POLISH_EPOCHS,
    'full_polish_stages':V123_FULL_POLISH_STAGES,'full_polish_lr':V123_FULL_POLISH_LR,
    'full_polish_epochs':V123_FULL_POLISH_EPOCHS,
    'full_polish_patience':V123_FULL_POLISH_PATIENCE,
    'final_adaptation':FINAL_ADAPTATION_CONFIG,
    'head_epochs':V12_HEAD_EPOCHS,'polish_stages':V12_POLISH_STAGES,
    'polish_epochs':V12_POLISH_EPOCHS,'mapping_amplitude_fraction':V12_MAPPING_AMPLITUDE_FRACTION,
}
V12_CONFIG['head_lr']=V12_HEAD_LR
V12_CONFIG['head_patience']=V12_HEAD_PATIENCE
V12_CONFIG['information_head_config']=INFORMATION_HEAD_CONFIG
V12_CONFIG['information_probe']={'train_images':INFORMATION_TRAIN_IMAGES,'local_holdout_images':INFORMATION_LOCAL_HOLDOUT_IMAGES,'batch_size':INFORMATION_BATCH_SIZE,'indices_sha256':INFORMATION_PROBE_SHA256}
CONFIG.update(V12_CONFIG)
(OUT_DIR/'config.json').write_text(json.dumps(_plain(CONFIG),indent=2))

LOCAL_ROWS=[]; TRANSITION_ROWS=[]; NORMALIZATION_ROWS=[]; V12_ENDPOINT_ROWS=[]
MATCH_COUNT=V12_LOCAL_TRAIN_SAMPLES+V12_LOCAL_HOLDOUT_SAMPLES
match_local=balanced_indices(y_train,MATCH_COUNT,SEED+12001)
x_match=x_train[match_local]

def set_physical_trainability(backbone,active_stages):
    active=set(map(int,active_stages))
    for i in range(1,11):
        backbone.stage(i).trainable=(i in active)
        if i<10:
            backbone.bn(i).trainable=(i in active)
            backbone.bn(i).set_stats_frozen(i not in active)

def set_suffix_trainability(suffix,flag=True):
    suffix.trainable=bool(flag)
    for layer in suffix.layers: layer.trainable=bool(flag)

def fresh_optimizer(model,lr):
    opt=tf.keras.optimizers.Adam(float(lr)); opt.build(model.trainable_variables); return opt

def eval_and_record(model,label,kind,extra=None):
    m=timed('validation_selection_sec',evaluate,model,x_val,y_val)
    row={'label':label,'kind':kind,'val_accuracy':m['accuracy'],'val_loss':m['loss'],
         'active_shift_stages':str([i for i in range(1,10) if backbone.uses_shift(i)])}
    if extra: row.update(extra)
    TRANSITION_ROWS.append(row);pd.DataFrame(TRANSITION_ROWS).to_csv(OUT_DIR/'transition_metrics.csv',index=False)
    print('TRANSITION',row);return m

def freeze_accepted_prefix(backbone,endpoint):
    for i in range(1,int(endpoint)+1):
        backbone.stage(i).trainable=False
        if i<10:
            backbone.bn(i).trainable=False;backbone.bn(i).set_stats_frozen(True)


## نرمال‌سازی دقیق teacher و نگاشت وزن به رسانایی

برای هر رابط دیجیتال `a` یک ضریب مثبت `s` اعمال می‌شود و kernel لایه بعد روی محور ورودی بر `s` تقسیم می‌شود. چون ReLU، max-pool و GAP برای scale مثبت channel-wise همگن‌اند، logits teacher تغییر نمی‌کند. guard عددی این ادعا را پیش از ادامه بررسی می‌کند.

نگاشت kernel فقط initialization است: زوج `Gp/Gn` حول یک common-mode ثابت ساخته می‌شود تا `Gp+Gn` برای همه سطرها ثابت و `Gp-Gn` متناسب با وزن دیجیتال باشد. تمام Gها داخل 0.12 تا 8 µS هستند و سپس همان کالیبراسیون VTC/R قانونی V11 اجرا می‌شود.


In [ ]:
DIGITAL_STAGE_MAP={
    3:('c3','b3'),4:('c4','b4'),5:('c5','b5'),6:('c6','b6'),
    7:('c7','b7'),8:('c8','b8'),9:('fc','b9')}
NEXT_DIGITAL_LAYER={3:'c4',4:'c5',5:'c6',6:'c7',7:'c8',8:'fc',9:'out'}

def digital_stage_forward(suffix,stage,z,training=False):
    layer_name,bn_name=DIGITAL_STAGE_MAP[int(stage)]
    layer=getattr(suffix,layer_name);bn=getattr(suffix,bn_name)
    z=tf.nn.relu(bn(layer(z),training=training))
    if int(stage) in (4,6):z=tf.nn.max_pool2d(z,2,2,'SAME')
    if int(stage)==8:z=tf.reduce_mean(z,axis=(1,2))
    return z

def teacher_from_endpoint(suffix,prev_endpoint,stop_stage,z):
    for i in range(int(prev_endpoint)+1,int(stop_stage)+1):
        z=digital_stage_forward(suffix,i,z,training=False)
    return z

def hybrid_logits_array(hybrid,x,batch=64):
    out=[]
    for j in range(0,len(x),batch):out.append(hybrid(tf.convert_to_tensor(x[j:j+batch]),training=False).numpy())
    return np.concatenate(out)

def teacher_stage_channel_max(backbone,suffix,prev_endpoint,stage,x):
    mx=np.zeros(STAGE_SPECS[f'stage{stage}']['cout'],np.float64)
    for j in range(0,len(x),CAL_BATCH_SIZE):
        z=physical_prefix(backbone,tf.convert_to_tensor(x[j:j+CAL_BATCH_SIZE]),prev_endpoint,training=False)
        z=teacher_from_endpoint(suffix,prev_endpoint,stage,z)
        axes=tuple(range(z.shape.rank-1));mx=np.maximum(mx,tf.reduce_max(z,axis=axes).numpy())
    return mx

def compensate_next_kernel(suffix,stage,scale):
    layer=getattr(suffix,NEXT_DIGITAL_LAYER[int(stage)])
    kernel=layer.kernel.numpy();scale=np.asarray(scale,np.float32)
    if kernel.ndim==4:kernel=kernel/scale[None,None,:,None]
    elif kernel.ndim==2:kernel=kernel/scale[:,None]
    else:raise RuntimeError('unexpected digital kernel rank')
    layer.kernel.assign(kernel)

def normalize_teacher_interface(backbone,suffix,hybrid,prev_endpoint,stage):
    t=time.perf_counter();guard_x=x_val[:128]
    before=hybrid_logits_array(hybrid,guard_x)
    mx=teacher_stage_channel_max(backbone,suffix,prev_endpoint,stage,x_match)
    scale=np.minimum(1.0,V12_TARGET_VMAX/np.maximum(mx,1e-8)).astype(np.float32)
    bn=getattr(suffix,DIGITAL_STAGE_MAP[int(stage)][1])
    bn.gamma.assign(bn.gamma*scale);bn.beta.assign(bn.beta*scale)
    compensate_next_kernel(suffix,stage,scale)
    after=hybrid_logits_array(hybrid,guard_x)
    err=float(np.max(np.abs(before-after)))
    if err>V12_FUNCTION_GUARD_ATOL:
        raise RuntimeError(f'function-preserving guard failed S{stage}: {err}')
    row={'stage':int(stage),'prev_endpoint':int(prev_endpoint),'logit_max_abs_error':err,
         'scale_min':float(scale.min()),'scale_median':float(np.median(scale)),
         'scale_max':float(scale.max()),'teacher_max_before':float(mx.max()),
         'elapsed_sec':time.perf_counter()-t}
    NORMALIZATION_ROWS.append(row);pd.DataFrame(NORMALIZATION_ROWS).to_csv(OUT_DIR/'teacher_normalization.csv',index=False)
    TIMING['function_preserving_sec']+=row['elapsed_sec'];print('NORMALIZE',row)
    return scale

def _theta_from_conductance(g):
    p=(np.asarray(g,np.float64)-SIG_MIN_US)/(SIG_MAX_US-SIG_MIN_US)
    p=np.clip(p,1e-6,1-1e-6);return np.log(p/(1-p)).astype(np.float32)

def initialize_physical_from_digital(backbone,suffix,stage):
    stage=int(stage);digital=getattr(suffix,DIGITAL_STAGE_MAP[stage][0]);w=digital.kernel.numpy().astype(np.float64)
    s=STAGE_SPECS[f'stage{stage}']
    if s['kind']=='conv':
        if stage==1:target=w.reshape(27,s['cout'])
        else:target=w.reshape(9,s['cin']//64,64,s['cout'])
    else:target=w.reshape(s['cin']//64,64,s['cout'])
    axes=tuple(range(target.ndim-1));peak=np.max(np.abs(target),axis=axes,keepdims=True)
    normalized=target/np.maximum(peak,1e-9)
    center=0.5*(SIG_MIN_US+SIG_MAX_US)
    amp=V12_MAPPING_AMPLITUDE_FRACTION*(SIG_MAX_US-SIG_MIN_US)
    gp=center+amp*normalized;gn=center-amp*normalized
    layer=backbone.stage(stage);layer.theta_wp.assign(_theta_from_conductance(gp));layer.theta_wn.assign(_theta_from_conductance(gn))
    # BN beta learns the exact bias in the local match; keep analog bias common-mode initially.
    layer.theta_bp.assign(_theta_from_conductance(np.full(layer.theta_bp.shape,center)))
    layer.theta_bn.assign(_theta_from_conductance(np.full(layer.theta_bn.shape,center)))
    if np.min(gp)<SIG_MIN_US-1e-9 or np.max(gp)>SIG_MAX_US+1e-9:raise RuntimeError('mapping G violation')
    bump_hardware_revision('digital_weight_mapping_stage_'+str(stage))
    return {'stage':stage,'digital_weight_abs_max':float(np.max(np.abs(target))),
            'mapped_gp_min':float(gp.min()),'mapped_gp_max':float(gp.max()),
            'mapped_gn_min':float(gn.min()),'mapped_gn_max':float(gn.max())}


## تطبیق محلی

بودجه و روش انتخاب بهترین وضعیت طبق کد اصلی است.


In [ ]:
def build_local_cache(backbone,suffix,prev_endpoint,stage,x):
    t=time.perf_counter();inputs=[];targets=[];stage=int(stage)
    for j in range(0,len(x),CAL_BATCH_SIZE):
        xb=tf.convert_to_tensor(x[j:j+CAL_BATCH_SIZE],tf.float32)
        z0=physical_prefix(backbone,xb,prev_endpoint,training=False)
        teacher=teacher_from_endpoint(suffix,prev_endpoint,stage,z0)
        student_in=z0
        for i in range(int(prev_endpoint)+1,stage):
            lin=backbone.stage(i)(student_in,training=False)
            student_in=backbone.interface_after(i,lin,training=False,quantize=True,pool_gap=True)
        inputs.append(student_in.numpy().astype(np.float16));targets.append(teacher.numpy().astype(np.float16))
    xin=np.concatenate(inputs);yt=np.concatenate(targets)
    if float(np.max(yt))>V12_TARGET_VMAX+2e-4:
        raise RuntimeError(f'teacher target exceeds normalized range S{stage}: {np.max(yt)}')
    sec=time.perf_counter()-t;TIMING['local_cache_sec']+=sec
    print('CACHE',stage,xin.shape,yt.shape,'MiB',(xin.nbytes+yt.nbytes)/2**20,'sec',sec)
    return xin,yt

def local_loss_value(pred,target):
    pred=tf.cast(pred,tf.float32);target=tf.cast(target,tf.float32)
    sq=tf.square(pred-target);mse=tf.reduce_mean(sq)
    energy=tf.reduce_mean(tf.square(target))+1e-4
    reduce_axes=tf.range(tf.rank(pred)-1)
    channel_mse=tf.reduce_mean(sq,axis=reduce_axes)
    channel_energy=tf.reduce_mean(tf.square(target),axis=reduce_axes)+1e-4
    channel_nmse=tf.reduce_mean(channel_mse/channel_energy)
    global_nmse=mse/energy
    pf=tf.reshape(pred,[tf.shape(pred)[0],-1]);tfv=tf.reshape(target,[tf.shape(target)[0],-1])
    cosine=tf.reduce_mean(tf.keras.losses.cosine_similarity(tfv,pf,axis=-1))
    loss=0.45*global_nmse+0.50*channel_nmse+0.05*(1.0+cosine)
    return loss,mse,cosine,channel_nmse

def local_match_stage(backbone,stage,xin,target):
    stage=int(stage);ntr=V12_LOCAL_TRAIN_SAMPLES
    xtr,ytr=xin[:ntr].astype(np.float32),target[:ntr].astype(np.float32)
    xva,yva=xin[ntr:].astype(np.float32),target[ntr:].astype(np.float32)
    layer=backbone.stage(stage);bn=backbone.bn(stage)
    layer.trainable=True;bn.trainable=True;bn.set_stats_frozen(False)
    variables=layer.trainable_variables+bn.trainable_variables
    state_variables=layer.variables+bn.variables
    opt=tf.keras.optimizers.Adam(V12_LOCAL_LR);opt.build(variables)

    @tf.function(reduce_retracing=True)
    def step(x,y):
        with tf.GradientTape() as tape:
            lin=layer(x,training=True)
            pred=backbone.interface_after(stage,lin,training=True,quantize=True,pool_gap=True)
            loss,mse,cos,channel_nmse=local_loss_value(pred,y)
        grads=tape.gradient(loss,variables);bn_apply_gradients(opt,[(g,v) for g,v in zip(grads,variables) if g is not None])
        return loss,mse,cos,channel_nmse

    def local_eval():
        sums=np.zeros(5,np.float64)
        for j in range(0,len(xva),V12_LOCAL_BATCH):
            xb=tf.convert_to_tensor(xva[j:j+V12_LOCAL_BATCH]);yb=tf.convert_to_tensor(yva[j:j+V12_LOCAL_BATCH])
            pred=backbone.interface_after(stage,layer(xb,training=False),training=False,quantize=True,pool_gap=True)
            loss,mse,cos,channel_nmse=local_loss_value(pred,yb);n=len(xb);sums+=np.array([loss.numpy()*n,mse.numpy()*n,cos.numpy()*n,channel_nmse.numpy()*n,n])
        return {'loss':float(sums[0]/sums[4]),'mse':float(sums[1]/sums[4]),
                'cosine_loss_sign':float(sums[2]/sums[4]),'channel_nmse':float(sums[3]/sums[4])}

    def local_warmup():
        # Compile the local step for the full and remainder batch, then restore bit-exactly.
        if not TIMING_WARMUP_BEFORE_PHASE: return
        ov=list(opt.variables);sm=_state_arrays(state_variables);so=_state_arrays(ov)
        b=V12_LOCAL_BATCH;rem=len(xtr)%b
        out=step(tf.convert_to_tensor(xtr[:b]),tf.convert_to_tensor(ytr[:b]))
        if rem:out=step(tf.convert_to_tensor(xtr[:rem]),tf.convert_to_tensor(ytr[:rem]))
        _=float(out[0].numpy())
        _assign_arrays(state_variables,sm);_assign_arrays(ov,so)
        if TIMING_VERIFY_EXACT_RESTORE and not (_arrays_equal(state_variables,sm) and _arrays_equal(ov,so)):
            raise RuntimeError(f'local warm-up restore is not bit-exact S{stage}')

    best=timed('local_eval_sec',local_eval);best_weights=[v.numpy().copy() for v in state_variables]
    max_epochs=int(V123_LOCAL_MAX_EPOCHS.get(stage,V12_LOCAL_EPOCHS));wait=0;best_epoch=0;epoch_secs=[]
    timed('compile_warmup_sec',local_warmup)
    for epoch in range(1,max_epochs+1):
        tb=time.perf_counter()
        ds=tf.data.Dataset.from_tensor_slices((xtr,ytr)).shuffle(len(xtr),seed=SEED+stage*100+epoch,reshuffle_each_iteration=False).batch(V12_LOCAL_BATCH).prefetch(tf.data.AUTOTUNE)
        TIMING['input_pipeline_build_sec']+=time.perf_counter()-tb
        t0=time.perf_counter()
        for xb,yb in ds:step(xb,yb)
        _sync_on(variables)
        epoch_secs.append(time.perf_counter()-t0)
        # Everything below is outside the local-optimization timer.
        timed('bn_experiment_diagnostic_sec',bn_record_parameters,backbone,f'local_stage{stage}_epoch{epoch}')
        va=timed('local_eval_sec',local_eval);row={'stage':stage,'epoch':epoch,'val_local_loss':va['loss'],'val_local_mse':va['mse'],
                             'val_cosine_loss_sign':va['cosine_loss_sign'],'val_channel_nmse':va['channel_nmse'],
                             'max_epochs':max_epochs,'train_epoch_sec':epoch_secs[-1]}
        LOCAL_ROWS.append(row);pd.DataFrame(LOCAL_ROWS).to_csv(OUT_DIR/'local_matching_history.csv',index=False)
        print('LOCAL',row)
        if va['loss']<best['loss']-1e-12:
            best=va;best_weights=[v.numpy().copy() for v in state_variables];wait=0;best_epoch=epoch
        else:
            wait+=1
            if wait>=V123_LOCAL_PATIENCE:
                print('LOCAL EARLY STOP',stage,'at',epoch,'best_loss',best['loss']);break
    for v,a in zip(state_variables,best_weights):v.assign(a)
    sec_all=float(sum(epoch_secs));sec_best=float(sum(epoch_secs[:best_epoch]))
    TIMING['local_matching_sec']+=sec_all;TIMING['local_matching_to_best_sec']+=sec_best
    LOCAL_TIMING_ROWS.append({'stage':stage,'epochs_executed':len(epoch_secs),'best_local_epoch':best_epoch,
                              'train_all_sec':sec_all,'train_to_best_sec':sec_best,
                              'epoch_secs':[round(s,4) for s in epoch_secs]})
    pd.DataFrame(LOCAL_TIMING_ROWS).to_csv(OUT_DIR/'local_timing.csv',index=False)
    bn.set_stats_frozen(True);bump_hardware_revision('local_match_selected_stage_'+str(stage))
    return best,sec_best


# Outside local matching's measured training interval; no extra parameter updates.
_local_match_without_information=local_match_stage
def local_match_stage(backbone,stage,xin,target):
    timed('information_diagnostic_sec',information_local_fit,backbone,stage,xin,target,'LOCAL_INITIAL')
    result=_local_match_without_information(backbone,stage,xin,target)
    timed('information_diagnostic_sec',information_local_fit,backbone,stage,xin,target,'LOCAL_SELECTED')
    return result


## endpoint2: دوازده epoch اجباری با انتخاب بهترین state

این فاز عمداً early stop ندارد، چون افت تک‌epochی بخشی از منحنی عادی آن است. هر 12 epoch اجرا می‌شوند، اما snapshot بهترین validation مانند قبل بازگردانده می‌شود. بلافاصله پس از فاز، guard هم تعداد epoch و هم حداقل دقت مورد انتظار را بررسی می‌کند.

> **تغییر نسخه Quality-Guard Fixed:** آستانهٔ Validation=87.50% صرفاً معیار تشخیصی است. شکست آن با هشدار ثبت می‌شود تا آزمایش LUT از صفر تا آخر اجرا شود؛ محدودیت‌های ولتاژ، رسانایی و تعداد اینورترها تغییر نکرده‌اند.


In [ ]:
backbone=VGGPHAXBackbone()
for i in range(1,10):
    if not backbone.bn(i).built:backbone.bn(i).build((None,backbone.bn(i).channels))
bn_initialize_hard_from_start(backbone)
suffix=ProgressiveDigitalSuffix()
_=suffix.call_from(2,tf.zeros([1,16,16,64],tf.float32),training=False)

timed('hardware_codesign_sec',calibrate_stages,backbone,x_cal,(1,2),'front/initial_v12')
bump_hardware_revision('v12_calibrated_initial_stages_1_2')
hybrid=ProgressiveHybridClassifier(backbone,suffix);_=hybrid(tf.convert_to_tensor(x_val[:1]),training=False)

set_physical_trainability(backbone,(1,2));set_suffix_trainability(suffix,True)
if EXP_ENDPOINT2_PROTOCOL=='reference':
    opt=fresh_optimizer(hybrid,PROGRESSIVE_PLAN[2]['lr'])
    endpoint2_float=run_phase_v111(hybrid,backbone,opt,'v12_endpoint2_float',PROGRESSIVE_PLAN[2]['lr'],V123_ENDPOINT2_PATIENCE,V123_ENDPOINT2_MAX_EPOCHS,SEED+12020,
                                    save_best=False,diagnose_gradient=False)
else:
    # EXPERIMENT E1: baseline-like plateau protocol (1e-3 at batch 32 = 5e-4 at batch 16
    # under linear scaling); patience-driven, validation-only selection.
    P=EXP_E2_PLATEAU
    opt=fresh_optimizer(hybrid,P['lr'])
    endpoint2_float=run_phase_v111(hybrid,backbone,opt,'v12_endpoint2_float',P['lr'],P['patience'],P['max_epochs'],SEED+12020,
                                    save_best=False,diagnose_gradient=False)
eval_and_record(hybrid,'endpoint2_float_selected','endpoint')

if EXP_ENDPOINT2_PROTOCOL=='reference' and PHASE_ROWS[-1]['epochs_executed'] != V123_ENDPOINT2_MAX_EPOCHS:
    raise RuntimeError('endpoint2 schedule guard: all 12 high-LR epochs must execute')
# Scientific quality diagnostic, NOT a physics/voltage safety constraint.
# For discrete-LUT from-scratch training, a close miss (e.g. 0.8744 vs 0.875)
# should be recorded instead of discarding the run before its refinement/QAT stages.
# The 12-epoch schedule and all physical/electrical legality checks stay strict.
ENDPOINT2_QUALITY_REFERENCE_ACC = 0.875
STRICT_ENDPOINT2_QUALITY_GUARD = False  # True restores the old early-failure behavior.
endpoint2_quality_passed = (endpoint2_float['accuracy'] >= ENDPOINT2_QUALITY_REFERENCE_ACC)
record_event('endpoint2_quality_check',
             validation_accuracy=endpoint2_float['accuracy'],
             target_accuracy=ENDPOINT2_QUALITY_REFERENCE_ACC,
             passed=endpoint2_quality_passed,
             strict=STRICT_ENDPOINT2_QUALITY_GUARD)
if not endpoint2_quality_passed:
    msg=(f"Endpoint-2 quality warning: validation={endpoint2_float['accuracy']:.4%}, "
         f"reference={ENDPOINT2_QUALITY_REFERENCE_ACC:.4%}. "
         "Continuing unchanged training schedule to gather Direct-LUT results; "
         "this does NOT certify accuracy equivalence with tanh/digital baseline.")
    warnings.warn(msg)
    if STRICT_ENDPOINT2_QUALITY_GUARD:
        raise RuntimeError(msg)
record_event('endpoint2_schedule_guard_passed',
             epochs=PHASE_ROWS[-1]['epochs_executed'],
             validation_accuracy=endpoint2_float['accuracy'])

set_physical_trainability(backbone,(1,2));set_suffix_trainability(suffix,True)
if EXP_ENDPOINT2_PROTOCOL=='reference':
    opt=fresh_optimizer(hybrid,V123_ENDPOINT2_REFINE_LR)
    endpoint2_refined=run_phase_v111(hybrid,backbone,opt,'v12_endpoint2_float_refine',V123_ENDPOINT2_REFINE_LR,
                                     V123_ENDPOINT2_REFINE_PATIENCE,V123_ENDPOINT2_REFINE_MAX_EPOCHS,
                                     SEED+12070,save_best=False,diagnose_gradient=False)
else:
    opt=fresh_optimizer(hybrid,P['low_lr'])
    run_phase_v111(hybrid,backbone,opt,'v12_endpoint2_float_low',P['low_lr'],P['low_patience'],P['low_max_epochs'],
                   SEED+12040,save_best=False,diagnose_gradient=False)
    opt=fresh_optimizer(hybrid,P['lower_lr'])
    endpoint2_refined=run_phase_v111(hybrid,backbone,opt,'v12_endpoint2_float_refine',P['lower_lr'],
                                     P['lower_patience'],P['lower_max_epochs'],
                                     SEED+12070,save_best=False,diagnose_gradient=False)
eval_and_record(hybrid,'endpoint2_float_refined','endpoint_refine')
if endpoint2_refined['accuracy'] < ENDPOINT2_QUALITY_REFERENCE_ACC:
    record_event('endpoint2_refinement_quality_warning',
                 validation_accuracy=endpoint2_refined['accuracy'],
                 reference_accuracy=ENDPOINT2_QUALITY_REFERENCE_ACC,
                 continued=True)
    print('QUALITY WARNING: Endpoint-2 remains below its reference after refinement; '
          'continuing solely to measure the full hardware-aware Direct-LUT experiment.')

for i in (1,2):timed('hardware_codesign_sec',synthesize_one_bn,backbone,x_cal,i,'v12_block_shift_1_2')
backbone.activate_shift_stages((1,2))
timed('information_diagnostic_sec',information_conversion,backbone,(1,2),'CONVERSION_S1_S2')
shiftadd_hard_legality_audit(backbone,'v12_block_shift_1_2_immediate')
endpoint2_shift_immediate=eval_and_record(hybrid,'endpoint2_shift_immediate','conversion')
set_physical_trainability(backbone,(1,2));set_suffix_trainability(suffix,True)
opt=fresh_optimizer(hybrid,V12_BLOCK_QAT_LR)
endpoint2_shift=run_phase_v111(hybrid,backbone,opt,'v12_endpoint2_shift_qat',V12_BLOCK_QAT_LR,V123_ENDPOINT2_QAT_PATIENCE,V123_BLOCK_QAT_MAX_EPOCHS[2],
                                SEED+12120,save_best=False,diagnose_gradient=False)
freeze_accepted_prefix(backbone,2)
V12_ENDPOINT_ROWS.append({'endpoint':2,'float_selected':endpoint2_float,'float_refined':endpoint2_refined,
                          'transition':endpoint2_shift_immediate,'shift_selected':endpoint2_shift})


## تبدیل بلوک‌ها

بودجه و روش انتخاب بهترین وضعیت طبق کد اصلی است.


In [ ]:
V12_BLOCKS=((2,4,(3,4)),(4,6,(5,6)),(6,8,(7,8)),(8,9,(9,)))
MAPPING_ROWS=[]
for prev_endpoint,new_endpoint,stages in V12_BLOCKS:
    if hybrid.endpoint!=prev_endpoint:raise RuntimeError('unexpected hybrid endpoint')
    for stage in stages:
        normalize_teacher_interface(backbone,suffix,hybrid,prev_endpoint,stage)
        mapping=initialize_physical_from_digital(backbone,suffix,stage);MAPPING_ROWS.append(mapping)
        inp=stage_input_for(backbone,x_cal,stage)
        timed('hardware_codesign_sec',calibrate_stage,backbone.stage(stage),inp,f'v12_endpoint{new_endpoint}/stage{stage}')
        bump_hardware_revision(f'v12_calibrated_stage_{stage}')
        xin,yt=build_local_cache(backbone,suffix,prev_endpoint,stage,x_match)
        best,sec=local_match_stage(backbone,stage,xin,yt)
        del xin,yt;gc.collect()
        MAPPING_ROWS[-1].update({'local_best_loss':best['loss'],'local_best_mse':best['mse'],'local_match_sec':sec})
        pd.DataFrame(MAPPING_ROWS).to_csv(OUT_DIR/'digital_to_physical_mapping.csv',index=False)

    old_metric=evaluate(hybrid,x_val,y_val)
    hybrid.set_endpoint(new_endpoint)
    transition=eval_and_record(hybrid,f'endpoint{new_endpoint}_after_local_match','function_preserving_transition',
                               {'previous_endpoint_accuracy':old_metric['accuracy']})

    set_physical_trainability(backbone,stages);set_suffix_trainability(suffix,True)
    opt=fresh_optimizer(hybrid,V12_RECOVERY_LR)
    recovered=run_phase_v111(hybrid,backbone,opt,f'v12_endpoint{new_endpoint}_recovery',V12_RECOVERY_LR,1,
                              V123_RECOVERY_MAX_EPOCHS[new_endpoint],SEED+13000+new_endpoint*100,
                              save_best=False,diagnose_gradient=False)

    for i in stages:timed('hardware_codesign_sec',synthesize_one_bn,backbone,x_cal,i,f'v12_endpoint{new_endpoint}_shift')
    backbone.activate_shift_stages(stages)
    timed('information_diagnostic_sec',information_conversion,backbone,stages,f'CONVERSION_ENDPOINT_{new_endpoint}')
    shiftadd_hard_legality_audit(backbone,f'v12_endpoint{new_endpoint}_shift_immediate')
    shift_immediate=eval_and_record(hybrid,f'endpoint{new_endpoint}_shift_immediate','conversion')

    set_physical_trainability(backbone,stages);set_suffix_trainability(suffix,True)
    opt=fresh_optimizer(hybrid,V12_BLOCK_QAT_LR)
    shift_selected=run_phase_v111(hybrid,backbone,opt,f'v12_endpoint{new_endpoint}_shift_qat',V12_BLOCK_QAT_LR,1,
                                   V123_BLOCK_QAT_MAX_EPOCHS[new_endpoint],SEED+14000+new_endpoint*100,
                                   save_best=False,diagnose_gradient=False)
    freeze_accepted_prefix(backbone,new_endpoint)
    V12_ENDPOINT_ROWS.append({'endpoint':new_endpoint,'transition':transition,'float_recovered':recovered,
                              'shift_immediate':shift_immediate,'shift_selected':shift_selected})
    pd.DataFrame([{'endpoint':r['endpoint'],
                   'transition_accuracy':r.get('transition',{}).get('accuracy'),
                   'shift_selected_accuracy':r.get('shift_selected',{}).get('accuracy')}
                  for r in V12_ENDPOINT_ROWS]).to_csv(OUT_DIR/'v12_endpoint_summary.csv',index=False)

if hybrid.endpoint!=9 or not all(backbone.shift_active.values()):
    raise RuntimeError('V12 block conversion incomplete')


## آموزش نهایی مشترک

بودجه و روش انتخاب بهترین وضعیت طبق کد اصلی است.


In [ ]:
# Stage 10 uses the compensated digital output kernel as its initialization.
digital_out=suffix.out;w=digital_out.kernel.numpy().astype(np.float64).reshape(8,64,10)
# INFO00 normalizes every class column by its own peak (relative logit scale lost).
# EXPERIMENT EXP_S10_RELATIVE_SCALE: one global peak keeps Gp-Gn proportional to w
# across classes; the column denominator Gp+Gn is identical for every column.
peak=(np.max(np.abs(w)) if EXP_S10_RELATIVE_SCALE else np.max(np.abs(w),axis=(0,1),keepdims=True));wn=w/np.maximum(peak,1e-9)
center=0.5*(SIG_MIN_US+SIG_MAX_US);amp=V12_MAPPING_AMPLITUDE_FRACTION*(SIG_MAX_US-SIG_MIN_US)
backbone.stage(10).theta_wp.assign(_theta_from_conductance(center+amp*wn))
backbone.stage(10).theta_wn.assign(_theta_from_conductance(center-amp*wn))
backbone.stage(10).theta_bp.assign(_theta_from_conductance(np.full((10,),center)))
backbone.stage(10).theta_bn.assign(_theta_from_conductance(np.full((10,),center)))
bump_hardware_revision('digital_weight_mapping_stage_10')
timed('hardware_codesign_sec',calibrate_stages,backbone,x_cal,(10,),'v12_stage10_initial')
bump_hardware_revision('v12_calibrated_stage_10')
timed('information_diagnostic_sec',information_head,backbone,suffix,'S10_MAPPED_BEFORE_RECOVERY')

full_model=FullPHAXClassifier(backbone);_=full_model(tf.convert_to_tensor(x_val[:1]),training=False)
set_physical_trainability(backbone,(10,));backbone.set_bn_stats_frozen(True)
head_opt=fresh_optimizer(full_model,V12_HEAD_LR)
head_metric=run_phase_v111(full_model,backbone,head_opt,'v12_stage10_head',V12_HEAD_LR,V12_HEAD_PATIENCE,V12_HEAD_EPOCHS,
                           SEED+15000,save_best=False,diagnose_gradient=False)
if V12_HEAD_EPOCHS>0:
    timed('information_diagnostic_sec',information_head,backbone,suffix,'S10_RECOVERY_SELECTED')

set_physical_trainability(backbone,V12_POLISH_STAGES);backbone.set_bn_stats_frozen(True)
polish_opt=fresh_optimizer(full_model,V12_POLISH_LR)
polish_metric=run_phase_v111(full_model,backbone,polish_opt,'v12_tail_coadapt',V12_POLISH_LR,2,V12_POLISH_EPOCHS,
                             SEED+15100,save_best=False,diagnose_gradient=True)

set_physical_trainability(backbone,V123_FULL_POLISH_STAGES);backbone.set_bn_stats_frozen(True)
full_polish_opt=fresh_optimizer(full_model,V123_FULL_POLISH_LR)
full_polish_metric=run_phase_v111(full_model,backbone,full_polish_opt,'v12_guarded_full_polish',
                                  V123_FULL_POLISH_LR,V123_FULL_POLISH_PATIENCE,V123_FULL_POLISH_EPOCHS,
                                  SEED+15200,save_best=False,diagnose_gradient=True)
backbone.set_bn_stats_frozen(True)
final_validation=timed('validation_selection_sec',evaluate,full_model,x_val,y_val)
print('V12.3.1 FINAL VALIDATION',final_validation)


## audit، test و خروجی

test فقط یک‌بار و پس از تمام انتخاب‌های validation بارگذاری می‌شود. checkpoint نهایی هیچ suffix دیجیتال ندارد. optimizer ذخیره‌شده عمداً روی تمام متغیرهای مدل ساخته می‌شود تا checkpoint در اجرای بعدی بدون mismatch قابل load باشد.


In [ ]:
# Discrete R and real gain audit: one CSV for every single inverter/column.
rows=[]
for stage_i in range(1,11):
    layer=backbone.stage(stage_i)
    for path in ('main_chain','post_chain'):
        if path not in layer.hardware:continue
        hw=layer.hardware[path]
        req=np.asarray(hw['column_requested_gain'])
        tgt=np.asarray(hw['column_target_gain'])
        effective=np.asarray(hw['column_actual_secant_gain'])
        center=np.asarray(hw['column_actual_center_slope_gain'])
        for lev,(ii,rr) in enumerate(zip(hw['lut_index_levels'],hw['R_kohm_levels']),1):
            for ch,(ind,r) in enumerate(zip(ii,rr)):
                rows.append({'stage':stage_i,'chain':path,'level':lev,'channel':ch,
                             'R_kohm':r,'LUT_R_index':int(ind),'R_physical_slope_abs':float(RAW_SLOPE_ABS[ind]),
                             'chain_requested_gain':float(req[ch]),'chain_target_gain':float(tgt[ch]),
                             'chain_product_central_slopes':float(center[ch]),
                             'chain_secant_effective_gain':float(effective[ch]),
                             'chain_depth':int(hw['n_inv']),
                             'memristors_in_level':int(hw['memristor_count_levels'][lev-1][ch])})
pd.DataFrame(rows).to_csv(OUT_DIR/'direct_hspice_selected_resistors_and_gains.csv',index=False)
print('Selected measured resistors:',len(rows),'entries')


In [ ]:
timed('information_diagnostic_sec',information_head,backbone,suffix,'FINAL_SELECTED')
timed('information_diagnostic_sec',information_final_boundaries,backbone,'FINAL_SELECTED')
information_report=timed('information_diagnostic_sec',information_write_summary)
timed('bn_experiment_diagnostic_sec',bn_record_parameters,backbone,'FINAL_SELECTED')
bn_export_final_channels(backbone)
final_audit=timed('physical_audit_sec',deep_audit,backbone,x_diag,y_diag,'v12_final')
final_certificate=timed('physical_audit_sec',shiftadd_hard_legality_audit,backbone,'v12_final')
trained_gain_audit(backbone,x_diag,'v12_final')

# Canonical all-variable optimizer makes the full checkpoint structurally reloadable.
set_physical_trainability(backbone,range(1,11));backbone.set_bn_stats_frozen(True)
canonical_opt=tf.keras.optimizers.Adam(V12_POLISH_LR);canonical_opt.build(full_model.trainable_variables)
final_checkpoint=save_checkpoint(full_model,backbone,EXPERIMENT_NAME+'_final',final_validation,canonical_opt,'full')
checkpoint_zip=Path('/kaggle/working')/f'{EXPERIMENT_NAME}_FINAL_CHECKPOINT_{RUN_ID}.zip'
t=time.perf_counter()
with zipfile.ZipFile(checkpoint_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in final_checkpoint.iterdir():
        if p.is_file():z.write(p,p.name)
TIMING['checkpoint_sec']+=time.perf_counter()-t

root=find_cifar10_batches_dir();x_test,y_test=load_cifar_batch(root/'test_batch')
x_test=x_test.astype(np.float32)/255.0-0.5
test_metric=timed('final_test_sec',evaluate,full_model,x_test,y_test)
train_metric=timed('train_diagnostic_sec',evaluate,full_model,x_train,y_train) if RUN_FINAL_TRAIN_EVAL else None
dataset_sha=hashlib.sha256(b''.join((root/f'data_batch_{i}').read_bytes() for i in range(1,6))).hexdigest()

# INFO01 definition (same concept as INFO00, cleaner instruments):
#   classification: pure train-loop seconds up to each phase's selected epoch (0 if epoch 0)
#   local matching: pure optimization seconds up to each stage's selected local epoch
#   compile/warm-up, data/pipeline construction, caches, calibration, R/depth selection,
#   teacher scaling, validation/test, diagnostics, logging and checkpoints are excluded.
selected_full_phase_sec=float(sum(r['train_to_best_sec'] for r in PHASE_ROWS))
selected_local_matching_sec=float(TIMING['local_matching_to_best_sec'])
selected_path_train_sec=selected_full_phase_sec+selected_local_matching_sec
actual_all_training_sec=float(TIMING['train_forward_backward_sec']+TIMING['local_matching_sec'])
if selected_path_train_sec>actual_all_training_sec+1e-6:
    raise RuntimeError('selected path cannot exceed all executed training')
TIMING_DEFINITION={'selected_path':'sum(train_to_best_sec over classification phases)+sum(local to-best optimization sec)',
    'excluded':['compile_warmup_sec','input_pipeline_build_sec','local_cache_sec','hardware_codesign_sec',
                'function_preserving_sec','validation_selection_sec','local_eval_sec','restore_verification_sec',
                'final_test_sec','train_diagnostic_sec','information_diagnostic_sec','bn_experiment_diagnostic_sec',
                'physical_audit_sec','diagnostic_sec','checkpoint_sec','benchmark_sec','setup_sec'],
    'gpu_sync':'host read of on-device accumulator / trained-variable sum at interval end',
    'warmup':'separate 1-2 batch dataset; bit-exact restore verified' if TIMING_WARMUP_BEFORE_PHASE else 'disabled',
    'INFO00_legacy_reference':{'selected_full_phase_sec':3307.070204288,'local_matching_sec':358.19750007599987,
        'selected_path_train_sec':3665.2677043639997,
        'note':'legacy local timer included local eval, BN logging, CSV writes, dataset construction and compilation; '
               'legacy phase timers included first-epoch tracing/compilation'}}
pd.DataFrame([{k:v for k,v in r.items() if k not in ('initial_validation','validation','restored_validation_check')}
              for r in PHASE_ROWS]).to_csv(OUT_DIR/'timing_phase_breakdown.csv',index=False)
digital_reference={'validation_accuracy':0.9214,'test_accuracy':0.9193,
                   'selected_path_train_sec':2072.356839279,'actual_training_sec':2661.1027997659985,
                   'run_id':'20261005_093611'}
v11_reference={'validation_accuracy':0.9126,'test_accuracy':0.9043,
               'selected_path_train_sec':13618.74,'actual_training_sec':17592.53,
               'run_id':'20261006_084142'}

comparison=[
    {'model':'digital_baseline',**{k:digital_reference[k] for k in ('validation_accuracy','test_accuracy','selected_path_train_sec')}},
    {'model':'V11.0',**{k:v11_reference[k] for k in ('validation_accuracy','test_accuracy','selected_path_train_sec')}},
    {'model':EXPERIMENT_NAME,'validation_accuracy':final_validation['accuracy'],'test_accuracy':test_metric['accuracy'],
     'selected_path_train_sec':selected_path_train_sec},
]
for r in comparison:r['selected_ratio_vs_digital']=r['selected_path_train_sec']/digital_reference['selected_path_train_sec']
pd.DataFrame(comparison).to_csv(OUT_DIR/'digital_v11_v12_comparison.csv',index=False)

hist=pd.DataFrame(EVAL_ROWS)
if len(hist):
    fig,ax=plt.subplots(figsize=(12,5));ax.plot(hist['cumulative_optimizer_epoch'],100*hist['val_accuracy'],marker='o',ms=3,label='validation')
    ax.plot(hist['cumulative_optimizer_epoch'],100*hist['train_accuracy'],alpha=.5,label='train')
    ax.axhline(100*digital_reference['validation_accuracy'],ls='--',color='black',label='digital validation')
    ax.set(xlabel='optimizer epoch',ylabel='accuracy (%)',title=EXPERIMENT_NAME)
    ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(OUT_DIR/'accuracy_by_epoch.png',dpi=160);plt.close(fig)

TIMING['total_wall_sec']=time.perf_counter()-RUN_T0
# suffix_training_sec and local_matching_to_best_sec are subsets of other buckets.
identified=sum(v for k,v in TIMING.items() if k not in ('total_wall_sec','suffix_training_sec','local_matching_to_best_sec','unattributed_session_sec'))
TIMING['unattributed_session_sec']=TIMING['total_wall_sec']-identified
if TIMING['unattributed_session_sec']<-1.0:
    raise RuntimeError(f"timer overlap: exclusive buckets exceed wall time by {-TIMING['unattributed_session_sec']:.2f}s")
summary={'artifact':ARTIFACT_VERSION,'run_id':RUN_ID,'run_mode':'from_scratch_patience_fixed_selected_path_with_guarded_full_polish',
         'dataset':'CIFAR-10','seed':SEED,'split_fingerprint':SPLIT_FINGERPRINT,'dataset_sha256':dataset_sha,
         'vtc_model_sha256':VTC_MODEL_SHA256,'vtc_source':MODEL_SOURCE_MODE,'lut_gpu_execution':FAST_LUT_MODE,'validation':final_validation,'test':test_metric,'train_full':train_metric,
         'endpoint2_float':endpoint2_float,'endpoint2_float_refined':endpoint2_refined,
         'endpoint2_shift':endpoint2_shift,'endpoints':V12_ENDPOINT_ROWS,
         'head_metric':head_metric,'tail_coadapt':polish_metric,
         'guarded_full_polish':full_polish_metric,
         'final_adaptation_config':FINAL_ADAPTATION_CONFIG,'phases':PHASE_ROWS,
         'optimizer_epochs':CUM_EPOCH,'local_matching_rows':LOCAL_ROWS,
         'selected_path_train_sec':selected_path_train_sec,'selected_full_phase_sec':selected_full_phase_sec,
         'selected_local_matching_sec':selected_local_matching_sec,
         'local_matching_all_executed_sec':float(TIMING['local_matching_sec']),
         'actual_all_training_sec_including_local':actual_all_training_sec,
         'compile_warmup_sec':float(TIMING['compile_warmup_sec']),
         'total_wall_sec':float(TIMING['total_wall_sec']),
         'timing_definition':TIMING_DEFINITION,'local_timing':LOCAL_TIMING_ROWS,
         'info01_config':CONFIG['INFO01'],
         'temporary_digital_suffix_used_for_training_only':True,'external_teacher_or_checkpoint_used':False,
         'function_preserving_teacher_scaling_guard':NORMALIZATION_ROWS,
         'posthoc_gain_refresh_used':False,'accepted_prefix_frozen_after_each_block':True,
         'channel_balanced_local_loss':True,'adaptive_epoch_budget':True,
         'endpoint2_low_lr_refinement':True,'tail_coadapt_stages':list(V12_POLISH_STAGES),
         'direct_selected_epoch_schedule':True,'endpoint2_full_schedule_guard':True,
         'guarded_full_polish_stages':list(V123_FULL_POLISH_STAGES),
         'temporary_suffix_in_final_checkpoint':False,'all_nine_BNs_shiftadd_at_final':all(backbone.shift_active.values()),
         'information_diagnostics':information_report,'information_head_config':INFORMATION_HEAD_CONFIG,
         'experiment':EXPERIMENT_NAME,'base_notebook_sha256':CONFIG['base_notebook_sha256'],'bn_experiment_config':CONFIG['BN_experiment'],'bn_parameter_history':BN_PARAMETER_ROWS,'bn_stats_reestimation':BN_STATS_ROWS,'hardware_cost':hardware_cost(backbone),'physical_audit':final_audit,
         'conductance_uS':[SIG_MIN_US,SIG_MAX_US],'ADC_codes':[-127,127],'DAC_codes':[0,255],
         'BN_accumulator_bits':BN_ACCUMULATOR_BITS,'loading_mode':LOADING_MODE,'repeat_policy':REPEAT_POLICY,
         'real_HSPICE_LUT_used':True,'environment':ENVIRONMENT,'test_used_for_selection':False,'timing':TIMING,
         'digital_reference':digital_reference,'v11_reference':v11_reference,'checkpoint_file':checkpoint_zip.name,
         'comparison_limits':['single V12.3.1 seed; repeat seeds before a paper claim',
                              'local teacher is internal to the same random-initialized run and absent from final checkpoint']}
(OUT_DIR/'summary.json').write_text(json.dumps(_plain(summary),indent=2))
(OUT_DIR/'final_hardware.json').write_text(json.dumps(_plain(snapshot_hardware(backbone))))
(OUT_DIR/'final_shiftadd_bn.json').write_text(json.dumps({str(i):_plain(backbone.bn(i).export_shift_spec()) for i in range(1,10)}))
flush_reports()

results_zip=Path('/kaggle/working')/f'{EXPERIMENT_NAME}_RESULTS_{RUN_ID}.zip'
with zipfile.ZipFile(results_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.iterdir():
        if p.is_file() and p.suffix in ('.csv','.json','.png'):z.write(p,p.name)
print(json.dumps(_plain(summary),indent=2))
print(pd.DataFrame(comparison).to_string(index=False))
print('RESULTS',results_zip)
print('FINAL CHECKPOINT',checkpoint_zip)
